# **Bloque 1**

In [1]:
# ============================================================
# BLOQUE 1: Instalación, imports y configuración de Supabase
# ============================================================
import importlib
import io
import threading
import time
import unicodedata
import json
import uuid
import re
import traceback
import difflib
import datetime
import os
import zipfile
import tempfile
import shutil
import numpy as np
import pandas as pd
from flask import Flask, jsonify, request, render_template_string, make_response
from google.colab import userdata
import socket

def instalar_e_importar(paquete):
    module_name = paquete
    if paquete == 'scikit-learn':
        module_name = 'sklearn'
    try:
        importlib.import_module(module_name)
    except ImportError:
        print(f"Instalando {paquete}...")
        !pip install {paquete}
        print(f"{paquete} instalado exitosamente.")
        importlib.invalidate_caches()
        importlib.import_module(module_name)

instalar_e_importar('pandas')
instalar_e_importar('flask')
instalar_e_importar('pyngrok')
instalar_e_importar('supabase')
instalar_e_importar('scikit-learn')

from supabase import create_client, Client
from pyngrok import ngrok
from pyngrok.exception import PyngrokNgrokHTTPError

SUPABASE_URL = userdata.get('SUPABASE_URL').strip()
SUPABASE_KEY = userdata.get('SUPABASE_KEY').strip()
supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)
print("Cliente de Supabase inicializado.")

try:
    response = supabase.from_('variables').select('*').limit(1).execute()
    if response.data is not None:
        print("La tabla 'variables' existe y es accesible.")
    else:
        print("La tabla 'variables' puede no existir o no contiene datos.")
except Exception as e:
    print(f"Error al verificar la tabla 'variables': {e}")

try:
    res = supabase.from_('cargas').select('*').limit(1).execute()
    print("La tabla 'cargas' existe.")
except Exception:
    print("La tabla 'cargas' no existe. Se recomienda crearla con columnas: id, fecha, archivo, filas_agregadas, estado.")

Cliente de Supabase inicializado.
La tabla 'variables' existe y es accesible.
La tabla 'cargas' existe.


# **Bloque 2**

In [2]:
# ============================================================
# BLOQUE 2: CONSTANTES
# ============================================================
COLUMN_DISPLAY_NAMES = [
    "ID", "Versión", "Proceso", "Eje", "Tema", "Nombre",
    "Institución", "Cobertura", "Periodicidad", "Liga Web", "Tipo de dato",
    "Fuente", "Año", "Valor"
]

DYNAMIC_OPTIONS_COLUMNS = [
    "version", "proceso", "eje", "tema",
    "institucion", "cobertura", "periodicidad", "tipo_de_dato"
]

EXPECTED_COLUMNS = [
    'id', 'version', 'proceso', 'eje', 'tema', 'nombre',
    'institucion', 'cobertura', 'periodicidad', 'liga_web', 'tipo_de_dato',
    'fuente', 'año', 'valor'
]

# ============================================================
# KEYWORDS – DEFINIDOS COMPLETAMENTE
# ============================================================
KEYWORDS = {
    'proceso': {
        'A. Prevención': [
            'existencia', 'mecanismos', 'cobertura', 'plan', 'programa',
            'profesionalización', 'capacitación', 'servicio civil', 'reglas',
            'lineamientos', 'diseño', 'atributos', 'calidad', 'indicadores',
            'objetivos', 'metas', 'participación ciudadana', 'contraloría social',
            'educación', 'campañas', 'comunicaciones', 'código de ética',
            'declaración patrimonial', 'evaluación de control y confianza',
            'personal', 'recursos humanos'
        ],
        'B. Detección': [
            'percibe', 'percepción', 'frecuencia', 'prevalencia', 'incidencia',
            'experimentaron', 'usuarios', 'unidades económicas', 'denuncias recibidas',
            'carpetas de investigación', 'averiguaciones previas', 'quejas recibidas',
            'actos de corrupción reportados', 'conocimiento por terceros',
            'inconformidades', 'observaciones', 'investigación iniciada', 'víctimas',
            'imputados', 'inculpados', 'causas penales ingresadas'
        ],
        'C. Sanción': [
            'sanciones', 'condenatorias', 'absolutorias', 'multas', 'inhabilitación',
            'destitución', 'amonestación', 'resarcitoria', 'pliego de observaciones',
            'procedimientos de responsabilidad', 'sentencias', 'fincamiento',
            'sanciones económicas', 'revocación', 'castigo', 'ejecución de sentencia',
            'cumplimiento de órdenes'
        ],
        'D. Fiscalización y control de recursos': [
            'auditoría', 'fiscalización', 'control interno', 'observaciones',
            'presupuesto', 'recursos', 'contratos', 'monto', 'donativos', 'compras',
            'obra pública', 'licitaciones', 'convocatorias', 'adjudicaciones',
            'proveedores', 'contratistas', 'cuenta pública', 'armonización contable',
            'estado de situación financiera', 'ingresos', 'egresos', 'fideicomisos'
        ]
    },
    'eje': {
        '1. Combatir la corrupción y la impunidad': [
            'quejas', 'denuncias', 'faltas administrativas', 'justicia', 'delitos',
            'cohecho', 'peculado', 'enriquecimiento', 'abuso de autoridad',
            'ejercicio indebido', 'procuración', 'impartición', 'ministerio público',
            'causas penales', 'sentencias', 'sanciones', 'responsabilidad administrativa',
            'investigación de servidores', 'tribunales', 'jueces', 'magistrados',
            'fiscalía', 'averiguación previa', 'carpeta de investigación'
        ],
        '2. Combatir la arbitrariedad y el abuso de poder': [
            'profesionalización', 'servicio civil', 'carrera', 'capacitación',
            'evaluación de desempeño', 'declaración patrimonial', 'planes anticorrupción',
            'programas sociales', 'procesos institucionales', 'indicadores',
            'reglas de operación', 'riesgos', 'auditoría', 'fiscalización',
            'control interno', 'obra pública', 'contrataciones', 'adquisiciones',
            'arrendamientos', 'proveedores', 'contratistas', 'testigos sociales',
            'planeación', 'programación', 'presupuestación', 'seguimiento',
            'evaluación', 'marco jurídico', 'transparencia', 'datos abiertos',
            'rendición de cuentas'
        ],
        '3. Promover la mejora de la gestión y los puntos de contacto gobierno-sociedad': [
            'trámites', 'servicios públicos', 'puntos de contacto', 'ciudadanía',
            'iniciativa privada', 'licitaciones', 'convocatorias', 'permisos',
            'pagos', 'predial', 'tenencia', 'atención ciudadana',
            'solicitudes de información', 'oficina de transparencia',
            'módulos de orientación', 'infracciones', 'licencias de conducir',
            'registro civil', 'afiliación', 'consulta médica', 'inscripción escolar',
            'compras del gobierno', 'contratación pública', 'MIPYMES',
            'competencia económica', 'prácticas monopólicas'
        ],
        '4. Involucrar a la sociedad y el sector privado': [
            'participación ciudadana', 'contraloría social', 'vigilancia',
            'colaboración', 'cocreación', 'educación', 'campañas anticorrupción',
            'comunicación', 'integridad empresarial', 'código de ética',
            'compliance', 'denuncias corporativas', 'testigos sociales',
            'observadores', 'organizaciones civiles', 'ONG', 'sindicatos',
            'medios de comunicación', 'redes sociales', 'parlamento abierto',
            'gobierno abierto', 'consulta ciudadana', 'presupuesto participativo'
        ]
    },
    'tema': {
        '1.1. Prevención, detección, denuncia, investigación, substanciación y sanción de faltas administrativas': [
            'quejas', 'denuncias', 'oficina de contraloría', 'procedimientos de responsabilidad',
            'declaración patrimonial', 'conflictos de interés', 'sistema informático para quejas',
            'buzón de quejas', 'atención ciudadana', 'faltas administrativas',
            'sanciones administrativas', 'amonestación', 'suspensión', 'destitución',
            'inhabilitación', 'responsabilidad resarcitoria'
        ],
        '1.2. Procuración e impartición de justicia en materia de delitos por hechos de corrupción': [
            'delitos', 'cohecho', 'peculado', 'enriquecimiento ilícito',
            'tráfico de influencias', 'abuso de autoridad', 'ejercicio indebido',
            'carpetas de investigación', 'averiguaciones previas', 'causas penales',
            'sentencias condenatorias', 'sentencias absolutorias', 'jueces',
            'magistrados', 'ministerio público', 'fiscalía', 'imputados', 'víctimas'
        ],
        '2.1. Profesionalización e integridad en el servicio público': [
            'servicio civil', 'carrera', 'capacitación', 'evaluación de control y confianza',
            'desempeño', 'competencias', 'personal', 'licenciatura', 'recursos humanos',
            'profesionalización', 'integridad', 'código de ética', 'declaración patrimonial',
            'evaluación de desempeño'
        ],
        '2.2. Procesos institucionales': [
            'plan anticorrupción', 'programas sociales', 'riesgos', 'indicadores',
            'metas', 'cobertura', 'desempeño', 'presupuesto', 'reglas de operación',
            'lineamientos', 'diseño', 'calidad', 'cumplimiento de metas', 'cuadrante',
            'targeting', 'instrumentos de medición', 'panel de control', 'seguimiento',
            'evaluación de diseño', 'consistencia y resultados'
        ],
        '2.3. Auditoría y fiscalización': [
            'auditoría', 'fiscalización', 'control interno', 'observaciones',
            'revisión', 'órgano de control', 'visitaduría', 'entidad de fiscalización',
            'armonización contable', 'ejercicio y control', 'seguimiento',
            'evaluación', 'indicadores de resultados', 'marco jurídico',
            'planeación', 'programación', 'presupuestación'
        ],
        '3.1. Puntos de contacto gobierno-ciudadanía: trámites, servicios y programas públicos': [
            'trámites', 'servicios', 'solicitudes de información', 'atención ciudadana',
            'pagos', 'predial', 'tenencia', 'permisos', 'construcción', 'licencias',
            'usuarios', 'confianza en información', 'módulos de orientación',
            'oficina de transparencia', 'solicitudes de acceso a la información',
            'datos abiertos', 'quejas de servicios', 'contacto con autoridades'
        ],
        '3.2. Puntos de contacto gobierno-iniciativa privada': [
            'contrataciones', 'licitaciones', 'obra pública', 'proveedores',
            'contratistas', 'MIPYMES', 'testigos sociales', 'convocatorias',
            'adquisiciones', 'arrendamientos', 'servicios relacionados',
            'prácticas monopólicas', 'competencia', 'registro de contratistas',
            'sistema electrónico de contrataciones', 'invitación restringida',
            'adjudicación directa'
        ],
        '4.1. Participación ciudadana: vigilancia, colaboración y cocreación': [
            'contraloría social', 'órganos de participación', 'consulta ciudadana',
            'vigilancia', 'presupuesto participativo', 'parlamento abierto',
            'gobierno abierto', 'mecanismos de participación', 'comités de contraloría',
            'consejos ciudadanos', 'rendición de cuentas', 'espacios de participación'
        ],
        '4.2. Corresponsabilidad e integridad empresarial': [
            'empresa', 'integridad', 'código de ética', 'compliance', 'proveedores',
            'denuncias corporativas', 'programa de integridad', 'anticorrupción',
            'políticas de sobornos', 'conflicto de intereses', 'transparencia corporativa'
        ],
        '4.3. Educación y comunicación para el control de la corrupción': [
            'educación', 'campañas', 'conocimiento cívico', 'estudiantes',
            'comunicación', 'redes sociales', 'medios de comunicación',
            'divulgación', 'sensibilización', 'normalización de la corrupción',
            'cultura de la legalidad', 'formación'
        ]
    }
}

# **Bloque 3**

In [3]:
# ============================================================
# BLOQUE 3: FUNCIONES AUXILIARES BÁSICAS
# ============================================================
def remover_acentos(s):
    if not s: return ""
    return "".join(c for c in unicodedata.normalize('NFKD', str(s)) if not unicodedata.combining(c))

def safe_clean(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return ""
    s = str(v).replace('\n', ' ').replace('\r', ' ').strip()
    s = re.sub(r'\s+', ' ', s)
    return s

def normalizar_texto(texto):
    if not texto:
        return ""
    return remover_acentos(str(texto).lower().strip())

def _clean_data_for_json(data):
    if isinstance(data, dict):
        return {k: _clean_data_for_json(v) for k, v in data.items()}
    elif isinstance(data, list):
        return [_clean_data_for_json(elem) for elem in data]
    elif pd.isna(data):
        return None
    return data

def _get_all_existing_ids():
    try:
        res = supabase.from_('variables').select('id').execute()
        return {str(item['id']) for item in res.data} if res.data else set()
    except Exception as e:
        print(f"Error al obtener IDs existentes: {e}")
        traceback.print_exc()
        return set()

def _get_all_variables_data():
    """Obtiene todos los datos de Supabase y normaliza números, pasando la columna."""
    try:
        res = supabase.from_('variables').select('*').order('id', desc=True).execute()
        if not res.data:
            return []
        data = []
        for item in res.data:
            row = {}
            for k, v in item.items():
                if k in ['version', 'año', 'valor']:
                    v = normalizar_numero_texto(v, k)
                row[k] = safe_clean(v)
            data.append(row)
        return data
    except Exception as e:
        print(f"Error al obtener datos de variables: {e}")
        traceback.print_exc()
        return []

def _get_non_id_hash(row_dict):
    """
    Genera un hash a partir de todas las columnas excepto 'id',
    aplicando normalización numérica a 'version', 'año' y 'valor'
    para garantizar consistencia.
    """
    relevant_keys = sorted([k for k in EXPECTED_COLUMNS if k != 'id'])
    values = []
    for key in relevant_keys:
        val = row_dict.get(key, '')
        if pd.isna(val):
            val = ''
        if key in ['version', 'año']:
            val_original = val
            val = normalizar_numero_texto(val, key)
            if key == 'año':
                print(f"🔍 [HASH] Año original: '{val_original}' -> normalizado: '{val}'")
        # <--- CAMBIO: Para 'valor', lo normalizamos a string con un decimal sin usar normalizar_numero_texto
        elif key == 'valor':
            try:
                # Convertir a float y luego a string con 1 decimal
                num = float(val)
                val = f"{num:.1f}"
            except (ValueError, TypeError):
                val = ''
        val = safe_clean(val)
        values.append(val)
    result_tuple = tuple(values)
    if len(values) > 0:
        print(f"🔹 Hash generado: {result_tuple[:3]}... (año: {values[relevant_keys.index('año')]})")
    return result_tuple

def generar_siguiente_id(existing_ids, used_ids_in_batch):
    try:
        nums = []
        for id_str in existing_ids.union(used_ids_in_batch):
            match = re.search(r'A-(\d+)', id_str)
            if match: nums.append(int(match.group(1)))
        proximo_num = max(nums) + 1 if nums else 1
        new_id = f"A-{proximo_num:05d}"
        while new_id in existing_ids or new_id in used_ids_in_batch:
            proximo_num += 1
            new_id = f"A-{proximo_num:05d}"
        return new_id
    except Exception as e:
        print(f"Error en generación secuencial, usando UUID: {e}")
        traceback.print_exc()
        new_id_uuid = f"A-{uuid.uuid4().hex[:5].upper()}"
        while new_id_uuid in existing_ids or new_id_uuid in used_ids_in_batch:
            new_id_uuid = f"A-{uuid.uuid4().hex[:5].upper()}"
        return new_id_uuid

def _normalize_col_for_matching(col_name):
    return remover_acentos(str(col_name).lower().replace(" ", "_")).replace("-", "_")

def _find_best_column_match(uploaded_col_normalized, available_expected_cols_normalized, threshold=0.8):
    best_match, highest_score = None, threshold
    for expected_col_norm in available_expected_cols_normalized:
        score = difflib.SequenceMatcher(None, uploaded_col_normalized, expected_col_norm).ratio()
        if score > highest_score:
            highest_score, best_match = score, expected_col_norm
    return best_match

def _get_catalog_options(column_key):
    try:
        res = supabase.from_(column_key).select('name').execute()
        return sorted({r['name'] for r in res.data if r.get('name')}) if res.data else []
    except Exception as e:
        print(f"Error al obtener opciones para '{column_key}': {e}")
        traceback.print_exc()
        return []

def _find_best_value_match(input_value, available_options, threshold=0.8):
    if not input_value or not available_options:
        return None, 0.0
    input_norm = remover_acentos(str(input_value).lower())
    best_match_val, highest_score = None, 0.0
    for option_original in available_options:
        option_norm = remover_acentos(str(option_original).lower())
        score = difflib.SequenceMatcher(None, input_norm, option_norm).ratio()
        if score > highest_score:
            highest_score, best_match_val = score, option_original
    return (best_match_val, highest_score) if highest_score >= threshold else (None, highest_score)

# **Bloque 4**

In [4]:
# ============================================================
# BLOQUE 4: NORMALIZACIÓN DE NÚMEROS Y CATÁLOGOS
# ============================================================
def normalizar_numero_texto(valor, columna=None):
    """
    Normaliza valores numéricos según la columna.
    - Para 'año' y 'version': convierte a entero (sin decimales).
    - Para 'valor': convierte a float con un decimal.
    - Para otras: lo deja como texto sin modificar.
    """
    if valor is None:
        return ""
    try:
        if isinstance(valor, (int, float)) and not pd.isna(valor):
            if columna == 'año' or columna == 'version':
                return str(int(valor))
            elif columna == 'valor':
                return f"{float(valor):.1f}"
            else:
                return str(valor)
        if isinstance(valor, str):
            try:
                num = float(valor.replace(',', ''))
                if columna == 'año' or columna == 'version':
                    return str(int(num))
                elif columna == 'valor':
                    return f"{num:.1f}"
                else:
                    return str(valor)
            except ValueError:
                return str(valor)
        return valor
    except (ValueError, TypeError):
        return valor

def format_numero(valor, columna=None):
    """Versión para visualización en frontend."""
    try:
        if isinstance(valor, (int, float)) and not pd.isna(valor):
            if columna == 'año':
                return str(int(valor))
            elif columna == 'valor':
                return f"{float(valor):.1f}"
            else:
                return str(valor)
        return valor
    except (ValueError, TypeError):
        return valor

def normalizar_valor_con_catalogo(valor, tabla, threshold=0.5):
    if not valor or not isinstance(valor, str):
        return valor
    valor_limpio = safe_clean(valor)
    if not valor_limpio:
        return valor

    sinonimos = {
        'institucion': {
            'inegi': 'Instituto Nacional de Estadística y Geografía',
            'secretaria de educacion': 'Secretaría de Educación Pública',
            'sep': 'Secretaría de Educación Pública',
        }
    }
    if tabla in sinonimos:
        valor_lower = valor_limpio.lower()
        for clave, canonico in sinonimos[tabla].items():
            if clave in valor_lower:
                return canonico

    opciones = _get_catalog_options(tabla)
    valor_norm = remover_acentos(valor_limpio.lower().strip())
    for opt in opciones:
        if remover_acentos(opt.lower().strip()) == valor_norm:
            return opt

    match, score = _find_best_value_match(valor_limpio, opciones, threshold)
    if match:
        return match

    try:
        res = supabase.from_(tabla).insert({"name": valor_limpio}).execute()
        if res.data:
            return valor_limpio
        else:
            return valor
    except Exception as e:
        print(f"Error al insertar '{valor_limpio}' en tabla '{tabla}': {e}")
        return valor

def normalizar_categoria(valor, tipo, debug=False):
    if not valor:
        return valor
    valor_limpio = safe_clean(valor)
    if not valor_limpio:
        return valor

    if tipo == 'proceso':
        opciones = list(KEYWORDS['proceso'].keys())
        palabras_por_opcion = KEYWORDS['proceso']
    elif tipo == 'eje':
        opciones = list(KEYWORDS['eje'].keys())
        palabras_por_opcion = KEYWORDS['eje']
    elif tipo == 'tema':
        opciones = list(KEYWORDS['tema'].keys())
        palabras_por_opcion = KEYWORDS['tema']
    else:
        return valor

    valor_norm = remover_acentos(valor_limpio.lower().strip())

    # 1. Coincidencia exacta
    for opt in opciones:
        if remover_acentos(opt.lower().strip()) == valor_norm:
            if debug:
                print(f"✅ Normalizado {tipo}: '{valor}' -> '{opt}'")
            return opt

    # 2. Coincidencia por palabras clave
    def texto_contiene_palabra_clave(texto, lista_palabras):
        texto_norm = remover_acentos(texto.lower())
        for palabra in lista_palabras:
            if remover_acentos(palabra.lower()) in texto_norm:
                return True
        return False

    for categoria, palabras in palabras_por_opcion.items():
        if texto_contiene_palabra_clave(valor_limpio, palabras):
            if debug:
                print(f"✅ Normalizado {tipo} por keyword: '{valor}' -> '{categoria}'")
            return categoria

    # 3. Coincidencia numérica
    if tipo == 'eje' and valor_limpio.startswith('Eje '):
        num_match = re.search(r'Eje\s*(\d+)', valor_limpio, re.IGNORECASE)
        if num_match:
            num = num_match.group(1)
            for opt in opciones:
                if opt.startswith(f'Eje {num}.'):
                    if debug:
                        print(f"✅ Normalizado {tipo}: '{valor}' -> '{opt}'")
                    return opt
    if tipo == 'tema' and re.match(r'^\d+\.', valor_limpio):
        num_match = re.match(r'^(\d+)\.', valor_limpio)
        if num_match:
            num = num_match.group(1)
            for opt in opciones:
                if opt.startswith(f'{num}.'):
                    if debug:
                        print(f"✅ Normalizado {tipo}: '{valor}' -> '{opt}'")
                    return opt

    # 4. Coincidencia difusa con umbral 0.5
    for opt in opciones:
        ratio = difflib.SequenceMatcher(None, remover_acentos(valor_limpio.lower()), remover_acentos(opt.lower())).ratio()
        if ratio > 0.5:
            if debug:
                print(f"⚠️ Normalizado {tipo} (difuso): '{valor}' -> '{opt}'")
            return opt

    if debug:
        print(f"❌ No se pudo normalizar {tipo}: '{valor}'")
    return valor

# **Bloque 5**

In [5]:
# ============================================================
# BLOQUE 5: PROCESAMIENTO DE CSV (CORREGIDO)
# ============================================================
def _process_csv_data(df_input, existing_supabase_data_full, debug=False):
    """Procesa el DataFrame, detecta duplicados y prepara previsualización."""
    messages = []
    df = df_input.copy()

    # ---- PASO EXTRA: Buscar columna de año por alias ----
    year_aliases = ['año', 'ano', 'anio', 'Año', 'AÑO', 'anho']
    for col in df.columns:
        col_clean = remover_acentos(str(col).lower().strip().replace(' ', '_'))
        if col_clean in year_aliases or col_clean == 'año':
            if col != 'año':
                df.rename(columns={col: 'año'}, inplace=True)
                messages.append(f"Columna '{col}' renombrada a 'año' por alias.")
            break

    # 1. Renombrar columnas (normalización estándar)
    all_known_normalized_keys = set(EXPECTED_COLUMNS)
    for display_name in COLUMN_DISPLAY_NAMES:
        all_known_normalized_keys.add(_normalize_col_for_matching(display_name))

    column_rename_map = {}
    processed_target_cols = set()
    for original_col in df.columns:
        normalized_original_col = _normalize_col_for_matching(original_col)
        target_col_name = None
        if normalized_original_col in EXPECTED_COLUMNS:
            target_col_name = normalized_original_col
            if original_col != target_col_name:
                messages.append(f"Columna renombrada: '{original_col}' a '{target_col_name}' (normalización).")
        else:
            fuzzy_match = _find_best_column_match(normalized_original_col, all_known_normalized_keys)
            if fuzzy_match and fuzzy_match in EXPECTED_COLUMNS:
                target_col_name = fuzzy_match
                messages.append(f"Columna renombrada: '{original_col}' a '{target_col_name}' (sugerencia).")
        if target_col_name:
            if target_col_name not in processed_target_cols:
                column_rename_map[original_col] = target_col_name
                processed_target_cols.add(target_col_name)
            else:
                messages.append(f"Columna '{original_col}' ignorada (ya mapeada a '{target_col_name}').")
        else:
            column_rename_map[original_col] = original_col

    df.rename(columns=column_rename_map, inplace=True)

    extra_cols = [col for col in df.columns if col not in EXPECTED_COLUMNS and col != 'id']
    if extra_cols:
        df.drop(columns=extra_cols, inplace=True)
        messages.append(f"Columnas ignoradas: {', '.join(extra_cols)}")

    missing_cols = [col for col in EXPECTED_COLUMNS if col not in df.columns]
    for col in missing_cols:
        df[col] = pd.NA
        messages.append(f"Columna añadida (faltante): '{col}'")

    df = df[EXPECTED_COLUMNS]

    # 2. Normalizar números en el CSV (pasando la columna)
    # Usamos una función auxiliar para evitar problemas con Series
    def safe_normalize(val, col_name):
        # Si es una Serie, extraer el primer elemento
        if isinstance(val, pd.Series):
            val = val.iloc[0] if len(val) > 0 else ''
        try:
            return normalizar_numero_texto(val, col_name)
        except:
            return str(val) if val is not None else ''

    # <--- CAMBIO: Solo normalizamos 'version' y 'año', no 'valor'
    for col in ['version', 'año']:
        if col in df.columns:
            df[col] = df[col].apply(lambda x: safe_normalize(x, col))

    # ---- FORZAR QUE 'año' SEA STRING SIN DECIMAL Y 'version' SEA '-' SI VACÍO ----
    if 'año' in df.columns:
        df['año'] = df['año'].apply(lambda x: str(int(float(x))) if pd.notna(x) and str(x).strip() != '' else '')
    if 'version' in df.columns:
        df['version'] = df['version'].apply(lambda x: '-' if pd.isna(x) or str(x).strip() == '' else str(x).strip())

    # 3. Aplicar ML a categorías vacías (si predict_categories está disponible)
    try:
        for index, row in df.iterrows():
            nombre = row.get('nombre', '')
            if pd.notna(nombre) and str(nombre).strip():
                if (pd.isna(row.get('proceso')) or row.get('proceso') == '') or \
                   (pd.isna(row.get('eje')) or row.get('eje') == '') or \
                   (pd.isna(row.get('tema')) or row.get('tema') == ''):
                    try:
                        proc, eje, tema = predict_categories(str(nombre))
                        if proc and (pd.isna(row.get('proceso')) or row.get('proceso') == ''):
                            df.at[index, 'proceso'] = proc
                        if eje and (pd.isna(row.get('eje')) or row.get('eje') == ''):
                            df.at[index, 'eje'] = eje
                        if tema and (pd.isna(row.get('tema')) or row.get('tema') == ''):
                            df.at[index, 'tema'] = tema
                    except NameError:
                        pass
    except NameError:
        pass

    # 4. Normalizar columnas catalogadas en el CSV (umbral 0.5)
    columnas_a_normalizar = ['institucion', 'cobertura', 'periodicidad', 'tipo_de_dato', 'proceso', 'eje', 'tema']
    for idx, row in df.iterrows():
        for col in columnas_a_normalizar:
            if col in row and pd.notna(row[col]) and row[col]:
                try:
                    if col in ['institucion', 'cobertura', 'periodicidad', 'tipo_de_dato']:
                        valor_normalizado = normalizar_valor_con_catalogo(row[col], col, threshold=0.5)
                    else:
                        valor_normalizado = normalizar_categoria(row[col], col, debug=debug)
                    if valor_normalizado != row[col]:
                        df.at[idx, col] = valor_normalizado
                        if debug:
                            print(f"🔄 CSV: {col} '{row[col]}' -> '{valor_normalizado}'")
                except Exception as e:
                    if debug:
                        print(f"⚠️ Error normalizando {col}: {e}")

    # 5. Normalizar datos existentes de Supabase (pasando columna)
    normalized_existing_data = []
    for db_row in existing_supabase_data_full:
        norm_row = db_row.copy()
        for col in ['institucion', 'cobertura', 'periodicidad', 'tipo_de_dato']:
            if col in norm_row and norm_row[col]:
                norm_row[col] = normalizar_valor_con_catalogo(norm_row[col], col, threshold=0.5)
        for col in ['proceso', 'eje', 'tema']:
            if col in norm_row and norm_row[col]:
                norm_row[col] = normalizar_categoria(norm_row[col], col, debug=debug)
        # <--- CAMBIO: Normalizamos 'version' y 'año', pero no 'valor' (lo dejamos numérico)
        for col in ['version', 'año']:
            if col in norm_row and norm_row[col]:
                norm_row[col] = safe_normalize(norm_row[col], col)
        # ---- FORZAR que año sea string sin decimal, version sea '-' si vacío ----
        if 'año' in norm_row and not pd.isna(norm_row['año']) and str(norm_row['año']).strip() != '':
            try:
                norm_row['año'] = str(int(float(norm_row['año'])))
            except:
                pass
        else:
            norm_row['año'] = ''
        if 'version' in norm_row and (pd.isna(norm_row['version']) or str(norm_row['version']).strip() == ''):
            norm_row['version'] = '-'
        normalized_existing_data.append(norm_row)

    # 6. Construir hashes de los datos existentes
    existing_hashes = {}
    for db_row in normalized_existing_data:
        h = _get_non_id_hash(db_row)
        if h not in existing_hashes:
            existing_hashes[h] = db_row

    # 7. Detectar duplicados en el CSV
    all_rows = []
    duplicate_count = 0
    duplicate_ids = []

    for idx, row_series in df.iterrows():
        csv_row = row_series.to_dict()
        csv_hash = _get_non_id_hash(csv_row)
        row_status = {
            'original_csv_index': idx,
            'data': csv_row,
            '_is_duplicate': False,
            '_supabase_matching_id': None,
            '_hash': csv_hash
        }
        if csv_hash in existing_hashes:
            matching = existing_hashes[csv_hash]
            row_status['_is_duplicate'] = True
            row_status['_supabase_matching_id'] = matching.get('id')
            duplicate_count += 1
            duplicate_ids.append(matching.get('id'))
        all_rows.append(row_status)

    if duplicate_count > 0:
        messages.append(f"Se encontraron {duplicate_count} filas duplicadas (IDs: {', '.join(duplicate_ids[:5])}{'...' if len(duplicate_ids)>5 else ''}).")
        if debug:
            messages.append(f"Hashes de duplicados: {[r['_hash'] for r in all_rows if r['_is_duplicate']]}")
    else:
        messages.append("No se encontraron duplicados.")
        if debug:
            messages.append("Hashes de todas las filas del CSV: " + str([r['_hash'] for r in all_rows]))

    # 8. Asignar IDs a nuevas filas
    existing_ids_from_db = _get_all_existing_ids()
    temp_ids = set(existing_ids_from_db)
    for row in all_rows:
        if not row['_is_duplicate']:
            new_id = generar_siguiente_id(temp_ids, set())
            temp_ids.add(new_id)
            row['data']['id'] = new_id

    print("🔹 Hashes de datos existentes (muestra):", list(existing_hashes.keys())[:5])
    print("🔸 Hashes del CSV (muestra):", [r['_hash'] for r in all_rows if not r['_is_duplicate']][:5])
    if all_rows:
        print("📋 Ejemplo de fila CSV normalizada:", all_rows[0]['data'])
    if existing_hashes:
        first_existing = next(iter(existing_hashes.values()))
        print("📋 Ejemplo de fila existente normalizada:", first_existing)

    return all_rows, messages

# **Bloque 6**

In [6]:
# ============================================================
# BLOQUE 6: PROCESAMIENTO DE CARPETAS – PARTE 1 (auxiliares)
# ============================================================

import csv
import os
import re
import unicodedata
import numpy as np
import pandas as pd

def normalize_string(s):
    if not isinstance(s, str):
        return ""
    s = remover_acentos(s)
    s = s.lower()
    s = re.sub(r'[^a-z0-9\s]', '', s)
    s = re.sub(r'\s+', '_', s).strip('_')
    return s

def parse_global_metadatos(metadatos_text):
    """
    Extrae metadatos del texto, priorizando campos estándar.
    El año se extrae exclusivamente del contenido de los metadatos (Title, Description, etc.).
    """
    global_meta = {
        'version': '',
        'proceso': '',
        'eje': '',
        'tema': '',
        'nombre': '',
        'institucion': '',
        'cobertura': '',
        'periodicidad': '',
        'liga_web': '',
        'tipo_de_dato': '',
        'fuente': '',
        'año': ''
    }

    # 1. Buscar Title (fuente principal)
    title_match = re.search(r'Title:\s*(.+)', metadatos_text, re.IGNORECASE)
    if title_match:
        global_meta['fuente'] = title_match.group(1).strip()
        year_match = re.search(r'\b(19|20)\d{2}\b', global_meta['fuente'])
        if year_match:
            global_meta['año'] = year_match.group(0)

    # 2. Si no se encontró año, buscar en todo el texto
    if not global_meta['año']:
        year_match = re.search(r'\b(19|20)\d{2}\b', metadatos_text)
        if year_match:
            global_meta['año'] = year_match.group(0)
            print(f"🔍 Año extraído de metadatos: {global_meta['año']}")

    # 3. Resto de campos
    description_match = re.search(r'Description:\s*(.+)', metadatos_text, re.IGNORECASE)
    if description_match:
        institucion_raw = description_match.group(1).strip()
        institucion_raw = re.sub(r'\b(19|20)\d{2}\b', '', institucion_raw).strip()
        global_meta['institucion'] = institucion_raw

    cobertura_found = False
    if title_match:
        title_value = title_match.group(1).strip().lower()
        if any(term in title_value for term in {'federal','federales'}):
            global_meta['cobertura'] = 'Federal'
            cobertura_found = True
        elif any(term in title_value for term in {'estatal','estatales'}):
            global_meta['cobertura'] = 'Estatal'
            cobertura_found = True
        elif any(term in title_value for term in {'municipal','municipales'}):
            global_meta['cobertura'] = 'Municipal'
            cobertura_found = True

    if not cobertura_found:
        spatial_match = re.search(r'Spatial:\s*(.+)', metadatos_text, re.IGNORECASE)
        if spatial_match:
            spatial_value = spatial_match.group(1).strip()
            if 'Estados Unidos Mexicanos' in spatial_value:
                global_meta['cobertura'] = 'Federal'
            else:
                global_meta['cobertura'] = spatial_value

    periodicidad_match = re.search(r'AccrualPeriodicity:\s*(.+)', metadatos_text, re.IGNORECASE)
    if periodicidad_match:
        global_meta['periodicidad'] = periodicidad_match.group(1).strip()

    distribution_match = re.search(r'Distribution:\s*(.+)', metadatos_text, re.IGNORECASE)
    if distribution_match:
        global_meta['liga_web'] = distribution_match.group(1).strip()

    if not global_meta['fuente'] and title_match:
        global_meta['fuente'] = title_match.group(1).strip()

    return global_meta

# Función auxiliar para procesar líneas de datos de una sección
def _process_section_data(data_lines, col_idx, desc_idx, section_description, target_list, log_messages):
    """Procesa las líneas de datos de una sección y las agrega a target_list."""
    if not data_lines:
        return
    try:
        reader = csv.reader(data_lines, delimiter=',', quotechar='"')
        row_count = 0
        for row in reader:
            if not row or len(row) <= max(col_idx, desc_idx):
                continue
            col_name = row[col_idx].strip() if col_idx < len(row) else ''
            if not col_name or col_name.isdigit() or 'NSS' in col_name or 'ND' in col_name:
                continue
            col_desc = row[desc_idx].strip() if desc_idx < len(row) else ''
            if not col_desc and section_description:
                col_desc = section_description
            unnamed_col_8 = row[7] if len(row) > 7 else ''
            row_dict = {
                'Nombre de la columna': col_name,
                'Descripcion': col_desc,
                '_seccion_desc': section_description,
                'UNNAMED_COL_8': unnamed_col_8
            }
            target_list.append(row_dict)
            row_count += 1
        if row_count == 0:
            log_messages.append(f"   ⚠️ No se encontraron filas de datos válidas en esta sección.")
    except Exception as e:
        log_messages.append(f"   ❌ Error al procesar datos: {e}")

def parse_dictionary_csv(file_path, format_type='old', log_messages=None):
    """
    Parsea archivos CSV de diccionario en formato antiguo o nuevo.
    - Formato nuevo: archivo con cabecera "Columna, Descripción, ..."
    - Formato antiguo: archivo con líneas "Archivo: nombre.dbf" y luego tabla.
    Ahora soporta múltiples secciones dentro del mismo CSV (formato antiguo).
    Para formato antiguo, procesa línea por línea con regex, sin depender de csv.reader
    para la detección de secciones.
    """
    if log_messages is None:
        log_messages = []

    if format_type not in ['old', 'new']:
        raise ValueError(f"Tipo de formato desconocido: {format_type}")

    # --- LECTURA DEL ARCHIVO CON MANEJO DE CODIFICACIÓN ---
    lines = None
    try:
        with open(file_path, 'r', encoding='utf-8-sig') as f:
            lines = f.readlines()
        log_messages.append(f"📄 Archivo leído con UTF-8: {file_path} ({len(lines)} líneas)")
    except UnicodeDecodeError:
        try:
            with open(file_path, 'r', encoding='latin-1') as f:
                lines = f.readlines()
            log_messages.append(f"📄 Archivo leído con Latin-1: {file_path} ({len(lines)} líneas)")
        except Exception as e:
            log_messages.append(f"❌ Error al leer archivo con Latin-1: {e}")
            return [] if format_type == 'new' else {}, None
    except Exception as e:
        log_messages.append(f"❌ Error al leer archivo '{file_path}': {e}")
        return [] if format_type == 'new' else {}, None

    if not lines:
        log_messages.append("⚠️ El archivo está vacío.")
        return [] if format_type == 'new' else {}, None

    # --- FORMATO NUEVO (sin cambios) ---
    if format_type == 'new':
        try:
            reader = csv.reader(lines, delimiter=',', quotechar='"')
            header = next(reader, None)
            if not header:
                log_messages.append("⚠️ No se encontró encabezado en formato nuevo.")
                return [], None
            col_idx = None
            desc_idx = None
            for i, h in enumerate(header):
                h_norm = remover_acentos(h.strip().lower()).replace(' ', '_')
                if h_norm in ['columna', 'nombre_de_la_columna', 'campo', 'variable']:
                    col_idx = i
                elif h_norm in ['descripcion', 'descripción', 'desc', 'definicion', 'definición']:
                    desc_idx = i
            if col_idx is None or desc_idx is None:
                log_messages.append(f"⚠️ No se encontraron columnas 'Columna' y 'Descripción' en el encabezado: {header}")
                return [], None
            definitions = []
            for row in reader:
                if len(row) > max(col_idx, desc_idx):
                    col_name = row[col_idx].strip()
                    desc = row[desc_idx].strip()
                    if col_name:
                        definitions.append({
                            'Nombre de la columna': col_name,
                            'Descripcion': desc
                        })
            log_messages.append(f"✅ Formato nuevo: {len(definitions)} definiciones encontradas.")
            return definitions, None
        except Exception as e:
            log_messages.append(f"❌ Error parsing nuevo formato: {e}")
            return [], None

    # --- FORMATO ANTIGUO (procesamiento línea por línea mejorado) ---
    else:
        log_messages.append("🔍 Procesando formato antiguo (múltiples secciones).")
        all_data_file_column_definitions = {}
        current_section = None
        current_description = ""
        data_lines = []
        header_found = False
        col_idx = 1
        desc_idx = 2

        i = 0
        while i < len(lines):
            line = lines[i].strip()
            clean_line = line.strip('"')

            # Detectar sección
            if 'Archivo:' in clean_line and '.dbf' in clean_line:
                # Si ya teníamos una sección abierta, procesarla
                if current_section is not None and data_lines:
                    all_data_file_column_definitions[current_section] = []
                    _process_section_data(data_lines, col_idx, desc_idx, current_description,
                                          all_data_file_column_definitions[current_section], log_messages)
                    data_lines = []
                    log_messages.append(f"✅ Sección '{current_section}' procesada.")

                # Extraer nombre del archivo
                import re
                match = re.search(r'Archivo:\s*([^\s.]+)\.dbf', clean_line, re.IGNORECASE)
                if match:
                    current_section = match.group(1).lower()
                    log_messages.append(f"🔎 Sección detectada: '{current_section}'")
                    # Extraer descripción
                    desc_match = re.search(r'\((.*?)\)', clean_line)
                    if desc_match:
                        current_description = desc_match.group(1).strip()
                    else:
                        after_dbf = re.sub(r'.*\.dbf', '', clean_line).strip()
                        current_description = after_dbf.strip(' ,') if after_dbf else ""
                    log_messages.append(f"   Descripción: '{current_description[:60]}{'...' if len(current_description)>60 else ''}'")
                    header_found = False
                    data_lines = []
                    i += 1
                    continue

            # Si estamos dentro de una sección, acumular líneas de datos
            if current_section is not None:
                # Buscar encabezado si no se ha encontrado
                if not header_found and ('Núm.de campo' in line or 'Nombre de la columna' in line):
                    header_found = True
                    try:
                        header_parts = next(csv.reader([line], delimiter=',', quotechar='"'))
                        for idx, h in enumerate(header_parts):
                            h_norm = remover_acentos(h.strip().lower()).replace(' ', '_')
                            if h_norm in ['nombre_de_la_columna', 'nombre de la columna', 'columna']:
                                col_idx = idx
                            elif h_norm in ['descripcion', 'descripción', 'desc']:
                                desc_idx = idx
                        log_messages.append(f"   Encabezado: columna={col_idx}, descripción={desc_idx}")
                    except:
                        log_messages.append(f"   ⚠️ Error parseando encabezado, usando índices por defecto (col=1, desc=2)")
                    i += 1
                    continue

                # Si no es encabezado y no está vacío, es una línea de datos
                if line and not line.startswith('"Censo') and not line.startswith('Censo') and line != ','*10:
                    if line.replace(',', '').strip():
                        data_lines.append(line)
                i += 1
            else:
                i += 1

        # Procesar la última sección
        if current_section is not None and data_lines:
            all_data_file_column_definitions[current_section] = []
            _process_section_data(data_lines, col_idx, desc_idx, current_description,
                                  all_data_file_column_definitions[current_section], log_messages)
            log_messages.append(f"✅ Sección '{current_section}' procesada.")

        # Limpiar secciones vacías
        to_remove = [k for k, v in all_data_file_column_definitions.items() if not v]
        for k in to_remove:
            del all_data_file_column_definitions[k]

        log_messages.append(f"✅ Total de secciones procesadas: {len(all_data_file_column_definitions)}")
        for key, defs in all_data_file_column_definitions.items():
            log_messages.append(f"   - {key}: {len(defs)} definiciones")

        if not all_data_file_column_definitions:
            log_messages.append("⚠️ No se encontraron definiciones. Verifica que el archivo tenga secciones con 'Archivo: nombre.dbf'.")

        return all_data_file_column_definitions, None

def infer_consecutive_descriptions(column_defs):
    processed_column_defs = []
    grouped_defs = {}

    for col_def in column_defs:
        col_name = col_def.get('Nombre de la columna', '')
        match = re.match(r'([a-zA-Z_]+?)(\d+)$', col_name)
        if match:
            base_name = match.group(1)
            suffix = int(match.group(2))
            if base_name not in grouped_defs:
                grouped_defs[base_name] = []
            grouped_defs[base_name].append((suffix, col_def))
        else:
            processed_column_defs.append(col_def)

    for base_name, definitions in grouped_defs.items():
        definitions.sort(key=lambda x: x[0])
        common_base_description_segment = ""

        for suffix, col_def_original in definitions:
            col_def = col_def_original.copy()
            current_raw_description = col_def.get('Descripcion', '').strip()

            differentiator_value = ""
            if 'Opciones' in col_def and col_def['Opciones'].strip():
                differentiator_value = col_def['Opciones'].strip()
            elif 'Categoría' in col_def and col_def['Categoría'].strip():
                differentiator_value = col_def['Categoría'].strip()
            elif 'UNNAMED_COL_8' in col_def and col_def['UNNAMED_COL_8'].strip():
                differentiator_value = col_def['UNNAMED_COL_8'].strip()

            if current_raw_description:
                cleaned_desc_for_current = current_raw_description.rstrip('.,;')
                if differentiator_value:
                    final_description_for_current = f"{cleaned_desc_for_current} {differentiator_value}"
                else:
                    final_description_for_current = cleaned_desc_for_current
                col_def['Descripcion'] = final_description_for_current

                if not common_base_description_segment:
                    common_base_description_segment = cleaned_desc_for_current
            else:
                if common_base_description_segment:
                    if differentiator_value:
                        final_description_for_current = f"{common_base_description_segment} {differentiator_value}"
                    else:
                        final_description_for_current = common_base_description_segment
                    col_def['Descripcion'] = final_description_for_current
                else:
                    col_def['Descripcion'] = ""

            processed_column_defs.append(col_def)

    return processed_column_defs

def _get_dictionary_files(dict_dir, new_format, old_dict_filename=None):
    if new_format:
        return [f for f in os.listdir(dict_dir) if f.endswith('.csv')]
    else:
        if old_dict_filename and os.path.exists(os.path.join(dict_dir, old_dict_filename)):
            return [old_dict_filename]
        else:
            return [f for f in os.listdir(dict_dir) if f.endswith('.csv')]

def _is_aggregation_candidate(col_nombre, col_descripcion, matched_df_column, new_format, custom_aggregation_keywords=None):
    if not matched_df_column:
        return False, "no se encontró una columna coincidente en el DataFrame"

    keywords_to_check = []
    keywords_for_error_msg = []

    if custom_aggregation_keywords is not None and len(custom_aggregation_keywords) > 0:
        keywords_to_check = [normalize_string(k) for k in custom_aggregation_keywords]
        keywords_for_error_msg = custom_aggregation_keywords
    else:
        if new_format:
            keywords_to_check = ['total', 'totales', 'cantidad', 'sum']
            keywords_for_error_msg = ['total', 'totales', 'cantidad', 'sum']
        else:
            keywords_to_check = [
                normalize_string('cantidad de'),
                normalize_string('cantidad'),
                normalize_string('total'),
                normalize_string('suma'),
                normalize_string('totales')
            ]
            keywords_for_error_msg = ['cantidad de', 'cantidad', 'total', 'suma', 'totales']

    # Normalizar el nombre de la columna del DataFrame (para revisar si contiene total)
    normalized_col_df = normalize_string(matched_df_column)
    if any(k in normalized_col_df for k in ['total', 'sum', 'cantidad']):
        return True, ""

    # Revisar nombre y descripción de la definición
    normalized_col_nombre = normalize_string(col_nombre)
    normalized_col_descripcion = normalize_string(col_descripcion)

    for term in keywords_to_check:
        if term in normalized_col_nombre or term in normalized_col_descripcion:
            return True, ""
    return False, f"no coincide con los criterios de suma para ({', '.join(keywords_for_error_msg)})"

def _get_output_nombre_value(col_nombre, col_descripcion, col_def, selected_data_file_name, new_format, indice_descriptions):
    """
    Genera el nombre de la variable.
    - Para formato antiguo: prioriza col_descripcion, luego _seccion_desc, luego col_nombre.
    - Para formato nuevo: usa el índice y la descripción.
    - Elimina prefijos comunes y no añade origen si hay descripción.
    """
    final_nombre_value = ""

    def limpiar_descripcion(desc):
        if not desc:
            return desc
        patrones = [
            r'^Contiene variables que caracterizan a la Administración Pública Federal de acuerdo con\s*',
            r'^Contiene variables que caracterizan a la Administración Pública Federal de acuerdo a\s*',
            r'^Contiene variables que caracterizan a la Administración Pública Federal según\s*',
            r'^Contiene variables que caracterizan a la Administración Pública Federal\s*',
            r'^Contiene variables que caracterizan\s*',
            r'^Contiene variables que\s*',
            r'^Variables que caracterizan\s*',
            r'^Descripción:\s*',
            r'^Definición:\s*',
        ]
        desc_limpia = desc
        for patron in patrones:
            desc_limpia = re.sub(patron, '', desc_limpia, flags=re.IGNORECASE)
        desc_limpia = desc_limpia.strip()
        if not desc_limpia:
            return desc
        desc_limpia = re.sub(r'^de acuerdo con\s*', '', desc_limpia, flags=re.IGNORECASE)
        desc_limpia = re.sub(r'^de acuerdo a\s*', '', desc_limpia, flags=re.IGNORECASE)
        desc_limpia = re.sub(r'^según\s*', '', desc_limpia, flags=re.IGNORECASE)
        if desc_limpia:
            desc_limpia = desc_limpia[0].upper() + desc_limpia[1:]
        return desc_limpia.strip()

    # Para formato antiguo, usar descripción como nombre principal
    if not new_format:
        # Primero intentar con la descripción directa
        desc_limpia = limpiar_descripcion(col_descripcion)
        if desc_limpia:
            final_nombre_value = desc_limpia
        else:
            # Si no hay descripción, usar la descripción de la sección
            seccion_desc = col_def.get('_seccion_desc', '').strip()
            if seccion_desc:
                seccion_limpia = limpiar_descripcion(seccion_desc)
                final_nombre_value = seccion_limpia if seccion_limpia else seccion_desc
            else:
                # Fallback: usar nombre de columna
                final_nombre_value = col_nombre

        # Añadir diferenciador si existe UNNAMED_COL_8 (solo si no hay descripción)
        unnamed_col_8 = col_def.get('UNNAMED_COL_8', '').strip()
        if unnamed_col_8 and final_nombre_value and not col_descripcion:
            final_nombre_value = f"{final_nombre_value} {unnamed_col_8}"

        # Si aún no hay nombre, usar col_nombre
        if not final_nombre_value:
            final_nombre_value = col_nombre

        # Solo agregar origen si no hay descripción (para no ensuciar)
        if not col_descripcion and not col_def.get('_seccion_desc'):
            final_nombre_value = f"{final_nombre_value} (origen: {selected_data_file_name})"

    else:
        # Formato nuevo (código existente)
        temp_name_without_ext = selected_data_file_name.split('.')[0].lower()
        base_file_name_for_lookup = temp_name_without_ext.split('_')[0] if '_' in temp_name_without_ext else temp_name_without_ext
        indice_content = indice_descriptions.get(base_file_name_for_lookup, '').strip()
        if indice_content:
            if col_descripcion:
                desc_limpia = limpiar_descripcion(col_descripcion)
                final_nombre_value = f"{indice_content}: {desc_limpia if desc_limpia else col_descripcion}"
            else:
                final_nombre_value = indice_content
        else:
            desc_limpia = limpiar_descripcion(col_descripcion)
            final_nombre_value = desc_limpia if desc_limpia else col_descripcion
        if not final_nombre_value:
            final_nombre_value = col_nombre

    # Limpieza final (eliminar "durante el año")
    if isinstance(final_nombre_value, str):
        final_nombre_value = re.sub(r',\s*durante el a\u00f1o\s*\d{4}\s*\.?\s*', '', final_nombre_value, flags=re.IGNORECASE)
        final_nombre_value = final_nombre_value.strip().strip(',').strip('.')

    return final_nombre_value

# **Bloque 7**

In [7]:
# ============================================================
# BLOQUE 7: PROCESAMIENTO DE CARPETAS – PARTE 2 (CORREGIDO CON LOGS MEJORADOS)
# ============================================================

import csv
import os
import re
import unicodedata
import numpy as np
import pandas as pd
from difflib import SequenceMatcher

# ---------- FUNCIONES AUXILIARES ACTUALIZADAS ----------

def normalize_string(s):
    """Normaliza eliminando todo excepto letras y números."""
    if not isinstance(s, str):
        return ""
    s = remover_acentos(s)
    s = s.lower()
    s = re.sub(r'[^a-z0-9]', '', s)
    return s

def _is_aggregation_candidate(col_nombre, col_descripcion, matched_df_column, new_format, custom_aggregation_keywords=None):
    """
    Determina si una columna es candidata a ser agregada (suma).
    Ahora revisa también el nombre de la columna del DataFrame.
    """
    if not matched_df_column:
        return False, "no se encontró una columna coincidente en el DataFrame"

    # --- NUEVO: Excluir si contiene "clasificación" ---
    col_nombre_lower = col_nombre.lower()
    col_desc_lower = col_descripcion.lower()
    if 'clasificación' in col_nombre_lower or 'clasificacion' in col_nombre_lower or \
       'clasificación' in col_desc_lower or 'clasificacion' in col_desc_lower:
        return False, "contiene 'clasificación', no se suma"

    keywords_to_check = []
    keywords_for_error_msg = []

    if custom_aggregation_keywords is not None and len(custom_aggregation_keywords) > 0:
        keywords_to_check = [normalize_string(k) for k in custom_aggregation_keywords]
        keywords_for_error_msg = custom_aggregation_keywords
    else:
        if new_format:
            keywords_to_check = ['total', 'totales', 'cantidad', 'sum']
            keywords_for_error_msg = ['total', 'totales', 'cantidad', 'sum']
        else:
            # Para formato antiguo, la clave principal es "cantidad de"
            keywords_to_check = [
                normalize_string('cantidad de'),
                normalize_string('cantidad'),
                normalize_string('total'),
                normalize_string('suma'),
                normalize_string('totales')
            ]
            keywords_for_error_msg = ['cantidad de', 'cantidad', 'total', 'suma', 'totales']

    # 1. Revisar el nombre de la columna del DataFrame (ej. totalca1, re_snn1)
    normalized_col_df = normalize_string(matched_df_column)
    if any(k in normalized_col_df for k in ['total', 'sum', 'cantidad']):
        return True, ""

    # 2. Revisar nombre y descripción de la definición del diccionario
    normalized_col_nombre = normalize_string(col_nombre)
    normalized_col_descripcion = normalize_string(col_descripcion)

    for term in keywords_to_check:
        if term in normalized_col_nombre or term in normalized_col_descripcion:
            return True, ""

    return False, f"no coincide con los criterios de suma para ({', '.join(keywords_for_error_msg)})"


# ---------- FUNCIÓN PRINCIPAL ACTUALIZADA ----------
def process_data_folders_with_return(data_dir, dict_dir, metadatos_path=None, new_format=False,
                                    indice_filename=None, aggregation_keywords=None,
                                    log_messages=None):
    if log_messages is None:
        log_messages = []

    if not os.path.isdir(dict_dir) or not os.path.isdir(data_dir):
        log_messages.append("❌ Error: No se encontraron las carpetas 'diccionario_de_datos' o 'conjunto_de_datos'.")
        return pd.DataFrame(columns=EXPECTED_COLUMNS + ['valor']), {}

    all_results = []
    dict_files = [f for f in os.listdir(dict_dir) if f.endswith('.csv')]
    log_messages.append(f"📂 Encontrados {len(dict_files)} archivos de diccionario en '{dict_dir}'.")
    if dict_files:
        log_messages.append(f"   Archivos: {', '.join(dict_files)}")

    if not dict_files:
        log_messages.append("❌ No se encontraron archivos CSV de diccionario.")
        return pd.DataFrame(columns=EXPECTED_COLUMNS + ['valor']), {}

    all_available_data_csv_files = [f for f in os.listdir(data_dir) if f.endswith('.csv')]
    log_messages.append(f"📂 Archivos de datos encontrados: {len(all_available_data_csv_files)}")
    if all_available_data_csv_files:
        log_messages.append(f"   Archivos: {', '.join(all_available_data_csv_files)}")

    data_dfs = {}
    for data_file_name in all_available_data_csv_files:
        data_csv_path = os.path.join(data_dir, data_file_name)
        try:
            try:
                df_temp = pd.read_csv(data_csv_path, encoding='utf-8-sig')
            except UnicodeDecodeError:
                df_temp = pd.read_csv(data_csv_path, encoding='latin-1')

            df_temp.columns = (
                df_temp.columns
                .str.replace(r'^\ufeff', '', regex=True)
                .str.strip()
                .str.lower()
            )

            data_dfs[data_file_name] = df_temp
            log_messages.append(f"  ✅ Dataset cargado: {data_file_name} ({len(df_temp.columns)} cols, {len(df_temp)} filas)")
            log_messages.append(f"     Columnas: {', '.join(df_temp.columns[:5])}{'...' if len(df_temp.columns) > 5 else ''}")
        except Exception as e:
            log_messages.append(f"  ⚠️ Error al cargar '{data_file_name}': {e}")
            continue

    current_metadatos_text = ""
    if metadatos_path and os.path.isfile(metadatos_path):
        try:
            with open(metadatos_path, 'r', encoding='utf-8') as f:
                current_metadatos_text = f.read()
            log_messages.append(f"📄 Metadatos cargados desde: {metadatos_path}")
        except Exception as e:
            log_messages.append(f"⚠️ Error al leer metadatos: {e} (posiblemente no es un archivo de texto)")

    global_metadata_dict = parse_global_metadatos(current_metadatos_text)
    log_messages.append("📊 Metadatos globales analizados.")
    log_messages.append(f"   Año extraído de metadatos: {global_metadata_dict.get('año', '')}")

    indice_descriptions = {}
    if new_format and indice_filename:
        indice_file_path_full = os.path.join(data_dir, indice_filename)
        try:
            indice_df = pd.read_csv(indice_file_path_full, encoding='utf-8')
            if 'ARCHIVO' in indice_df.columns and 'CONTENIDO' in indice_df.columns:
                indice_df['ARCHIVO_BASE'] = indice_df['ARCHIVO'].apply(lambda x: x.split('_')[0].lower() if isinstance(x, str) else '')
                indice_descriptions = dict(zip(indice_df['ARCHIVO_BASE'], indice_df['CONTENIDO']))
                log_messages.append(f"📑 Archivo de índice cargado: {len(indice_descriptions)} descripciones.")
            else:
                log_messages.append("⚠️ El índice no tiene columnas 'ARCHIVO' y 'CONTENIDO'.")
        except Exception as e:
            log_messages.append(f"⚠️ Error al cargar índice: {e}")

    # Función auxiliar robusta para normalizar cualquier valor
    def safe_normalize(val, col_name):
        if isinstance(val, pd.Series):
            if len(val) > 0:
                val = val.iloc[0]
            else:
                val = ''
        if isinstance(val, (list, np.ndarray)):
            if len(val) > 0:
                val = val[0]
            else:
                val = ''
        if pd.isna(val):
            val = ''
        try:
            return normalizar_numero_texto(val, col_name)
        except Exception:
            return str(val) if val is not None else ''

    # Función para limpiar y convertir a numérico (más robusta)
    def to_numeric_clean(series):
        if series.dtype == object:
            cleaned = series.astype(str).str.strip()

            def clean_number(val):
                if not val or val == '':
                    return np.nan
                val = val.replace(' ', '')
                if re.match(r'^[\d.]+$', val):
                    if val.count('.') == 1 and len(val.split('.')[1]) <= 2:
                        return val
                    else:
                        return val.replace('.', '')
                if ',' in val and '.' in val:
                    if val.rfind(',') > val.rfind('.'):
                        val = val.replace(',', '.')
                        parts = val.split('.')
                        if len(parts) > 2:
                            return ''.join(parts[:-1]) + '.' + parts[-1]
                        return val
                    else:
                        return val.replace(',', '')
                if ',' in val and '.' not in val:
                    parts = val.split(',')
                    if len(parts) == 2 and len(parts[1]) <= 2:
                        return val.replace(',', '.')
                    else:
                        return val.replace(',', '')
                if '.' in val and ',' not in val:
                    parts = val.split('.')
                    if len(parts) == 2 and len(parts[1]) <= 2:
                        return val
                    else:
                        return val.replace('.', '')
                return val

            cleaned = cleaned.apply(clean_number)
            numeric = pd.to_numeric(cleaned, errors='coerce')
            nan_count = numeric.isna().sum()
            if nan_count > 0:
                print(f"⚠️ {nan_count} valores no numéricos en la columna (ej: {series.head(3).tolist()})")
            return numeric
        else:
            return pd.to_numeric(series, errors='coerce')

    # ---------- PROCESAMIENTO PRINCIPAL ----------
    for dict_file_name in dict_files:
        dict_file_path = os.path.join(dict_dir, dict_file_name)
        try:
            format_type_param = 'new' if new_format else 'old'
            parsed_dict_content, _ = parse_dictionary_csv(dict_file_path, format_type=format_type_param, log_messages=log_messages)

            if new_format:
                current_dict_column_definitions = parsed_dict_content
                if not current_dict_column_definitions:
                    log_messages.append(f"  ⚠️ Diccionario {dict_file_name}: sin definiciones.")
                    continue
                current_dict_column_definitions = infer_consecutive_descriptions(current_dict_column_definitions)
                log_messages.append(f"  📋 Diccionario {dict_file_name}: {len(current_dict_column_definitions)} definiciones procesadas.")
                for col_def in current_dict_column_definitions:
                    col_nombre = col_def.get('Nombre de la columna', '')
                    col_descripcion = col_def.get('Descripcion', '')
                    found = False
                    for data_df_key, current_df in data_dfs.items():
                        matched_df_column = None
                        if col_nombre in current_df.columns:
                            matched_df_column = col_nombre
                        else:
                            normalized_col_nombre = normalize_string(col_nombre)
                            for df_col in current_df.columns:
                                if normalize_string(df_col) == normalized_col_nombre:
                                    matched_df_column = df_col
                                    break
                        if matched_df_column:
                            is_total, reason = _is_aggregation_candidate(
                                col_nombre, col_descripcion, matched_df_column, new_format, aggregation_keywords
                            )
                            if is_total:
                                log_messages.append(f"    🔍 Columna identificada: '{matched_df_column}' en archivo '{data_df_key}'")
                                sample_values = current_df[matched_df_column].head(3).tolist()
                                log_messages.append(f"    📋 Muestra de valores: {sample_values}")
                                numeric_series = to_numeric_clean(current_df[matched_df_column])
                                non_null_count = numeric_series.notna().sum()
                                total_sum = numeric_series.sum()
                                log_messages.append(f"    ➕ Elementos no nulos: {non_null_count}, Suma: {total_sum}")
                                if non_null_count == 0:
                                    log_messages.append(f"    ⚠️ La columna '{matched_df_column}' no contiene valores numéricos válidos después de la limpieza.")
                                output_row = global_metadata_dict.copy()
                                output_row['version'] = safe_normalize(output_row.get('version', ''), 'version')
                                if pd.isna(output_row['version']) or str(output_row['version']).strip() == '':
                                    output_row['version'] = '-'
                                if not output_row.get('año'):
                                    if output_row.get('fuente'):
                                        year_match = re.search(r'\b(19|20)\d{2}\b', output_row['fuente'])
                                        if year_match:
                                            output_row['año'] = year_match.group(0)
                                            log_messages.append(f"   Año extraído de fuente: {output_row['año']}")
                                    if not output_row.get('año') and data_df_key:
                                        year_match = re.search(r'(19|20)\d{2}', data_df_key)
                                        if year_match:
                                            output_row['año'] = year_match.group(0)
                                            log_messages.append(f"   Año extraído del nombre del dataset: {output_row['año']}")
                                    if not output_row.get('año') and dict_file_name:
                                        year_match = re.search(r'(19|20)\d{2}', dict_file_name)
                                        if year_match:
                                            output_row['año'] = year_match.group(0)
                                            log_messages.append(f"   Año extraído del nombre del diccionario: {output_row['año']}")
                                if output_row.get('año'):
                                    try:
                                        output_row['año'] = str(int(float(output_row['año'])))
                                    except:
                                        pass
                                else:
                                    output_row['año'] = ''
                                output_row['valor'] = total_sum if not pd.isna(total_sum) else None
                                output_row['nombre'] = _get_output_nombre_value(
                                    col_nombre, col_descripcion, col_def, data_df_key, new_format, indice_descriptions
                                )
                                output_row['tipo_de_dato'] = 'Numérico'
                                all_results.append(output_row)
                                found = True
                                break
                    if not found:
                        log_messages.append(f"    ⚠️ Columna '{col_nombre}' no procesada (no encontrada en ningún dataset).")
            else:
                # ----- FORMATO ANTIGUO (MEJORADO) -----
                if not parsed_dict_content:
                    log_messages.append(f"  ⚠️ Diccionario {dict_file_name}: sin definiciones.")
                    continue
                log_messages.append(f"  📋 Diccionario {dict_file_name}: contiene {len(parsed_dict_content)} secciones.")
                for key, defs in parsed_dict_content.items():
                    log_messages.append(f"    - {key}: {len(defs)} definiciones")

                for data_file_key, column_definitions_for_file in parsed_dict_content.items():
                    inferred_column_definitions_for_file = infer_consecutive_descriptions(column_definitions_for_file)
                    log_messages.append(f"  📋 Procesando archivo '{data_file_key}' con {len(inferred_column_definitions_for_file)} definiciones.")
                    selected_data_file_name = None
                    for actual_csv_name in data_dfs.keys():
                        base_name = os.path.splitext(actual_csv_name)[0].lower()
                        if base_name == data_file_key or actual_csv_name.startswith(data_file_key + '_') or actual_csv_name.startswith(data_file_key + '.'):
                            selected_data_file_name = actual_csv_name
                            break
                    if not selected_data_file_name:
                        log_messages.append(f"    ⚠️ No se encontró archivo de datos para '{data_file_key}'. Los archivos disponibles: {list(data_dfs.keys())}")
                        continue
                    current_df = data_dfs[selected_data_file_name]
                    log_messages.append(f"    🔍 Columnas en '{selected_data_file_name}': {', '.join(current_df.columns[:10])}{'...' if len(current_df.columns) > 10 else ''}")

                    for col_def in inferred_column_definitions_for_file:
                        col_nombre = col_def.get('Nombre de la columna', '')
                        col_descripcion = col_def.get('Descripcion', '')
                        matched_df_column = None

                        # --- NUEVA LÓGICA DE MATCHING MEJORADA ---
                        # 1. Coincidencia exacta (en minúsculas)
                        col_nombre_lower = col_nombre.lower()
                        if col_nombre_lower in current_df.columns:
                            matched_df_column = col_nombre_lower
                        else:
                            # 2. Coincidencia normalizada (eliminar todo excepto letras/números)
                            normalized_col_nombre = normalize_string(col_nombre)
                            found = False
                            for df_col in current_df.columns:
                                if normalize_string(df_col) == normalized_col_nombre:
                                    matched_df_column = df_col
                                    found = True
                                    break
                            if not found:
                                # 3. Coincidencia difusa con umbral 0.8
                                best_match = None
                                best_score = 0.8
                                for df_col in current_df.columns:
                                    score = SequenceMatcher(None, normalize_string(col_nombre), normalize_string(df_col)).ratio()
                                    if score > best_score:
                                        best_score = score
                                        best_match = df_col
                                if best_match:
                                    matched_df_column = best_match
                                    log_messages.append(f"    🔍 Coincidencia difusa: '{col_nombre}' -> '{best_match}' (score {best_score:.2f})")

                        if matched_df_column:
                            log_messages.append(f"    ✅ Columna encontrada: '{matched_df_column}' para definición '{col_nombre}'")
                            is_total, reason = _is_aggregation_candidate(
                                col_nombre, col_descripcion, matched_df_column, new_format, aggregation_keywords
                            )
                            if is_total:
                                log_messages.append(f"    🔍 Columna identificada: '{matched_df_column}' en archivo '{selected_data_file_name}'")
                                sample_values = current_df[matched_df_column].head(3).tolist()
                                log_messages.append(f"    📋 Muestra de valores: {sample_values}")
                                numeric_series = to_numeric_clean(current_df[matched_df_column])
                                non_null_count = numeric_series.notna().sum()
                                total_sum = numeric_series.sum()
                                log_messages.append(f"    ➕ Elementos no nulos: {non_null_count}, Suma: {total_sum}")
                                if non_null_count == 0:
                                    log_messages.append(f"    ⚠️ La columna '{matched_df_column}' no contiene valores numéricos válidos después de la limpieza.")
                                output_row = global_metadata_dict.copy()
                                output_row['version'] = safe_normalize(output_row.get('version', ''), 'version')
                                if pd.isna(output_row['version']) or str(output_row['version']).strip() == '':
                                    output_row['version'] = '-'
                                if not output_row.get('año'):
                                    if output_row.get('fuente'):
                                        year_match = re.search(r'\b(19|20)\d{2}\b', output_row['fuente'])
                                        if year_match:
                                            output_row['año'] = year_match.group(0)
                                            log_messages.append(f"   Año extraído de fuente: {output_row['año']}")
                                    if not output_row.get('año') and selected_data_file_name:
                                        year_match = re.search(r'(19|20)\d{2}', selected_data_file_name)
                                        if year_match:
                                            output_row['año'] = year_match.group(0)
                                            log_messages.append(f"   Año extraído del nombre del dataset: {output_row['año']}")
                                    if not output_row.get('año') and dict_file_name:
                                        year_match = re.search(r'(19|20)\d{2}', dict_file_name)
                                        if year_match:
                                            output_row['año'] = year_match.group(0)
                                            log_messages.append(f"   Año extraído del nombre del diccionario: {output_row['año']}")
                                if output_row.get('año'):
                                    try:
                                        output_row['año'] = str(int(float(output_row['año'])))
                                    except:
                                        pass
                                else:
                                    output_row['año'] = ''
                                output_row['valor'] = total_sum if not pd.isna(total_sum) else None
                                output_row['nombre'] = _get_output_nombre_value(
                                    col_nombre, col_descripcion, col_def, selected_data_file_name, new_format, indice_descriptions
                                )
                                output_row['tipo_de_dato'] = 'Numérico'
                                all_results.append(output_row)
                            else:
                                log_messages.append(f"    ⚠️ Columna '{col_nombre}' omitida: {reason}")
                        else:
                            log_messages.append(f"    ❌ No se encontró columna para '{col_nombre}'")

        except Exception as e:
            log_messages.append(f"❌ Error al procesar diccionario '{dict_file_name}': {e}")
            continue

    # ---------- CONSTRUCCIÓN DEL DATAFRAME FINAL ----------
    if not all_results:
        final_df = pd.DataFrame(columns=EXPECTED_COLUMNS + ['valor'])
        log_messages.append("⚠️ No se encontraron filas que cumplan los criterios de agregación.")
    else:
        final_df = pd.DataFrame(all_results)
        if 'valor' in final_df.columns:
            final_df['valor'] = pd.to_numeric(final_df['valor'], errors='coerce')
        for col in EXPECTED_COLUMNS:
            if col not in final_df.columns:
                final_df[col] = ''
        final_df = final_df[EXPECTED_COLUMNS + ['valor']]
        log_messages.append(f"✅ Se generaron {len(final_df)} filas.")

        if 'año' in final_df.columns:
            final_df['año'] = final_df['año'].apply(lambda x: str(int(float(x))) if pd.notna(x) and str(x).strip() != '' else '')
        if 'version' in final_df.columns:
            final_df['version'] = final_df['version'].apply(lambda x: '-' if pd.isna(x) or str(x).strip() == '' else str(x).strip())

        log_messages.append("🔄 Normalizando valores con catálogos existentes...")
        columnas_a_normalizar = ['institucion', 'cobertura', 'periodicidad', 'tipo_de_dato']
        for idx, row in final_df.iterrows():
            for col in columnas_a_normalizar:
                if col in row and pd.notna(row[col]) and row[col]:
                    valor_original = row[col]
                    valor_normalizado = normalizar_valor_con_catalogo(valor_original, col, threshold=0.5)
                    if valor_normalizado != valor_original:
                        final_df.at[idx, col] = valor_normalizado
                        log_messages.append(f"    Normalizado: '{valor_original}' -> '{valor_normalizado}' en columna '{col}'")
        columnas_opcionales = ['version', 'proceso', 'eje', 'tema']
        for idx, row in final_df.iterrows():
            for col in columnas_opcionales:
                if col in row and pd.notna(row[col]) and row[col]:
                    valor_original = row[col]
                    valor_normalizado = normalizar_categoria(valor_original, col, debug=False)
                    if valor_normalizado != valor_original:
                        final_df.at[idx, col] = valor_normalizado
                        log_messages.append(f"    Normalizado: '{valor_original}' -> '{valor_normalizado}' en columna '{col}'")

    return final_df, global_metadata_dict

# **Bloque 8**

In [8]:
# ============================================================
# BLOQUE 8: INTERFAZ COMPLETA (CSS, HTML, JS SEPARADOS)
# ============================================================

## **CSS**

In [9]:
CSS = """
/* ===== VARIABLES ===== */
:root {
    --primary: #4A5568;
    --primary-bright: #718096;
    --primary-dark: #2D3748;
    --bg: #F8F9FA;
    --panel-bg: #ffffff;
    --text-main: #212529;
    --text-muted: #6C757D;
    --border: #DEE2E6;
    --dark-header: #2C3E50;
    --table-header-bg: #2C3E50;
    --sort-neutral: #6C757D;
    --sort-active: #E72000;
    --selection-gray: #E9ECEF;
    --shadow: 0 2px 4px rgba(0,0,0,0.08);
    --danger: #dc3545;
    --success: #28a745;
}
* { box-sizing: border-box; }
body {
    font-family: 'Inter', sans-serif;
    margin: 0; padding: 0;
    background: var(--bg);
    color: var(--text-main);
    line-height: 1.6;
}
@keyframes fadeInSlideUp {
    from { opacity: 0; transform: translateY(30px); }
    to { opacity: 1; transform: translateY(0); }
}
@keyframes modalIn {
    from { transform: translateY(30px); opacity: 0; }
    to { transform: translateY(0); opacity: 1; }
}
.header-bar {
    background: var(--dark-header);
    color: white;
    padding: 16px 32px;
    display: flex;
    align-items: center;
    justify-content: space-between;
    box-shadow: 0 2px 6px rgba(0,0,0,0.1);
    opacity: 0;
    animation: fadeInSlideUp 0.9s ease-out forwards;
    animation-delay: 0.1s;
}
.header-bar h1 { margin:0; font-size:1.25em; letter-spacing:-0.02em; }
.container {
    margin: 24px auto;
    width: 95%;
    max-width: 2200px;
}
.tool-panel {
    background: var(--panel-bg);
    padding: 16px 20px;
    border-radius: 8px;
    border: 1px solid var(--border);
    box-shadow: var(--shadow);
    margin-bottom: 24px;
    display: flex;
    justify-content: space-between;
    align-items: center;
    gap: 12px;
    flex-wrap: wrap;
    opacity: 0;
    animation: fadeInSlideUp 0.9s ease-out forwards;
    animation-delay: 0.6s;
}
.tool-panel .search-wrapper {
    flex: 1 1 250px;
    min-width: 180px;
}
.tool-panel .search-wrapper input {
    width: 100%;
    padding: 10px 16px;
    border: 1px solid var(--border);
    border-radius: 6px;
    outline: none;
    font-size: 0.95em;
    background: white;
    transition: border-color 0.2s, box-shadow 0.2s;
}
.tool-panel .search-wrapper input:focus {
    border-color: var(--primary);
    box-shadow: 0 0 0 3px rgba(74,85,104,0.15);
}
.tool-panel .button-group {
    display: flex;
    gap: 8px;
    flex-wrap: wrap;
}
.btn {
    padding: 8px 18px;
    border-radius: 6px;
    border: 1px solid transparent;
    font-weight: 500;
    font-size: 0.85rem;
    transition: all 0.2s ease;
    box-shadow: 0 1px 3px rgba(0,0,0,0.08);
    letter-spacing: 0.01em;
    cursor: pointer;
    display: inline-flex;
    align-items: center;
    gap: 6px;
    background: white;
    color: var(--text-main);
}
.btn:hover {
    transform: translateY(-2px);
    box-shadow: 0 6px 14px rgba(0,0,0,0.12);
}
.btn-tool { background: var(--sort-active); color: #fff; border-color: var(--sort-active); }
.btn-tool:hover { background: #cc1c00; border-color: #cc1c00; }
.btn-action { background: var(--primary-bright); color: #fff; border-color: var(--primary-bright); }
.btn-action:hover { background: #4a5568; border-color: #4a5568; }
.btn-success { background: var(--success); color: #fff; border-color: var(--success); }
.btn-success:hover { background: #1e7e34; border-color: #1e7e34; }
.btn-outline { background: transparent; color: var(--text-main); border: 1px solid var(--border); }
.btn-outline:hover { background: var(--selection-gray); border-color: var(--primary-bright); }
.btn-danger { background: var(--danger); color: #fff; border-color: var(--danger); }
.btn-danger:hover { background: #bd2130; border-color: #bd2130; }
.btn:disabled { opacity: 0.6; cursor: not-allowed; transform: none !important; }

.table-card {
    background: var(--panel-bg);
    border-radius: 8px;
    border: 1px solid var(--border);
    box-shadow: 0 4px 12px -2px rgba(0,0,0,0.06);
    overflow: hidden;
    position: relative;
    opacity: 0;
    animation: fadeInSlideUp 0.9s ease-out forwards;
    animation-delay: 1.1s;
}
.table-card.drag-over { border: 2px dashed var(--primary); }
.table-card.drag-over::after {
    content: 'Soltar CSV para importar';
    position: absolute; top:0; left:0; width:100%; height:100%;
    background: rgba(255,255,255,0.85);
    z-index:100;
    display:flex; align-items:center; justify-content:center;
    font-weight:600; font-size:1.5em; color:var(--primary);
    pointer-events:none;
}
.table-wrapper {
    overflow-x: auto;
    -webkit-overflow-scrolling: touch;
}
table {
    border-collapse: collapse;
    width: 100%;
    min-width: 800px;
}
th {
    background: var(--table-header-bg);
    color: #F8FAFC !important;
    font-weight: 600;
    font-size: 0.8em;
    text-transform: uppercase;
    letter-spacing: 0.05em;
    border-bottom: 1px solid #334155;
    padding: 10px 14px;
    word-wrap: break-word;
    white-space: normal;
    position: sticky;
    top: 0;
    z-index: 10;
    text-align: center;
    user-select: none;
}
.th-content {
    display: flex;
    justify-content: space-between;
    align-items: center;
    width: 100%;
}
.th-label {
    flex: 1;
    text-align: left;
    white-space: nowrap;
    overflow: hidden;
    text-overflow: ellipsis;
}
.sort-indicators {
    display: flex;
    flex-direction: column;
    align-items: center;
    flex-shrink: 0;
    margin-left: 6px;
    line-height: 0.6;
    font-size: 0.7em;
}
.tri-up, .tri-down {
    color: var(--sort-neutral);
    transition: color 0.1s, transform 0.1s;
    cursor: pointer;
    padding: 1px 3px;
    user-select: none;
}
.tri-up:hover, .tri-down:hover { color: var(--sort-active); transform:scale(1.1); }
th.sorted-asc .tri-up { color: var(--sort-active) !important; text-shadow: 0 0 8px rgba(231,32,0,0.4); }
th.sorted-desc .tri-down { color: var(--sort-active) !important; text-shadow: 0 0 8px rgba(231,32,0,0.4); }

td {
    vertical-align: middle;
    padding: 2px 4px !important;
    min-height: 28px !important;
    box-sizing: border-box;
    word-break: break-word;
    white-space: normal;
    overflow-wrap: break-word;
    text-align: left !important;
}
td:first-child {
    text-align: center !important;
    width: 40px !important;
    min-width: 40px !important;
    max-width: 40px !important;
}
td:first-child input[type="checkbox"] {
    margin: 0;
}

/* Controles dentro de las celdas */
.edit-control, .read-only-input {
    width: 100%;
    min-height: 28px !important;
    padding: 2px 4px !important;
    font-size: 0.8em !important;
    font-family: inherit;
    border: 1px solid transparent;
    background: transparent;
    outline: none;
    line-height: 1.4;
    box-sizing: border-box;
    transition: border-color 0.2s, box-shadow 0.2s;
    color: #212529;
    vertical-align: top;
    text-align: left;
}
textarea.edit-control {
    height: auto;
    resize: none;
    overflow: hidden;
    white-space: pre-wrap;
    word-break: break-word;
}
textarea.edit-control:focus {
    border-color: var(--primary) !important;
    background: white !important;
    box-shadow: 0 0 0 3px rgba(74,85,104,0.15) !important;
}
select.edit-control {
    height: 28px !important;
    padding: 2px 4px !important;
    font-size: 0.8em !important;
    appearance: none;
    -webkit-appearance: none;
    cursor: pointer;
    overflow: visible;
    text-align-last: left;
    width: 100%;
    text-overflow: clip;
    white-space: nowrap;
}
select.edit-control option {
    white-space: nowrap;
    overflow: visible;
}
.read-only-input {
    background: transparent;
    border: none;
    cursor: default;
    text-align: left;
}
td.cell-focused {
    box-shadow: 0 4px 12px rgba(0,0,0,0.3);
    z-index: 2;
    position: relative;
}
tr:hover td { background-color: var(--selection-gray); }
tr.selected td { background-color: var(--selection-gray) !important; }

.row-checkbox {
    width: 40px;
    min-width: 40px;
    max-width: 40px;
    text-align: center;
    vertical-align: middle;
    padding: 0 4px;
}
.row-checkbox input[type="checkbox"] {
    cursor: pointer;
    width: 16px;
    height: 16px;
    accent-color: var(--primary);
}

.custom-file-upload {
    display: inline-block;
    padding: 10px 20px;
    cursor: pointer;
    background: var(--primary-bright);
    color: white !important;
    border-radius: 6px;
    border: none;
    font-weight: 500;
    font-size: 0.9em;
    transition: background 0.2s;
}
.custom-file-upload:hover { background: #4a5568; }
.custom-file-upload input[type="file"] { display: none; }
.file-input-wrapper { margin: 10px 0; }

.modal-overlay {
    position: fixed; top:0; left:0; width:100%; height:100%;
    background: rgba(0,0,0,0.5);
    display: none;
    align-items: center;
    justify-content: center;
    z-index: 1000;
    backdrop-filter: blur(4px);
}
.modal-content {
    background: white;
    padding: 32px 40px;
    border-radius: 12px;
    width: 90%;
    max-width: 825px;
    box-shadow: 0 20px 40px -8px rgba(0,0,0,0.3);
    animation: modalIn 0.3s cubic-bezier(0.16,1,0.3,1);
    border: 1px solid var(--border);
    max-height: 90vh;
    overflow-y: auto;
    position: relative;
}
.modal-header {
    margin-bottom: 20px;
    border-bottom: 1px solid var(--border);
    padding-bottom: 14px;
    position: relative;
}
.modal-header h2 { margin:0; font-size:1.5em; color:var(--text-main); font-weight:600; }
.close-button {
    position: absolute;
    top: 12px;
    right: 16px;
    background: none;
    border: none;
    font-size: 1.6em;
    cursor: pointer;
    color: var(--text-muted);
    transition: color 0.2s;
    line-height: 1;
    padding: 0 4px;
}
.close-button:hover { color: var(--primary-dark); }
.form-group { margin-bottom: 18px; }
.form-group label {
    display: block;
    font-size: 0.8em;
    font-weight: 600;
    color: var(--text-muted);
    margin-bottom: 6px;
    text-transform: uppercase;
    letter-spacing: 0.04em;
}
.form-input {
    width: 100%;
    padding: 10px 14px;
    border: 1px solid var(--border);
    border-radius: 6px;
    font-family: inherit;
    font-size: 1em;
    background: #f8fafa;
    transition: border-color 0.2s, box-shadow 0.2s;
    line-height: 1.5;
    resize: none;
    overflow: hidden;
    color: #212529;
}
textarea.form-input { min-height: 32px; overflow-y:hidden; }
.form-input:focus {
    border-color: var(--primary);
    background: white;
    box-shadow: 0 0 0 3px rgba(74,85,104,0.15);
}
.form-input::placeholder { color: #adb5bd; }
#toast-container {
    position: fixed; bottom:30px; right:30px; z-index:9999;
    display: flex;
    flex-direction: column;
    gap: 8px;
}
.toast {
    background: var(--dark-header);
    color: white;
    padding: 12px 24px;
    border-radius: 6px;
    box-shadow: 0 4px 12px rgba(0,0,0,0.15);
    animation: fadeInSlideUp 0.3s ease-out;
}

/* -------- SECCIÓN DE PREVISUALIZACIÓN (MODIFICADA) -------- */
#uploadPreviewMessages {
    max-height:120px; overflow-y:auto; margin-bottom:15px; font-size:0.9em;
    color:var(--text-muted); border:1px solid var(--border); padding:10px;
    border-radius:6px; background:#fdfdfd;
}
#uploadPreviewMessages div { margin-bottom:4px; line-height:1.4; color:var(--text-main); padding:2px 0; }

#uploadDataPreview, #zipPreviewTable {
    border-radius: 8px;
    border: 1px solid var(--border);
    overflow: auto;
    box-shadow: 0 2px 6px rgba(0,0,0,0.05);
    max-height: 300px;
    margin-bottom: 15px;
}
#uploadDataPreview table, #zipPreviewTable table {
    width: 100%;
    border-collapse: collapse;
    font-size: 0.85rem;
    table-layout: fixed;            /* Volvemos a fixed para respetar los anchos */
}
#uploadDataPreview th, #zipPreviewTable th {
    background: var(--table-header-bg);
    color: #f8fafc;
    font-weight: 600;
    padding: 8px 10px;
    border-bottom: 2px solid #334155;
    text-transform: uppercase;
    letter-spacing: 0.04em;
    position: sticky;
    top: 0;
    z-index: 2;
    text-align: left;
    vertical-align: top;
    white-space: nowrap;
    overflow: hidden;
    text-overflow: ellipsis;
}
#uploadDataPreview td, #zipPreviewTable td {
    padding: 2px 4px !important;
    border-bottom: 1px solid var(--border);
    vertical-align: top;
    text-align: left !important;
    word-break: break-word;
    white-space: normal;
    overflow-wrap: break-word;
    /* El ancho se fija desde el JS */
}
#uploadDataPreview tbody tr:nth-child(even), #zipPreviewTable tbody tr:nth-child(even) {
    background: #f8fafc;
}
#uploadDataPreview tbody tr:hover, #zipPreviewTable tbody tr:hover {
    background: var(--selection-gray);
}
.duplicate-row-preview {
    background: #fff5f5 !important;
    border-left: 4px solid var(--danger);
}
.duplicate-row-preview td {
    background: transparent !important;
}

/* Controles de previsualización: textarea, select y span */
#uploadDataPreview textarea.preview-edit,
#zipPreviewTable textarea.preview-edit {
    width: 100%;
    min-height: 28px;
    padding: 2px 4px;
    border: none !important;
    background: transparent !important;
    font-family: inherit;
    font-size: 0.8em;
    color: #212529;
    outline: none;
    resize: none;              /* ← ELIMINADO el control de redimensionamiento manual */
    overflow: hidden;          /* Oculta el scroll, se expande con el contenido */
    white-space: pre-wrap;
    word-break: break-word;
    box-shadow: none;
    cursor: default;
    text-align: left;
    vertical-align: top;
    box-sizing: border-box;
    display: block;
    height: auto;              /* Para que crezca automáticamente */
}
#uploadDataPreview select.preview-edit,
#zipPreviewTable select.preview-edit,
#uploadDataPreview span.preview-edit,
#zipPreviewTable span.preview-edit {
    width: 100%;
    min-height: 28px;
    padding: 2px 4px;
    border: none !important;
    background: transparent !important;
    font-family: inherit;
    font-size: 0.8em;
    color: #212529;
    outline: none;
    resize: none;
    overflow: hidden;
    white-space: pre-wrap;
    word-break: break-word;
    box-shadow: none;
    cursor: default;
    text-align: left;
    vertical-align: top;
    box-sizing: border-box;
    display: block;
}
#uploadDataPreview select.preview-edit,
#zipPreviewTable select.preview-edit {
    appearance: none;
    -webkit-appearance: none;
    cursor: pointer;
    padding-right: 16px;
    background: transparent;
}
/* -------- FIN SECCIÓN DE PREVISUALIZACIÓN -------- */

#paginationControls {
    display:flex; justify-content:center; align-items:center; gap:10px;
    margin-top:10px; margin-bottom:15px;
}
#paginationControls .btn {
    padding:6px 14px;
    border-radius:6px;
    font-size:0.85em;
    background:white;
    border:1px solid var(--border);
    color:var(--text-main);
}
#paginationControls .btn:disabled {
    background:#f1f3f5;
    color:#adb5bd;
    cursor:not-allowed;
    transform:none;
    box-shadow:none;
}
.table-pagination {
    display:flex; justify-content:center; align-items:center; gap:12px;
    padding:14px 20px;
    background:white;
    border-top:1px solid var(--border);
    border-bottom-left-radius:8px;
    border-bottom-right-radius:8px;
    flex-wrap:wrap;
}
.table-pagination button {
    padding:6px 16px;
    border-radius:6px;
    border:1px solid var(--border);
    background:white;
    cursor:pointer;
    font-weight:500;
    font-size:0.85em;
    transition:all 0.2s;
}
.table-pagination button:hover:not(:disabled) {
    background:var(--selection-gray);
    border-color:var(--primary-bright);
}
.table-pagination button:disabled {
    color:var(--text-muted);
    cursor:not-allowed;
    background:#f8f9fa;
}
.table-pagination span {
    font-size:0.9em;
    color:var(--text-main);
    display:flex;
    align-items:center;
    gap:6px;
}
.table-pagination input[type="number"] {
    width:56px;
    padding:6px 8px;
    border:1px solid var(--border);
    border-radius:6px;
    text-align:center;
    font-size:0.9em;
    background:white;
}
.table-pagination select {
    padding:6px 10px;
    border:1px solid var(--border);
    border-radius:6px;
    font-size:0.9em;
    background:white;
    min-width:65px;
}
#historyContainer {
    margin-top: 24px;
    background: var(--panel-bg);
    border-radius: 8px;
    border: 1px solid var(--border);
    padding: 16px 20px;
    box-shadow: var(--shadow);
}
#historyContainer h3 {
    margin-top: 0;
    margin-bottom: 12px;
    color: var(--text-main);
    font-size: 1.1em;
    font-weight: 600;
}
#historyList {
    max-height: 200px;
    overflow-y: auto;
    font-size: 0.9em;
}
#historyList table {
    width: 100%;
    border-collapse: collapse;
}
#historyList th, #historyList td {
    padding: 8px 12px;
    border-bottom: 1px solid var(--border);
    text-align: center;
}
#historyList th {
    background: var(--table-header-bg);
    color: #F8FAFC;
    font-weight: 600;
}
.status-success { color: var(--success); font-weight: 600; }
.status-error { color: var(--danger); font-weight: 600; }
.status-pending { color: #ffc107; font-weight: 600; }
.progress-bar {
    width: 100%;
    height: 20px;
    background: #e9ecef;
    border-radius: 6px;
    overflow: hidden;
    margin: 10px 0;
}
.progress-bar-fill {
    height: 100%;
    background: var(--sort-active);
    width: 0%;
    transition: width 0.5s ease;
}
@media (max-width: 768px) {
    .tool-panel {
        flex-direction: column;
        align-items: stretch;
    }
    .tool-panel .search-wrapper { flex: 1 1 auto; }
    .tool-panel .button-group { justify-content: center; }
    .modal-content { padding: 24px 20px; }
    .table-pagination { flex-wrap: wrap; gap: 8px; }
}
"""

## **HTML**

In [10]:
HTML_BODY = """
<!-- HEADER -->
<div class='header-bar'>
    <h1>Gestor de Variables <span style='font-weight:300; opacity:0.6'>| SESNA</span></h1>
    <div id='status-indicator' style='font-size:0.8em; color:#10B981'>● Sistema Activo</div>
</div>

<!-- MODALES -->
<div class='modal-overlay' id='modalOtraOpcion'>
    <div class='modal-content' style='max-width:450px;'>
        <div class='modal-header'>
            <h2 id='modalOtraTitle'>Nueva Opción</h2>
            <button class='close-button' onclick='handleCloseOtraOpcionModal()'>&times;</button>
        </div>
        <div class='form-group'>
            <label id='modalOtraLabel'>Ingrese el nuevo valor</label>
            <textarea id='otraOpcionInput' class='form-input' oninput='autoResizeModal(this)'></textarea>
        </div>
        <div style='display:flex; gap:12px; margin-top:24px;'>
            <button class='btn btn-outline' style='flex:1;' onclick='handleCloseOtraOpcionModal()'>Cerrar</button>
            <button class='btn btn-action' style='flex:1' id='btnConfirmOtra'>Agregar</button>
        </div>
    </div>
</div>

<div class='modal-overlay' id='modalSuggestion'>
    <div class='modal-content' style='max-width:450px;'>
        <div class='modal-header'>
            <h2>Sugerencia</h2>
            <button class='close-button' onclick='closeModal("modalSuggestion")'>&times;</button>
        </div>
        <p id='suggestionMessage'></p>
        <div style='display:flex; gap:12px; margin-top:24px;'>
            <button class='btn btn-outline' style='flex:1;' id='btnDeclineSuggestion'>Usar mi valor</button>
            <button class='btn btn-action' style='flex:1' id='btnAcceptSuggestion'>Usar sugerencia</button>
        </div>
    </div>
</div>

<!-- MODAL CARGAR CSV -->
<div class='modal-overlay' id='modalConfirmUpload'>
    <div class='modal-content'>
        <div class='modal-header'>
            <h2>Cargar CSV</h2>
            <button class='close-button' onclick='closeModal("modalConfirmUpload")'>&times;</button>
        </div>
        <div>
            <p><strong>Instrucciones:</strong> Selecciona un archivo CSV con el siguiente formato:</p>
            <ul>
                <li>Debe contener las columnas: <code>ID, Versión, Proceso, Eje, Tema, Nombre, Institución, Cobertura, Periodicidad, Liga Web, Tipo de dato, Fuente, Año, Valor</code>.</li>
                <li>La columna <code>Año</code> debe tener el año correspondiente (ej. 2018).</li>
                <li>El sistema detectará automáticamente duplicados y te permitirá sobrescribirlos.</li>
            </ul>
            <div class='file-input-wrapper'>
                <label class='custom-file-upload'>
                    <input type='file' id='csvFileInputModal' accept='.csv' onchange='handleFileUploadModal(this)'>
                    Seleccionar archivo CSV
                </label>
            </div>
            <div id='uploadProgress' style='display:none; margin-top:10px;'>
                <div class='progress-bar'>
                    <div id='uploadProgressFill' class='progress-bar-fill' style='width:0%;'></div>
                </div>
                <span id='uploadProgressText'>Procesando...</span>
            </div>
            <div id='uploadPreviewMessages' style='max-height:120px; overflow-y:auto; margin-top:10px;'></div>
            <div id='uploadDataPreview' style='display:none; margin-top:15px;'></div>
            <div id='paginationControls' style='display:none; justify-content:center; gap:10px; margin-top:10px;'>
                <button class='btn btn-outline' id='prevPageBtn' disabled>Anterior</button>
                <span id='pageInfo'>Página 1 de 1</span>
                <button class='btn btn-outline' id='nextPageBtn' disabled>Siguiente</button>
            </div>
            <div style='display:flex; gap:12px; margin-top:24px;'>
                <button class='btn btn-danger' style='flex:1;' onclick='closeModal("modalConfirmUpload")'>Cancelar</button>
                <button class='btn btn-success' style='flex:1' id='confirmUploadBtn' onclick='confirmUpload()' disabled>✓ Confirmar Carga</button>
            </div>
        </div>
    </div>
</div>

<!-- MODAL CARGAR CARPETA (ZIP) -->
<div class='modal-overlay' id='modalFolderUpload'>
    <div class='modal-content'>
        <div class='modal-header'>
            <h2>Cargar Carpeta (ZIP)</h2>
            <button class='close-button' onclick='closeModal("modalFolderUpload")'>&times;</button>
        </div>
        <div>
            <p><strong>Instrucciones:</strong> Selecciona un archivo ZIP que contenga la siguiente estructura:</p>
            <ul>
                <li><code>conjunto_de_datos/</code> (con archivos CSV de datos)</li>
                <li><code>diccionario_de_datos/</code> (con archivos CSV de diccionario, formato antiguo o nuevo)</li>
                <li><code>metadatos/</code> (opcional, con archivo <code>metadatos.txt</code>)</li>
            </ul>
            <p>El sistema detectará automáticamente el formato del diccionario.</p>
            <div class='file-input-wrapper'>
                <label class='custom-file-upload'>
                    <input type='file' id='zipFileInput' accept='.zip' onchange='handleZipUpload(this)'>
                    Seleccionar archivo ZIP
                </label>
            </div>
            <div id='zipProgress' style='display:none; margin-top:10px;'>
                <div class='progress-bar'>
                    <div id='zipProgressFill' class='progress-bar-fill' style='width:0%;'></div>
                </div>
                <span id='zipProgressText'>Procesando...</span>
            </div>
            <div id='folderLogMessages' style='max-height:250px; overflow-y:auto; background:#f8f9fa; padding:10px; border-radius:6px; border:1px solid var(--border); margin-top:10px; display:none;'>
                <pre id='folderLogContent' style='margin:0; font-size:0.85em; white-space:pre-wrap; word-break:break-word;'></pre>
            </div>
            <div id='zipPreviewContainer' style='display:none; margin-top:20px;'>
                <h3>Previsualización de datos extraídos</h3>
                <p style='font-size:0.85em; color:var(--danger); font-weight:500;'>Filas en rojo = duplicados. Las no duplicadas se insertarán.</p>
                <div style='margin-bottom:10px;'>
                    <input type='checkbox' id='selectAllDuplicatesForOverwriteZip'>
                    <label for='selectAllDuplicatesForOverwriteZip'>Marcar todos los duplicados para sobrescribir (no marcados = ignorar)</label>
                </div>
                <p>Filas a agregar: <span id='rowCount'></span></p>
                <div id='zipPreviewTable'></div>
                <!-- CONTROLES DE PAGINACIÓN PARA ZIP -->
                <div id='zipPaginationControls' style='display:none; justify-content:center; gap:10px; margin-top:10px;'>
                    <button class='btn btn-outline' id='zipPrevPageBtn' disabled>Anterior</button>
                    <span id='zipPageInfo'>Página 1 de 1</span>
                    <button class='btn btn-outline' id='zipNextPageBtn' disabled>Siguiente</button>
                </div>
            </div>
            <div style='display:flex; gap:12px; margin-top:24px;'>
                <button class='btn btn-danger' style='flex:1;' onclick='closeModal("modalFolderUpload")'>Cancelar</button>
                <button class='btn btn-success' style='flex:1;' id='confirmFolderBtn' onclick='confirmFolderUpload()' disabled>✓ Confirmar carga</button>
            </div>
        </div>
    </div>
</div>

<div id='toast-container'></div>

<!-- CUERPO PRINCIPAL -->
<div class='container'>
    <div class='tool-panel'>
        <div class='search-wrapper'>
            <input type='text' id='searchInput' placeholder='Filtrar variables...' onkeyup='handleSearch()'>
        </div>
        <div class='button-group'>
            <!-- SECCIÓN 1: Importar y descargar -->
            <button class='btn btn-tool' onclick='openModal("modalConfirmUpload")' title='Importar CSV'>📤 Importar CSV</button>
            <button class='btn btn-tool' onclick='openModal("modalFolderUpload")' title='Cargar carpeta ZIP'>📁 Importar Carpeta</button>
            <button class='btn btn-tool' onclick='downloadCSV()' title='Exportar CSV'>📥 Descargar CSV</button>

            <span style='width:2px; height:30px; background:var(--border); display:inline-block; margin:0 8px;'></span>

            <!-- SECCIÓN 2: Edición de filas -->
            <button class='btn btn-action' onclick='confirmAddRow()' title='Agregar nueva variable'>➕ Nueva Fila</button>
            <button class='btn btn-action' onclick='duplicateSelected()' title='Duplicar filas seleccionadas'>📋 Duplicar Fila</button>
            <button class='btn btn-danger' onclick='deleteSelectedRows()' title='Eliminar filas seleccionadas'>🗑 Eliminar Fila</button>

            <span style='width:2px; height:30px; background:var(--border); display:inline-block; margin:0 8px;'></span>

            <!-- SECCIÓN 3: Modelos -->
            <button class='btn btn-tool' onclick='retrainModels()' title='Reentrenar modelos ML'>🔄 Reentrenar Modelos</button>
            <button class='btn btn-outline' onclick='rollbackModels()' title='Restaurar modelos anteriores'>↩️ Rollback Modelos</button>
        </div>
    </div>

    <div class='table-card' id='dropZone'>
        <div class='table-wrapper'>
            <table id='dataTable'>
                <thead><tr id='headerRow'></tr></thead>
                <tbody></tbody>
            </table>
        </div>
        <div class='table-pagination'>
            <button onclick='prevPage()' id='prevPageBtnTable'>Anterior</button>
            <span>
                Página <input type='number' id='pageInput' value='1' min='1' onchange='goToPage(this.value)'>
                de <span id='totalPagesSpan'>1</span>
            </span>
            <select id='itemsPerPageSelect' onchange='changeItemsPerPage(this.value)'>
                <option value='10'>10</option>
                <option value='25'>25</option>
                <option value='50'>50</option>
                <option value='100'>100</option>
            </select>
            <button onclick='nextPage()' id='nextPageBtnTable'>Siguiente</button>
        </div>
    </div>

    <div id='historyContainer'>
        <h3>Historial de cargas</h3>
        <div id='historyList'>
            <p style='color:var(--text-muted);'>Cargando historial...</p>
        </div>
    </div>
</div>
"""


## **JS**

In [11]:
JS = """
console.log('Gestor de Variables iniciado.');
let originalData = [], filteredData = [];
let currentSort = [];
let currentFileToUpload = null;
let allPreviewData = [];
let previewCurrentPage = 1;
const previewItemsPerPage = 10;
let currentPage = 1;
let itemsPerPage = 10;
let _currentOtraOpcionCancelCallback = null;

let folderPreviewData = [];
let folderGlobalMetadata = null;
let currentZipFilename = null;

// ===== ANCHOS DE COLUMNAS (centralizados) =====
const COLUMN_WIDTHS = {
    id: '80px',
    version: '100px',
    proceso: '160px',
    eje: '160px',
    tema: '180px',
    nombre: '280px',
    institucion: '180px',
    cobertura: '160px',
    periodicidad: '160px',
    liga_web: '180px',
    tipo_de_dato: '160px',
    fuente: '160px',
    año: '110px',
    valor: '130px'
};

// ===== UTILIDADES =====
function formatNumero(val, columna) {
    if (val === null || val === undefined || val === '') return '';
    const num = Number(val);
    if (!isNaN(num) && Number.isFinite(num)) {
        if (columna === 'año') {
            return String(Math.round(num));
        } else {
            return num.toFixed(1);
        }
    }
    return val;
}

function escapeHtml(unsafe) {
    if (unsafe == null) return '';
    return String(unsafe)
        .replace(/&/g, '&amp;')
        .replace(/</g, '&lt;')
        .replace(/>/g, '&gt;')
        .replace(/\\\\\"/g, '&quot;')
        .replace(/'/g, '&#039;')
        .replace(/`/g, '&#96;');
}

function openModal(id) { document.getElementById(id).style.display = 'flex'; }

function closeModal(id) {
    document.getElementById(id).style.display = 'none';
    if (id === 'modalFolderUpload') resetFolderModal();
    if (id === 'modalConfirmUpload') resetCsvModal();
    document.getElementById('confirmUploadBtn').disabled = true;
    document.getElementById('confirmFolderBtn').disabled = true;
}

function showToast(msg) {
    const c = document.getElementById('toast-container');
    const t = document.createElement('div');
    t.className = 'toast';
    t.textContent = msg;
    c.appendChild(t);
    setTimeout(() => { t.style.opacity = '0'; setTimeout(() => t.remove(), 500); }, 3000);
}

function handleCloseOtraOpcionModal(callback = null) {
    closeModal('modalOtraOpcion');
    if (typeof callback === 'function') callback();
    else if (typeof _currentOtraOpcionCancelCallback === 'function') _currentOtraOpcionCancelCallback();
    _currentOtraOpcionCancelCallback = null;
}

function resetFolderModal() {
    folderPreviewData = [];
    folderGlobalMetadata = null;
    currentZipFilename = null;
    const previewContainer = document.getElementById('zipPreviewContainer');
    if (previewContainer) previewContainer.style.display = 'none';
    const logContainer = document.getElementById('folderLogMessages');
    if (logContainer) logContainer.style.display = 'none';
    const progressDiv = document.getElementById('zipProgress');
    if (progressDiv) progressDiv.style.display = 'none';
    const fileInput = document.getElementById('zipFileInput');
    if (fileInput) fileInput.value = '';
    document.getElementById('rowCount').textContent = '0';
    document.getElementById('zipPreviewTable').innerHTML = '';
    document.getElementById('folderLogContent').textContent = '';
    document.getElementById('confirmFolderBtn').disabled = true;
    previewCurrentPage = 1;
}

function resetCsvModal() {
    document.getElementById('csvFileInputModal').value = '';
    document.getElementById('uploadDataPreview').style.display = 'none';
    document.getElementById('paginationControls').style.display = 'none';
    document.getElementById('uploadPreviewMessages').innerHTML = '';
    const progressDiv = document.getElementById('uploadProgress');
    if (progressDiv) progressDiv.style.display = 'none';
    allPreviewData = [];
    currentFileToUpload = null;
    document.getElementById('confirmUploadBtn').disabled = true;
    previewCurrentPage = 1;
}

function autoResize(el) {
    if (!el || el.tagName !== 'TEXTAREA') return;
    const minH = parseFloat(getComputedStyle(el).minHeight) || 24;
    el.style.height = 'auto';
    const h = Math.max(el.scrollHeight, minH);
    el.style.height = h + 'px';
}

function syncRowHeights(tr) {
    const cells = tr.querySelectorAll('td');
    if (cells.length === 0) return;

    // Resetear alturas para calcular el contenido natural
    cells.forEach(td => {
        td.style.height = 'auto';
        td.style.minHeight = 'auto';
    });
    tr.style.height = 'auto';

    // Obtener padding vertical del td para sumarlo al scrollHeight del control
    const tdPadding = (() => {
        const firstTd = cells[0];
        const style = getComputedStyle(firstTd);
        return parseFloat(style.paddingTop) + parseFloat(style.paddingBottom);
    })();

    let maxHeight = 0;

    cells.forEach(td => {
        // Buscar control dentro de la celda: textarea o div.edit-control (para selects)
        const control = td.querySelector('textarea.edit-control') || td.querySelector('div.edit-control');
        if (control) {
            // Poner altura auto para que el control se expanda a su contenido
            control.style.height = 'auto';
            // Obtener la altura natural del contenido (scrollHeight)
            const contentHeight = control.scrollHeight;
            // Sumar el padding del td para obtener la altura total de la celda
            const totalHeight = contentHeight + tdPadding;
            if (totalHeight > maxHeight) maxHeight = totalHeight;
        } else {
            // Si no hay control (caso ID o checkbox), usar scrollHeight del td
            const h = td.scrollHeight;
            if (h > maxHeight) maxHeight = h;
        }
    });

    // Añadir un margen inferior de 2px (opcional)
    const finalHeight = maxHeight + 2;

    // Asignar la misma altura a todas las celdas y a la fila
    cells.forEach(td => {
        td.style.height = finalHeight + 'px';
        td.style.minHeight = finalHeight + 'px';
    });
    tr.style.height = finalHeight + 'px';

    // Forzar que los controles ocupen el 100% de la celda (para que se estiren)
    tr.querySelectorAll('textarea.edit-control, div.edit-control').forEach(ctrl => {
        ctrl.style.height = '100%';
        ctrl.style.minHeight = '100%';
    });
}

function autoResizeModal(el) {
    if (!el || el.tagName !== 'TEXTAREA') return;
    const minH = parseFloat(getComputedStyle(el).minHeight) || 32;
    const curH = el.offsetHeight;
    el.style.height = 'auto';
    const newH = Math.max(el.scrollHeight, minH);
    const lineH = parseFloat(getComputedStyle(el).lineHeight) || 20;
    const linesCur = Math.round(curH / lineH);
    const linesNew = Math.round(newH / lineH);
    if (Math.abs(newH - curH) > 5 || linesNew !== linesCur) {
        el.style.height = newH + 'px';
    } else {
        el.style.height = curH + 'px';
    }
}

function updateTemaOptions(tr, ejeValue, currentTema) {
    const idx = window.COLUMN_DEFINITIONS.findIndex(c => c.keyName === 'tema');
    if (idx === -1) return;
    const td = tr.cells[idx + 1];
    if (!td) return;
    const container = td.querySelector('div');
    if (!container) return;
    const sel = container.querySelector('select');
    if (!sel) return;

    if (!ejeValue) {
        sel.innerHTML = '<option value="">-seleccionar-</option>';
        return;
    }
    const ejeNum = ejeValue.charAt(0);
    const temaDef = window.COLUMN_DEFINITIONS.find(c => c.keyName === 'tema');
    let html = '<option value="">-seleccionar-</option>';
    (temaDef ? temaDef.options : []).forEach(opt => {
        if (opt.startsWith(ejeNum)) html += `<option value="${opt}" ${opt===currentTema?'selected':''}>${opt}</option>`;
    });
    html += '<option value="_OTRA_">Otra...</option>';
    sel.innerHTML = html;
}

async function openOpcionesModal(col, rowId, onDone, onCancel) {
    _currentOtraOpcionCancelCallback = onCancel;
    const modal = document.getElementById('modalOtraOpcion');
    const input = document.getElementById('otraOpcionInput');
    const btn = document.getElementById('btnConfirmOtra');
    document.getElementById('modalOtraTitle').textContent = 'Agregar ' + col.displayName;
    input.value = '';
    modal.style.display = 'flex';
    input.focus();
    requestAnimationFrame(() => setTimeout(() => autoResizeModal(input), 50));

    btn.onclick = async () => {
        const newVal = input.value.trim();
        if (!newVal) { handleCloseOtraOpcionModal(onCancel); return; }
        const resp = await fetch(`/api/catalog/${col.keyName}`, {
            method: 'POST',
            headers: { 'Content-Type': 'application/json' },
            body: JSON.stringify({ value: newVal })
        });
        if (!resp.ok) {
            handleCloseOtraOpcionModal(onCancel);
            showToast('Error al guardar nuevo valor.');
            return;
        }
        const data = await resp.json();
        if (data.status === 'already_exists') {
            showToast(`'${data.value}' ya existe.`);
            onDone(data.value);
            handleCloseOtraOpcionModal();
        } else if (data.status === 'suggestion') {
            openSuggestionModal(data.original_input, data.suggested_value,
                async (finalVal) => {
                    await addAndSaveOption(col, rowId, finalVal, onDone);
                    closeModal('modalSuggestion');
                },
                () => handleCloseOtraOpcionModal(onCancel)
            );
            closeModal('modalOtraOpcion');
        } else {
            await addAndSaveOption(col, rowId, newVal, onDone);
            handleCloseOtraOpcionModal();
        }
    };
}

async function addAndSaveOption(col, rowId, valueToAdd, onDone) {
    if (!col.options.includes(valueToAdd)) { col.options.push(valueToAdd); col.options.sort(); }
    const updateData = {};
    updateData[col.keyName] = valueToAdd;
    await fetch(`/api/variables/${rowId}`, {
        method: 'PUT',
        headers: { 'Content-Type': 'application/json' },
        body: JSON.stringify(updateData)
    });
    showToast('✓ Guardado');
    onDone(valueToAdd);
}

function openSuggestionModal(originalInput, suggestedValue, onAccept, onDecline) {
    document.getElementById('suggestionMessage').innerHTML =
        `Tu valor '<strong>${escapeHtml(originalInput)}</strong>' es similar a '<strong>${escapeHtml(suggestedValue)}</strong>'. ¿Deseas usar la sugerencia?`;
    openModal('modalSuggestion');
    document.getElementById('btnAcceptSuggestion').onclick = () => onAccept(suggestedValue);
    document.getElementById('btnDeclineSuggestion').onclick = () => onDecline(originalInput);
}

// ===== CARGA DE DATOS =====
async function loadData() {
    const r = await fetch('/api/variables');
    originalData = await r.json();
    console.log(`✅ Datos cargados: ${originalData.length} registros.`);
    handleSearch();
}

// ===== ORDENAMIENTO =====
function updateSortIndicators() {
    document.querySelectorAll('#headerRow th').forEach(th => {
        th.classList.remove('sorted-asc', 'sorted-desc');
        const key = th.dataset.key;
        if (key) {
            const sort = currentSort.find(s => s.key === key);
            if (sort) {
                th.classList.add(sort.dir === 'asc' ? 'sorted-asc' : 'sorted-desc');
            }
        }
        const orderSpan = th.querySelector('.sort-order');
        if (orderSpan) orderSpan.textContent = '';
    });
    currentSort.forEach((s, idx) => {
        const th = document.querySelector(`#headerRow th[data-key=\\"${s.key}\\"]`);
        if (th) {
            let orderSpan = th.querySelector('.sort-order');
            if (!orderSpan) {
                orderSpan = document.createElement('span');
                orderSpan.className = 'sort-order';
                const div = th.querySelector('.th-content');
                if (div) div.appendChild(orderSpan);
            }
            orderSpan.textContent = ` ${idx + 1}`;
        }
    });
}

function handleSearch() {
    const q = (document.getElementById('searchInput')?.value || '').toLowerCase();
    let data = (originalData || []).filter(r =>
        Object.values(r).some(v => String(v).toLowerCase().includes(q))
    );
    if (currentSort.length > 0) {
        data.sort((a, b) => {
            for (const s of currentSort) {
                let va = String(a[s.key] || '');
                let vb = String(b[s.key] || '');
                const cmp = va.localeCompare(vb, undefined, { numeric: true });
                if (cmp !== 0) {
                    return s.dir === 'asc' ? cmp : -cmp;
                }
            }
            return 0;
        });
    }
    filteredData = data;
    currentPage = 1;
    document.querySelectorAll('.row-selector').forEach(cb => cb.checked = false);
    renderHeaders();
    renderTable();
    updateSortIndicators();
    setTimeout(() => {
        initColumnResize();
        initRowResize();
    }, 100);
}

function setSort(key, dir, event) {
    const shift = event && event.shiftKey;
    if (!shift) {
        if (currentSort.length === 1 && currentSort[0].key === key && currentSort[0].dir === dir) {
            currentSort = [];
        } else {
            currentSort = [{ key, dir }];
        }
    } else {
        const existingIdx = currentSort.findIndex(s => s.key === key);
        if (existingIdx !== -1) {
            if (currentSort[existingIdx].dir === dir) {
                currentSort.splice(existingIdx, 1);
            } else {
                currentSort[existingIdx].dir = dir;
            }
        } else {
            currentSort.push({ key, dir });
        }
    }
    updateSortIndicators();
    handleSearch();
}

// ===== RENDERIZADO DE TABLA =====
function renderHeaders() {
    const hr = document.getElementById('headerRow');
    hr.innerHTML = '';
    const thCheck = document.createElement('th');
    thCheck.style.width = '40px';
    thCheck.style.minWidth = '40px';
    thCheck.style.maxWidth = '40px';
    thCheck.style.textAlign = 'center';
    thCheck.innerHTML = `<input type="checkbox" id="selectAllRows" onchange="toggleAllCheckboxes(this)">`;
    hr.appendChild(thCheck);

    window.COLUMN_DEFINITIONS.forEach((col, index) => {
        const th = document.createElement('th');
        th.dataset.key = col.keyName;
        const sort = currentSort.find(s => s.key === col.keyName);
        if (sort) th.classList.add(sort.dir === 'asc' ? 'sorted-asc' : 'sorted-desc');
        th.innerHTML = `<div class="th-content">
            <span class="th-label">${escapeHtml(col.displayName)}</span>
            <div class="sort-indicators">
                <span class="tri-up" onclick="setSort('${col.keyName}','asc', event)">▲</span>
                <span class="tri-down" onclick="setSort('${col.keyName}','desc', event)">▼</span>
            </div>
        </div>`;

        // 🔥 USO DE LA CONSTANTE CENTRALIZADA
        const key = col.keyName;
        const width = COLUMN_WIDTHS[key] || 'auto';
        th.style.width = width;
        th.style.minWidth = width;
        th.style.maxWidth = width;
        hr.appendChild(th);
    });
}

function toggleAllCheckboxes(master) {
    document.querySelectorAll('.row-selector').forEach(cb => cb.checked = master.checked);
}

async function deleteSelectedRows() {
    const selected = document.querySelectorAll('.row-selector:checked');
    if (selected.length === 0) {
        showToast('⚠️ Selecciona al menos una fila.');
        return;
    }
    const ids = Array.from(selected).map(cb => cb.dataset.id);
    if (!confirm(`¿Eliminar ${ids.length} fila(s)?`)) return;
    for (const id of ids) {
        await fetch(`/api/variables/${id}`, { method: 'DELETE' });
    }
    loadData();
    showToast(`✓ ${ids.length} fila(s) eliminada(s).`);
}

async function duplicateSelected() {
    const selected = document.querySelectorAll('.row-selector:checked');
    if (selected.length === 0) {
        showToast('⚠️ Selecciona al menos una fila para duplicar.');
        return;
    }
    const ids = Array.from(selected).map(cb => cb.dataset.id);
    let count = 0;
    for (const id of ids) {
        const r = await fetch(`/api/variables/${id}/duplicate`, { method: 'POST' });
        if (r.ok) count++;
    }
    loadData();
    showToast(`✓ ${count} fila(s) duplicada(s).`);
}

// ===== RENDERIZADO CON EDICIÓN =====
function renderTable() {
    const tbody = document.querySelector('#dataTable tbody');
    tbody.innerHTML = '';
    const totalPages = Math.ceil(filteredData.length / itemsPerPage) || 1;
    document.getElementById('totalPagesSpan').textContent = totalPages;
    document.getElementById('pageInput').value = currentPage;
    document.getElementById('prevPageBtnTable').disabled = currentPage <= 1;
    document.getElementById('nextPageBtnTable').disabled = currentPage >= totalPages;

    const start = (currentPage - 1) * itemsPerPage;
    const end = start + itemsPerPage;
    const pageData = filteredData.slice(start, end);

    pageData.forEach(rowData => {
        const tr = document.createElement('tr');
        tr.dataset.id = rowData.id;

        // Checkbox
        const tdCheck = document.createElement('td');
        tdCheck.className = 'row-checkbox';
        tdCheck.style.width = '40px';
        tdCheck.style.minWidth = '40px';
        tdCheck.style.maxWidth = '40px';
        tdCheck.style.textAlign = 'center';
        tdCheck.style.verticalAlign = 'middle';
        const checkbox = document.createElement('input');
        checkbox.type = 'checkbox';
        checkbox.className = 'row-selector';
        checkbox.dataset.id = rowData.id;
        tdCheck.appendChild(checkbox);
        tr.appendChild(tdCheck);

        // Columnas
        window.COLUMN_DEFINITIONS.forEach((col, colIndex) => {
            const td = document.createElement('td');
            td.style.verticalAlign = 'top';
            td.style.padding = '2px 4px';
            td.style.minHeight = '28px';
            td.style.height = '100%';

            const key = col.keyName;
            let width = 'auto';
            if (key === 'id') width = '90px';
            else if (key === 'version') width = '120px';
            else if (key === 'proceso') width = '160px';
            else if (key === 'eje') width = '160px';
            else if (key === 'tema') width = '180px';
            else if (key === 'nombre') width = '280px';
            else if (key === 'institucion') width = '180px';
            else if (key === 'cobertura') width = '120px';
            else if (key === 'periodicidad') width = '140px';
            else if (key === 'liga_web') width = '180px';
            else if (key === 'tipo_de_dato') width = '120px';
            else if (key === 'fuente') width = '160px';
            else if (key === 'año') width = '90px';
            else if (key === 'valor') width = '90px';
            td.style.width = width;
            td.style.minWidth = width;
            td.style.maxWidth = width;

            let val = '';
            if (rowData[col.keyName] !== undefined && rowData[col.keyName] !== null) {
                val = String(rowData[col.keyName]);
            }
            if (col.keyName === 'valor' && val !== '') {
                val = formatNumero(val);
            }

            // Columna ID (solo lectura)
            if (col.keyName === 'id') {
                const input = document.createElement('input');
                input.type = 'text';
                input.className = 'read-only-input';
                input.value = val;
                input.disabled = true;
                input.style.width = '100%';
                input.style.textAlign = 'center';
                input.style.height = '100%';
                input.style.minHeight = '28px';
                input.style.padding = '2px 4px';
                input.style.boxSizing = 'border-box';
                input.style.background = 'transparent';
                input.style.border = 'none';
                input.style.outline = 'none';
                input.style.fontFamily = 'inherit';
                input.style.fontSize = '0.8em';
                td.appendChild(input);
            }
            // Columnas tipo select
            else if (col.type === 'select') {
                const container = document.createElement('div');
                container.style.position = 'relative';
                container.style.width = '100%';
                container.style.height = '100%';
                container.style.minHeight = '28px';
                container.style.display = 'flex';
                container.style.alignItems = 'flex-start';
                container.style.cursor = 'pointer';

                const displayDiv = document.createElement('div');
                displayDiv.className = 'edit-control';
                displayDiv.textContent = val;
                displayDiv.style.width = '100%';
                displayDiv.style.minHeight = '28px';
                displayDiv.style.height = '100%';
                displayDiv.style.padding = '2px 6px';
                displayDiv.style.lineHeight = '1.4';
                displayDiv.style.boxSizing = 'border-box';
                displayDiv.style.border = '1px solid transparent';
                displayDiv.style.borderRadius = '0';
                displayDiv.style.background = 'transparent';
                displayDiv.style.outline = 'none';
                displayDiv.style.fontFamily = 'inherit';
                displayDiv.style.fontSize = '0.8em';
                displayDiv.style.whiteSpace = 'normal';
                displayDiv.style.wordWrap = 'break-word';
                displayDiv.style.cursor = 'pointer';
                displayDiv.style.display = 'flex';
                displayDiv.style.alignItems = 'flex-start';
                displayDiv.style.flex = '1';
                displayDiv.title = val;
                displayDiv.style.textAlign = 'left';
                displayDiv.style.zIndex = '2';
                displayDiv.style.pointerEvents = 'none';
                displayDiv.style.transition = 'border-color 0.2s, box-shadow 0.2s, background 0.2s';

                const select = document.createElement('select');
                select.className = 'edit-control';
                select.style.position = 'absolute';
                select.style.top = '0';
                select.style.left = '0';
                select.style.width = '100%';
                select.style.height = '100%';
                select.style.minHeight = '28px';
                select.style.padding = '2px 6px';
                select.style.lineHeight = '1.4';
                select.style.boxSizing = 'border-box';
                select.style.border = 'none';
                select.style.background = 'transparent';
                select.style.outline = 'none';
                select.style.fontFamily = 'inherit';
                select.style.fontSize = '0.8em';
                select.style.verticalAlign = 'top';
                select.style.overflow = 'visible';
                select.style.whiteSpace = 'nowrap';
                select.style.textOverflow = 'clip';
                select.style.opacity = '0';
                select.style.zIndex = '1';
                select.style.cursor = 'pointer';

                // Resaltado al focus/blur
                select.addEventListener('focus', function() {
                    displayDiv.style.border = '1px solid var(--primary)';
                    displayDiv.style.background = 'white';
                    displayDiv.style.boxShadow = '0 0 0 3px rgba(74,85,104,0.15)';
                });

                select.addEventListener('blur', function() {
                    setTimeout(() => {
                        displayDiv.style.border = '1px solid transparent';
                        displayDiv.style.background = 'transparent';
                        displayDiv.style.boxShadow = 'none';
                    }, 150);
                });

                // Llenar opciones
                let htmlOpts = `<option value="">-seleccionar-</option>`;
                (col.options || []).forEach(o => htmlOpts += `<option value="${o}" ${o===val?'selected':''}>${o}</option>`);
                htmlOpts += `<option value="_OTRA_">Otra...</option>`;
                select.innerHTML = htmlOpts;

                // Evento para abrir dropdown
                container.addEventListener('pointerdown', function(e) {
                    e.preventDefault();
                    e.stopPropagation();
                    select.focus();
                    if (select.showPicker) {
                        select.showPicker();
                    } else {
                        select.click();
                    }
                });

                // Evento change del select
                select.onchange = async function() {
                    if (this.value === '_OTRA_') {
                        const currentVal = val;
                        openOpcionesModal(col, rowData.id,
                            (newVal) => {
                                rowData[col.keyName] = newVal;
                                val = newVal;
                                displayDiv.textContent = newVal;
                                displayDiv.title = newVal;
                                const newOpts = col.options;
                                let newHtml = `<option value="">-seleccionar-</option>`;
                                newOpts.forEach(o => newHtml += `<option value="${o}" ${o===newVal?'selected':''}>${o}</option>`);
                                newHtml += `<option value="_OTRA_">Otra...</option>`;
                                select.innerHTML = newHtml;
                                select.value = newVal;
                                if (col.keyName === 'eje') updateTemaOptions(tr, newVal, rowData['tema']);
                                saveRow(tr);
                                displayDiv.style.background = 'transparent';
                                syncRowHeights(tr);
                            },
                            () => {
                                select.value = currentVal;
                                displayDiv.textContent = currentVal;
                                displayDiv.title = currentVal;
                                displayDiv.style.background = 'transparent';
                                val = currentVal;
                                rowData[col.keyName] = currentVal;
                                syncRowHeights(tr);
                            }
                        );
                        return;
                    }
                    const newVal = this.value;
                    val = newVal;
                    rowData[col.keyName] = newVal;
                    displayDiv.textContent = newVal;
                    displayDiv.title = newVal;
                    displayDiv.style.background = 'transparent';
                    if (col.keyName === 'eje') updateTemaOptions(tr, newVal, rowData['tema']);
                    saveRow(tr);
                    syncRowHeights(tr);
                };

                // Evento blur adicional para guardar si el valor cambió
                select.addEventListener('blur', function() {
                    const newVal = this.value;
                    if (newVal !== val && newVal !== '_OTRA_') {
                        val = newVal;
                        rowData[col.keyName] = newVal;
                        displayDiv.textContent = newVal;
                        displayDiv.title = newVal;
                        saveRow(tr);
                        syncRowHeights(tr);
                    }
                });

                container.appendChild(displayDiv);
                container.appendChild(select);
                td.appendChild(container);
            }
            // Columnas de texto (textarea) – CON EL CAMBIO PARA AUTO-AJUSTE
            else {
                const textarea = document.createElement('textarea');
                textarea.className = 'edit-control';
                textarea.value = val;
                textarea.style.width = '100%';
                textarea.style.height = '100%';
                textarea.style.minHeight = '28px';
                textarea.style.padding = '2px 6px';
                textarea.style.lineHeight = '1.4';
                textarea.style.boxSizing = 'border-box';
                textarea.style.border = '1px solid transparent';
                textarea.style.background = 'transparent';
                textarea.style.outline = 'none';
                textarea.style.fontFamily = 'inherit';
                textarea.style.fontSize = '0.8em';
                textarea.style.resize = 'none';
                // IMPORTANTE: NO se asigna overflow:auto → usa el overflow:hidden del CSS
                // textarea.style.overflow = 'auto';  // ELIMINAR ESTA LÍNEA
                textarea.style.verticalAlign = 'top';
                textarea.style.whiteSpace = 'pre-wrap';
                textarea.style.wordBreak = 'break-word';
                textarea.style.transition = 'border-color 0.2s, box-shadow 0.2s';

                const placeholderMap = {
                    nombre: 'Insertar nombre',
                    liga_web: 'Insertar liga web',
                    fuente: 'Insertar fuente',
                    año: 'Insertar año',
                    valor: 'Insertar valor'
                };
                if (placeholderMap[col.keyName] && (!val || val.trim() === '')) {
                    textarea.placeholder = placeholderMap[col.keyName];
                }

                textarea.onfocus = function() {
                    const parentTd = this.closest('td');
                    if (parentTd) parentTd.classList.add('cell-focused');
                    this.style.borderColor = 'var(--primary)';
                    this.style.background = 'white';
                    this.style.boxShadow = '0 0 0 3px rgba(74,85,104,0.15)';
                };
                textarea.onblur = function() {
                    const parentTd = this.closest('td');
                    if (parentTd) parentTd.classList.remove('cell-focused');
                    this.style.borderColor = 'transparent';
                    this.style.background = 'transparent';
                    this.style.boxShadow = 'none';
                    if (this.value !== val) {
                        val = this.value;
                        rowData[col.keyName] = this.value;
                        saveRow(tr);
                        syncRowHeights(tr);
                    }
                };
                textarea.oninput = function() {
                    // Usar setTimeout para dar tiempo al navegador a actualizar el layout
                    setTimeout(() => syncRowHeights(tr), 0);
                };
                td.appendChild(textarea);
            }

            tr.appendChild(td);
        });

        tbody.appendChild(tr);
        requestAnimationFrame(() => {
            syncRowHeights(tr);
        });
    });
}

async function saveRow(tr) {
    const id = tr.dataset.id;
    const data = {};
    const cells = tr.querySelectorAll('td');
    window.COLUMN_DEFINITIONS.forEach((col, idx) => {
        const td = cells[idx + 1];
        if (!td) return;
        let ctrl;
        if (col.type === 'select') {
            // Para columnas select, buscar el <select> dentro de la celda
            ctrl = td.querySelector('select');
        } else {
            // Para otras columnas (textarea, input), buscar .edit-control
            ctrl = td.querySelector('.edit-control');
        }
        if (ctrl) {
            let value = ctrl.value;
            data[col.keyName] = value;
            console.log(`📤 Guardando ${col.keyName}: ${value}`);
        }
    });
    console.log(`📤 Enviando PUT a /api/variables/${id} con datos:`, data);
    const resp = await fetch(`/api/variables/${id}`, {
        method: 'PUT',
        headers: { 'Content-Type': 'application/json' },
        body: JSON.stringify(data)
    });
    const result = await resp.json();
    console.log(`📥 Respuesta:`, result);
    if (resp.ok) {
        const idx = originalData.findIndex(r => r.id === id);
        if (idx !== -1) {
            originalData[idx] = { ...originalData[idx], ...data };
        }
        showToast('✓ Celda actualizada');
    } else {
        showToast('Error al actualizar celda.');
    }
}

async function confirmAddRow() {
    const r = await fetch('/api/variables', {
        method: 'POST',
        headers: { 'Content-Type': 'application/json' },
        body: JSON.stringify({})
    });
    if (r.ok) {
        const newRow = await r.json();
        originalData.unshift(newRow);
        handleSearch();
        showToast('✓ Fila creada');
    } else {
        const error = await r.json();
        showToast('Error: ' + (error.error || 'No se pudo crear la fila'));
        console.error('Error al crear fila:', error);
    }
}

function changeItemsPerPage(val) {
    itemsPerPage = parseInt(val);
    currentPage = 1;
    renderTable();
}

function goToPage(page) {
    const total = Math.ceil(filteredData.length / itemsPerPage) || 1;
    let p = parseInt(page);
    if (isNaN(p) || p < 1) p = 1;
    if (p > total) p = total;
    currentPage = p;
    renderTable();
}

function nextPage() {
    const total = Math.ceil(filteredData.length / itemsPerPage) || 1;
    if (currentPage < total) { currentPage++; renderTable(); }
}

function prevPage() {
    if (currentPage > 1) { currentPage--; renderTable(); }
}

function downloadCSV() {
    if (filteredData.length === 0) {
        showToast('No hay datos filtrados para descargar.');
        return;
    }
    const headers = window.COLUMN_DEFINITIONS.map(c => c.displayName);
    const rows = [];
    rows.push(headers.map(h => `"${h.replace(/"/g, '""')}"`).join(','));
    filteredData.forEach(row => {
        const vals = window.COLUMN_DEFINITIONS.map(c => {
            let v = row[c.keyName] != null ? String(row[c.keyName]) : '';
            if (c.keyName === 'valor' && v !== '') v = formatNumero(v);
            return `"${v.replace(/"/g, '""')}"`;
        });
        rows.push(vals.join(','));
    });
    const csvContent = rows.join('\\n');
    const blob = new Blob(['\uFEFF' + csvContent], { type: 'text/csv;charset=utf-8;' });
    const url = URL.createObjectURL(blob);
    const a = document.createElement('a');
    a.href = url;
    const now = new Date();
    const fechaHora = now.getFullYear() +
                      String(now.getMonth() + 1).padStart(2, '0') +
                      String(now.getDate()).padStart(2, '0') +
                      String(now.getHours()).padStart(2, '0') +
                      String(now.getMinutes()).padStart(2, '0') +
                      String(now.getSeconds()).padStart(2, '0');
    a.download = `variables_${fechaHora}.csv`;
    document.body.appendChild(a);
    a.click();
    document.body.removeChild(a);
    URL.revokeObjectURL(url);
    showToast('✓ CSV descargado');
}

// ============================================================
// REDIMENSIONAMIENTO DE COLUMNAS Y FILAS (por arrastre de bordes)
// ============================================================

let colResizeData = null;
let rowResizeData = null;

// ---- REDIMENSIONAMIENTO DE COLUMNAS (desde cualquier celda) ----
function initColumnResize() {
    const table = document.getElementById('dataTable');
    if (!table) return;
    const cells = table.querySelectorAll('th, td:not(.row-checkbox)');
    cells.forEach(cell => {
        cell.removeEventListener('mousemove', onColumnMouseMove, true);
        cell.removeEventListener('mouseleave', onColumnMouseLeave, true);
        cell.removeEventListener('mousedown', onColumnMouseDown, true);
        cell.addEventListener('mousemove', onColumnMouseMove, true);
        cell.addEventListener('mouseleave', onColumnMouseLeave, true);
        cell.addEventListener('mousedown', onColumnMouseDown, true);
    });
}

function onColumnMouseMove(e) {
    const cell = e.currentTarget;
    const rect = cell.getBoundingClientRect();
    const isRightEdge = (e.clientX - rect.left) > (rect.width - 20);
    if (isRightEdge) {
        cell.style.cursor = 'col-resize';
        cell.dataset.resizeableCol = 'true';
    } else {
        cell.style.cursor = '';
        cell.dataset.resizeableCol = 'false';
    }
}

function onColumnMouseLeave(e) {
    const cell = e.currentTarget;
    cell.style.cursor = '';
    cell.dataset.resizeableCol = 'false';
}

function onColumnMouseDown(e) {
    const cell = e.currentTarget;
    if (cell.dataset.resizeableCol !== 'true') return;
    const colIndex = cell.cellIndex;
    const startX = e.clientX;
    const startWidth = cell.offsetWidth;
    colResizeData = {
        cell: cell,
        colIndex: colIndex,
        startX: startX,
        startWidth: startWidth,
        table: document.getElementById('dataTable')
    };
    document.addEventListener('mousemove', onColumnResizeMove);
    document.addEventListener('mouseup', onColumnResizeUp);
    e.preventDefault();
}

function onColumnResizeMove(e) {
    if (!colResizeData) return;
    const dx = e.clientX - colResizeData.startX;
    const newWidth = Math.max(50, colResizeData.startWidth + dx);
    const colIndex = colResizeData.colIndex;
    const rows = colResizeData.table.querySelectorAll('tr');
    rows.forEach(row => {
        const cell = row.cells[colIndex];
        if (cell) {
            cell.style.width = newWidth + 'px';
            cell.style.minWidth = newWidth + 'px';
            cell.style.maxWidth = newWidth + 'px';
        }
    });
    const th = colResizeData.table.querySelector(`thead th:nth-child(${colIndex + 1})`);
    if (th) {
        th.style.width = newWidth + 'px';
        th.style.minWidth = newWidth + 'px';
        th.style.maxWidth = newWidth + 'px';
    }
}

function onColumnResizeUp() {
    document.removeEventListener('mousemove', onColumnResizeMove);
    document.removeEventListener('mouseup', onColumnResizeUp);
    colResizeData = null;
}

// ---- REDIMENSIONAMIENTO DE FILAS (desde cualquier celda) ----
function initRowResize() {
    const table = document.getElementById('dataTable');
    if (!table) return;
    const rows = table.querySelectorAll('tr');
    rows.forEach(tr => {
        const cells = tr.querySelectorAll('td');
        cells.forEach(td => {
            if (td.classList.contains('row-checkbox')) return;
            td.removeEventListener('mousemove', onRowMouseMove, true);
            td.removeEventListener('mouseleave', onRowMouseLeave, true);
            td.removeEventListener('mousedown', onRowMouseDown, true);
            td.addEventListener('mousemove', onRowMouseMove, true);
            td.addEventListener('mouseleave', onRowMouseLeave, true);
            td.addEventListener('mousedown', onRowMouseDown, true);
        });
    });
}

function onRowMouseMove(e) {
    const td = e.currentTarget;
    if (td.dataset.resizeableCol === 'true') return;
    const rect = td.getBoundingClientRect();
    const isBottomEdge = (e.clientY - rect.top) > (rect.height - 10);
    td.style.cursor = isBottomEdge ? 'row-resize' : '';
    td.dataset.resizeable = isBottomEdge ? 'true' : 'false';
}

function onRowMouseLeave(e) {
    const td = e.currentTarget;
    td.style.cursor = '';
    td.dataset.resizeable = 'false';
}

function onRowMouseDown(e) {
    const td = e.currentTarget;
    if (td.dataset.resizeable !== 'true') return;
    const tr = td.parentElement;
    const startY = e.clientY;
    const startHeight = tr.offsetHeight;
    rowResizeData = { tr, startY, startHeight };
    document.addEventListener('mousemove', onRowResizeMove);
    document.addEventListener('mouseup', onRowResizeUp);
    e.preventDefault();
}

function onRowResizeMove(e) {
    if (!rowResizeData) return;
    const dy = e.clientY - rowResizeData.startY;
    const newHeight = Math.max(20, rowResizeData.startHeight + dy);
    const cells = rowResizeData.tr.querySelectorAll('td');
    cells.forEach(td => {
        td.style.height = newHeight + 'px';
        td.style.minHeight = newHeight + 'px';
    });
    rowResizeData.tr.style.height = newHeight + 'px';
}

function onRowResizeUp() {
    document.removeEventListener('mousemove', onRowResizeMove);
    document.removeEventListener('mouseup', onRowResizeUp);
    rowResizeData = null;
}

// ============================================================
// PREVISUALIZACIÓN UNIFICADA
// ============================================================
function renderPreviewTable(containerId, data, isZip = false) {
    const container = document.getElementById(containerId);
    if (!container) return;
    let html = '<table>';

    console.log(`🔍 Renderizando previsualización en ${containerId}: ${data.length} filas`);

    if (data && data.length > 0) {
        // Calcular páginas
        const totalPages = Math.ceil(data.length / previewItemsPerPage) || 1;
        if (previewCurrentPage > totalPages) previewCurrentPage = totalPages;
        const start = (previewCurrentPage - 1) * previewItemsPerPage;
        const end = Math.min(start + previewItemsPerPage, data.length);
        const pageData = data.slice(start, end);

        const headerCols = window.COLUMN_DEFINITIONS.map(c => c.displayName);
        html += '<thead><tr>';
        const actionWidth = '90px';
        html += `<th style="width:${actionWidth}; min-width:${actionWidth}; max-width:${actionWidth};">Acción</th>`;
        headerCols.forEach(k => {
            const key = window.COLUMN_DEFINITIONS.find(c => c.displayName === k)?.keyName;
            const width = COLUMN_WIDTHS[key] || 'auto';
            html += `<th style="width:${width}; min-width:${width}; max-width:${width};">${escapeHtml(k)}</th>`;
        });
        html += '</tr></thead>';

        html += '<tbody>';
        for (let i = 0; i < pageData.length; i++) {
            const rowInfo = pageData[i];
            const rowData = rowInfo.data || rowInfo;
            const isDup = rowInfo._is_duplicate || false;
            const cls = isDup ? 'duplicate-row-preview' : '';
            const checked = isDup && rowInfo._action === 'overwrite' ? 'checked' : '';
            const realIndex = start + i; // índice real en el array original

            let actionHtml = `<td style="padding:2px 4px; vertical-align:top; text-align:left; width:${actionWidth}; min-width:${actionWidth}; max-width:${actionWidth};">
                ${isDup ? `<input type="checkbox" class="action-checkbox" data-index="${realIndex}" data-iszip="${isZip}" ${checked}> Sobrescribir` : '✓ Insertar'}
            </td>`;

            const rowSeg = window.COLUMN_DEFINITIONS.map(col => {
                let val = rowData[col.keyName] !== undefined && rowData[col.keyName] !== null ? String(rowData[col.keyName]) : '';
                if (col.keyName === 'valor' && val !== '' && !isNaN(val)) {
                    val = formatNumero(val);
                }
                let inputHtml = '';
                if (col.type === 'select' && col.keyName !== 'id') {
                    let opts = `<option value="">-seleccionar-</option>`;
                    (col.options || []).forEach(o => opts += `<option value="${escapeHtml(o)}" ${o===val?'selected':''}>${escapeHtml(o)}</option>`);
                    opts += `<option value="_OTRA_">Otra...</option>`;
                    inputHtml = `<select class="preview-edit" data-col="${col.keyName}" data-index="${realIndex}" data-iszip="${isZip}">${opts}</select>`;
                } else if (col.keyName === 'id') {
                    inputHtml = `<span class="preview-edit" style="display:block; width:100%;">${escapeHtml(val)}</span>`;
                } else {
                    inputHtml = `<textarea class="preview-edit" data-col="${col.keyName}" data-index="${realIndex}" data-iszip="${isZip}">${escapeHtml(val)}</textarea>`;
                }
                const width = COLUMN_WIDTHS[col.keyName] || 'auto';
                return `<td style="width:${width}; min-width:${width}; max-width:${width}; padding:2px 4px; vertical-align:top;">${inputHtml}</td>`;
            }).join('');

            html += `<tr class="${cls}">${actionHtml}${rowSeg}</tr>`;
        }
        html += '</tbody>';
    } else {
        html += '<tbody><tr><td colspan="14" style="padding:8px;text-align:center;color:#6c757d;">No hay datos para previsualizar.</td></tr></tbody>';
    }

    html += '</table>';
    container.innerHTML = html;

    // ---- Ajuste automático de altura (sin resize) ----
    const textareas = container.querySelectorAll('textarea.preview-edit');
    textareas.forEach(ta => {
        const adjustHeight = function() {
            this.style.height = 'auto';
            this.style.height = this.scrollHeight + 'px';
        };
        adjustHeight.call(ta);
        ta.addEventListener('input', adjustHeight);
        ta.addEventListener('change', adjustHeight);
    });

    // ---- Eventos para guardar cambios (textareas) ----
    document.querySelectorAll(`#${containerId} textarea.preview-edit`).forEach(el => {
        el.addEventListener('change', function() {
            const idx = parseInt(this.dataset.index);
            const col = this.dataset.col;
            const isZip = this.dataset.iszip === 'true';
            let targetData = isZip ? folderPreviewData : allPreviewData;
            if (targetData[idx]) {
                targetData[idx].data[col] = this.value;
            }
        });
    });

    // ---- Eventos para selects ----
    document.querySelectorAll(`#${containerId} select.preview-edit`).forEach(el => {
        const selectedOption = el.options[el.selectedIndex];
        if (selectedOption) el.title = selectedOption.text;

        el.addEventListener('change', function() {
            const idx = parseInt(this.dataset.index);
            const col = this.dataset.col;
            const isZip = this.dataset.iszip === 'true';
            let targetData = isZip ? folderPreviewData : allPreviewData;

            const selectedOpt = this.options[this.selectedIndex];
            if (selectedOpt) this.title = selectedOpt.text;

            if (targetData[idx]) {
                if (this.value === '_OTRA_') {
                    const colDef = window.COLUMN_DEFINITIONS.find(c => c.keyName === col);
                    if (colDef) {
                        const rowId = targetData[idx].data?.id || 'new';
                        openOpcionesModal(colDef, rowId,
                            (newVal) => {
                                targetData[idx].data[col] = newVal;
                                renderPreviewTable(containerId, targetData, isZip);
                            },
                            () => { /* cancelar */ }
                        );
                    }
                } else {
                    targetData[idx].data[col] = this.value;
                }
            }
        });
    });

    // ---- Checkboxes de acción ----
    document.querySelectorAll(`#${containerId} .action-checkbox`).forEach(cb => {
        cb.addEventListener('change', function() {
            const idx = parseInt(this.dataset.index);
            const isZip = this.dataset.iszip === 'true';
            let targetData = isZip ? folderPreviewData : allPreviewData;
            if (targetData[idx]) {
                targetData[idx]._action = this.checked ? 'overwrite' : 'ignore';
                updateSelectAllCheckbox(containerId, isZip);
            }
        });
    });

    updateSelectAllCheckbox(containerId, isZip);
    previewUpdateControls(containerId, isZip);
}

// ===== PREVISUALIZACIÓN: PAGINACIÓN =====
function previewPrevPage(containerId, isZip) {
    const targetData = isZip ? folderPreviewData : allPreviewData;
    const totalPages = Math.ceil(targetData.length / previewItemsPerPage) || 1;
    if (previewCurrentPage > 1) {
        previewCurrentPage--;
        renderPreviewTable(containerId, targetData, isZip);
        previewUpdateControls(containerId, isZip);
    }
}

function previewNextPage(containerId, isZip) {
    const targetData = isZip ? folderPreviewData : allPreviewData;
    const totalPages = Math.ceil(targetData.length / previewItemsPerPage) || 1;
    if (previewCurrentPage < totalPages) {
        previewCurrentPage++;
        renderPreviewTable(containerId, targetData, isZip);
        previewUpdateControls(containerId, isZip);
    }
}

function previewGoToPage(page, containerId, isZip) {
    const targetData = isZip ? folderPreviewData : allPreviewData;
    const totalPages = Math.ceil(targetData.length / previewItemsPerPage) || 1;
    let p = parseInt(page);
    if (isNaN(p) || p < 1) p = 1;
    if (p > totalPages) p = totalPages;
    previewCurrentPage = p;
    renderPreviewTable(containerId, targetData, isZip);
    previewUpdateControls(containerId, isZip);
}

function previewUpdateControls(containerId, isZip) {
    const targetData = isZip ? folderPreviewData : allPreviewData;
    const totalPages = Math.ceil(targetData.length / previewItemsPerPage) || 1;
    const controlsId = isZip ? 'zipPaginationControls' : 'paginationControls';
    const prevBtnId = isZip ? 'zipPrevPageBtn' : 'prevPageBtn';
    const nextBtnId = isZip ? 'zipNextPageBtn' : 'nextPageBtn';
    const infoId = isZip ? 'zipPageInfo' : 'pageInfo';

    const controls = document.getElementById(controlsId);
    const prevBtn = document.getElementById(prevBtnId);
    const nextBtn = document.getElementById(nextBtnId);
    const info = document.getElementById(infoId);

    if (!controls) return;
    controls.style.display = 'flex';
    prevBtn.disabled = previewCurrentPage <= 1;
    nextBtn.disabled = previewCurrentPage >= totalPages;
    info.textContent = `Página ${previewCurrentPage} de ${totalPages}`;
}

function updateSelectAllCheckbox(containerId, isZip) {
    const container = document.getElementById(containerId);
    if (!container) return;
    let targetData = isZip ? folderPreviewData : allPreviewData;
    const dups = targetData.filter(r => r._is_duplicate);
    const selAllId = isZip ? 'selectAllDuplicatesForOverwriteZip' : 'selectAllDuplicatesForOverwrite';
    const selAll = document.getElementById(selAllId);
    if (selAll) {
        selAll.checked = dups.every(r => r._action === 'overwrite');
        selAll.onchange = () => {
            targetData.forEach(r => { if (r._is_duplicate) r._action = selAll.checked ? 'overwrite' : 'ignore'; });
            renderPreviewTable(containerId, targetData, isZip);
        };
    }
}

// ============================================================
// MANEJO DE CARGA DE CSV DESDE EL MODAL
// ============================================================
async function handleFileUploadModal(input) {
    const file = input.files[0];
    if (!file) return;
    if (!file.name.endsWith('.csv')) {
        showToast('⚠️ Por favor, sube un archivo CSV.');
        input.value = '';
        return;
    }
    currentFileToUpload = file;
    document.getElementById('uploadPreviewMessages').innerHTML = '';
    document.getElementById('uploadDataPreview').innerHTML = '';
    document.getElementById('uploadDataPreview').style.display = 'none';
    document.getElementById('paginationControls').style.display = 'none';
    const progressDiv = document.getElementById('uploadProgress');
    const progressFill = document.getElementById('uploadProgressFill');
    const progressText = document.getElementById('uploadProgressText');
    progressDiv.style.display = 'block';
    progressFill.style.width = '0%';
    progressText.textContent = 'Subiendo archivo...';

    const fd = new FormData();
    fd.append('file', file);
    try {
        let progress = 0;
        const interval = setInterval(() => {
            progress += Math.random() * 15;
            if (progress > 90) progress = 90;
            progressFill.style.width = progress + '%';
            progressText.textContent = `Procesando... ${Math.round(progress)}%`;
        }, 300);

        const r = await fetch('/api/upload?preview=true', { method: 'POST', body: fd });
        clearInterval(interval);
        progressFill.style.width = '100%';
        progressText.textContent = '¡Completado!';
        setTimeout(() => { progressDiv.style.display = 'none'; }, 1000);

        if (r.ok) {
            const data = await r.json();
            document.getElementById('uploadPreviewMessages').innerHTML = data.messages.map(m => `<div>${escapeHtml(m)}</div>`).join('');
            allPreviewData = data.preview_data || [];
            console.log(`📥 Datos recibidos del backend (${allPreviewData.length} filas):`, allPreviewData);
            allPreviewData.forEach(row => { row._action = row._is_duplicate ? 'ignore' : 'insert'; });

            // INICIAR DESDE PÁGINA 1
            previewCurrentPage = 1;
            document.getElementById('uploadDataPreview').style.display = 'block';
            document.getElementById('paginationControls').style.display = 'flex';
            renderPreviewTable('uploadDataPreview', allPreviewData, false);
            document.getElementById('confirmUploadBtn').disabled = false;
        } else {
            const err = await r.json();
            document.getElementById('uploadPreviewMessages').innerHTML = `<div style="color:var(--danger);">Error: ${err.error || 'Algo salió mal'}</div>`;
            allPreviewData = [];
            document.getElementById('confirmUploadBtn').disabled = true;
        }
    } catch (e) {
        document.getElementById('uploadPreviewMessages').innerHTML = `<div style="color:var(--danger);">Error de conexión: ${e.message}</div>`;
        allPreviewData = [];
        document.getElementById('confirmUploadBtn').disabled = true;
    }
}

async function confirmUpload() {
    if (!currentFileToUpload) return showToast('Error: No hay archivo para cargar.');
    if (allPreviewData.length === 0) return showToast('No hay datos para confirmar.');

    const actions = allPreviewData.map(row => ({
        original_csv_index: row.original_csv_index,
        action: row._action,
        data: row.data,
        _is_duplicate: row._is_duplicate,
        _supabase_matching_id: row._supabase_matching_id
    }));

    console.log("%c 🚀 Enviando al backend (CSV): ", "background: #222; color: #ffaa00; font-size: 14px;", actions);

    const fd = new FormData();
    fd.append('file', currentFileToUpload);
    fd.append('actions_for_rows', JSON.stringify(actions));

    const r = await fetch('/api/upload', { method: 'POST', body: fd });
    const data = await r.json();

    if (data.debug) {
        console.log("%c 📡 Logs del backend (CSV): ", "background: #222; color: #00ffaa; font-size: 14px;", data.debug);
    }

    if (r.ok) {
        showToast(data.message);
        await loadData();
        loadHistory();
        closeModal('modalConfirmUpload');
        resetCsvModal();
    } else {
        showToast(`Error: ${data.error || 'Algo salió mal'}`);
    }
}

// ============================================================
// CARGA DE CARPETA ZIP
// ============================================================
async function handleZipUpload(input) {
    const file = input.files[0];
    if (!file) return;
    if (!file.name.endsWith('.zip')) {
        showToast('⚠️ Por favor, selecciona un archivo ZIP.');
        input.value = '';
        return;
    }

    const previewContainer = document.getElementById('zipPreviewContainer');
    const logContainer = document.getElementById('folderLogMessages');
    const logContent = document.getElementById('folderLogContent');
    const progressDiv = document.getElementById('zipProgress');
    const progressFill = document.getElementById('zipProgressFill');
    const progressText = document.getElementById('zipProgressText');

    if (previewContainer) previewContainer.style.display = 'none';
    if (logContainer) logContainer.style.display = 'none';

    if (progressDiv) {
        progressDiv.style.display = 'block';
        progressFill.style.width = '0%';
        progressText.textContent = 'Subiendo archivo...';
    }

    const formData = new FormData();
    formData.append('file', file);

    try {
        let progress = 0;
        const interval = setInterval(() => {
            progress += Math.random() * 15;
            if (progress > 90) progress = 90;
            progressFill.style.width = progress + '%';
            progressText.textContent = `Procesando... ${Math.round(progress)}%`;
        }, 300);

        const response = await fetch('/api/process-folder', {
            method: 'POST',
            body: formData
        });

        clearInterval(interval);
        progressFill.style.width = '100%';
        progressText.textContent = '¡Completado!';

        if (response.ok) {
            const data = await response.json();
            folderPreviewData = data.preview_data || [];
            console.log(`📥 Datos ZIP recibidos (${folderPreviewData.length} filas):`, folderPreviewData);
            folderPreviewData.forEach(row => {
                if (row._is_duplicate === undefined) row._is_duplicate = false;
                if (row._action === undefined) row._action = row._is_duplicate ? 'ignore' : 'insert';
            });
            folderGlobalMetadata = data.global_metadata || {};
            currentZipFilename = data.zip_filename || file.name;

            if (logContainer && logContent) {
                if (data.messages && data.messages.length > 0) {
                    logContainer.style.display = 'block';
                    logContent.innerHTML = data.messages.map(m => escapeHtml(m)).join('<br>');
                } else {
                    logContainer.style.display = 'none';
                }
            }

            if (previewContainer) {
                previewContainer.style.display = 'block';
                previewCurrentPage = 1;
                renderPreviewTable('zipPreviewTable', folderPreviewData, true);
                document.getElementById('rowCount').textContent = folderPreviewData.length;
            }
            document.getElementById('confirmFolderBtn').disabled = false;
            showToast(`✓ Procesamiento completado: ${folderPreviewData.length} filas extraídas.`);
        } else {
            const error = await response.json();
            showToast('Error: ' + (error.error || 'Algo salió mal'));
            if (logContainer && logContent) {
                logContainer.style.display = 'block';
                let html = '';
                if (error.messages && error.messages.length) {
                    html = error.messages.map(m => escapeHtml(m)).join('<br>');
                } else {
                    html = escapeHtml(error.error || 'Error desconocido');
                }
                if (error.traceback) {
                    html += '<br><br><strong>Traceback:</strong><br>' + escapeHtml(error.traceback);
                }
                logContent.innerHTML = html;
            }
            document.getElementById('confirmFolderBtn').disabled = true;
        }
    } catch (e) {
        showToast('Error de conexión: ' + e.message);
        document.getElementById('confirmFolderBtn').disabled = true;
        if (logContainer && logContent) {
            logContainer.style.display = 'block';
            logContent.innerHTML = 'Error de conexión: ' + escapeHtml(e.message);
        }
    } finally {
        setTimeout(() => {
            if (progressDiv) progressDiv.style.display = 'none';
        }, 1000);
    }
}

async function confirmFolderUpload() {
    if (!folderPreviewData || folderPreviewData.length === 0) {
        showToast('No hay datos para cargar.');
        return;
    }

    const actions = folderPreviewData.map((row, index) => ({
        original_csv_index: index,
        action: row._action || (row._is_duplicate ? 'ignore' : 'insert'),
        data: row.data || row,
        _is_duplicate: row._is_duplicate || false,
        _supabase_matching_id: row._supabase_matching_id || null
    }));

    console.log("%c 🚀 Enviando al backend (ZIP): ", "background: #222; color: #ffaa00; font-size: 14px;", actions);

    const fd = new FormData();
    fd.append('actions_for_rows', JSON.stringify(actions));
    fd.append('zip_filename', currentZipFilename || 'carga_carpeta');

    try {
        const response = await fetch('/api/confirm-folder', {
            method: 'POST',
            body: fd
        });
        const data = await response.json();

        if (data.debug) {
            console.log("%c 📡 Logs del backend (ZIP): ", "background: #222; color: #00ffaa; font-size: 14px;", data.debug);
        }

        if (response.ok) {
            showToast(data.message || '✓ Carga completada');
            await loadData();
            loadHistory();
            closeModal('modalFolderUpload');
            resetFolderModal();
        } else {
            showToast('Error: ' + (data.error || 'Algo salió mal'));
        }
    } catch (e) {
        showToast('Error de conexión: ' + e.message);
    }
}

// ============================================================
// HISTORIAL
// ============================================================
async function loadHistory() {
    try {
        const r = await fetch('/api/upload-history');
        if (r.ok) {
            const history = await r.json();
            const container = document.getElementById('historyList');
            if (history.length === 0) {
                container.innerHTML = '<p style=\\"color:var(--text-muted);\\">No hay cargas registradas.</p>';
                return;
            }
            let html = `<table>
                <thead><tr><th>Fecha</th><th>Archivo</th><th>Filas</th><th>Estado</th></tr></thead><tbody>`;
            history.forEach(item => {
                let fechaFormateada = '';
                if (item.fecha) {
                    try {
                        const date = new Date(item.fecha);
                        if (!isNaN(date.getTime())) {
                            const dia = String(date.getDate()).padStart(2, '0');
                            const mes = String(date.getMonth() + 1).padStart(2, '0');
                            const año = date.getFullYear();
                            const horas = String(date.getHours()).padStart(2, '0');
                            const minutos = String(date.getMinutes()).padStart(2, '0');
                            const segundos = String(date.getSeconds()).padStart(2, '0');
                            fechaFormateada = `${dia}/${mes}/${año} ${horas}:${minutos}:${segundos}`;
                        } else {
                            fechaFormateada = item.fecha;
                        }
                    } catch {
                        fechaFormateada = item.fecha;
                    }
                }
                const statusClass = item.estado === 'completado' ? 'status-success' :
                                  item.estado === 'error' ? 'status-error' : 'status-pending';
                html += `<tr>
                    <td>${escapeHtml(fechaFormateada)}</td>
                    <td>${escapeHtml(item.archivo || '')}</td>
                    <td>${escapeHtml(item.filas_agregadas || '')}</td>
                    <td class=\\"${statusClass}\\">${escapeHtml(item.estado || '')}</td>
                </tr>`;
            });
            html += '</tbody></table>';
            container.innerHTML = html;
        }
    } catch (e) {
        console.warn('Error al cargar historial:', e);
    }
}

// ============================================================
// MODELOS ML
// ============================================================
async function retrainModels() {
    const btn = document.querySelector('button[onclick=\\"retrainModels()\\"]');
    if (!btn) return;
    btn.disabled = true;
    btn.textContent = '⏳ Entrenando...';
    showToast('⏳ Entrenando modelos, esto puede tomar unos segundos...');

    try {
        const r = await fetch('/api/retrain-models', { method: 'POST' });
        const data = await r.json();
        if (r.ok) {
            let msg = '✅ ' + data.message;
            if (data.report) {
                const p = data.report.proceso || {};
                const e = data.report.eje || {};
                const t = data.report.tema || {};
                const precision = `P:${(p.accuracy*100||0).toFixed(1)}% | E:${(e.accuracy*100||0).toFixed(1)}% | T:${(t.accuracy*100||0).toFixed(1)}%`;
                msg += ` (${precision})`;
                if (p.accuracy < 0.7 || e.accuracy < 0.7 || t.accuracy < 0.7) {
                    msg += ' ⚠️ Precisión baja. Considera revisar datos.';
                }
            }
            showToast(msg);
        } else {
            showToast('❌ Error: ' + (data.error || 'Algo salió mal'));
        }
    } catch (e) {
        showToast('❌ Error de conexión: ' + e.message);
    } finally {
        btn.disabled = false;
        btn.textContent = '🔄 Reentrenar Modelos';
    }
}

async function rollbackModels() {
    if (!confirm('⚠️ ¿Restaurar los modelos a la versión anterior? Se perderán los cambios del último reentrenamiento.')) return;
    const r = await fetch('/api/rollback-models', { method: 'POST' });
    const data = await r.json();
    if (r.ok) {
        showToast('✅ ' + data.message);
    } else {
        showToast('❌ ' + (data.error || 'No hay backup disponible'));
    }
}

// ============================================================
// DRAG & DROP
// ============================================================
const dropZone = document.getElementById('dropZone');
if (dropZone) {
    ['dragenter', 'dragover', 'dragleave', 'drop'].forEach(evt => {
        dropZone.addEventListener(evt, e => { e.preventDefault(); e.stopPropagation(); }, false);
    });
    dropZone.addEventListener('dragenter', () => dropZone.classList.add('drag-over'), false);
    dropZone.addEventListener('dragover', () => dropZone.classList.add('drag-over'), false);
    dropZone.addEventListener('dragleave', () => dropZone.classList.remove('drag-over'), false);
    dropZone.addEventListener('drop', e => {
        dropZone.classList.remove('drag-over');
        const file = e.dataTransfer.files[0];
        if (file) {
            if (file.name.endsWith('.csv')) {
                openModal('modalConfirmUpload');
                const input = document.getElementById('csvFileInputModal');
                const dataTransfer = new DataTransfer();
                dataTransfer.items.add(file);
                input.files = dataTransfer.files;
                const event = new Event('change', { bubbles: true });
                input.dispatchEvent(event);
            } else if (file.name.endsWith('.zip')) {
                openModal('modalFolderUpload');
                const input = document.getElementById('zipFileInput');
                const dataTransfer = new DataTransfer();
                dataTransfer.items.add(file);
                input.files = dataTransfer.files;
                const event = new Event('change', { bubbles: true });
                input.dispatchEvent(event);
            } else {
                showToast('⚠️ Solo se permiten archivos CSV o ZIP.');
            }
        }
    }, false);
} else {
    console.warn('Elemento #dropZone no encontrado.');
}

// ============================================================
// INICIALIZACIÓN
// ============================================================
window.onload = async () => {
    // Carga inicial de datos
    await loadData();
    renderHeaders();
    document.getElementById('itemsPerPageSelect').value = itemsPerPage;
    await loadHistory();

    // ---- PAGINACIÓN DE PREVISUALIZACIÓN (CSV) ----
    const prevPageBtn = document.getElementById('prevPageBtn');
    const nextPageBtn = document.getElementById('nextPageBtn');
    if (prevPageBtn) {
        prevPageBtn.addEventListener('click', function() {
            previewPrevPage('uploadDataPreview', false);
        });
    }
    if (nextPageBtn) {
        nextPageBtn.addEventListener('click', function() {
            previewNextPage('uploadDataPreview', false);
        });
    }

    // ---- PAGINACIÓN DE PREVISUALIZACIÓN (ZIP) ----
    const zipPrevPageBtn = document.getElementById('zipPrevPageBtn');
    const zipNextPageBtn = document.getElementById('zipNextPageBtn');
    if (zipPrevPageBtn) {
        zipPrevPageBtn.addEventListener('click', function() {
            previewPrevPage('zipPreviewTable', true);
        });
    }
    if (zipNextPageBtn) {
        zipNextPageBtn.addEventListener('click', function() {
            previewNextPage('zipPreviewTable', true);
        });
    }

    // ---- TECLA ESC para cerrar modales ----
    document.addEventListener('keydown', e => {
        if (e.key === 'Escape') {
            if (document.getElementById('modalOtraOpcion').style.display === 'flex') {
                handleCloseOtraOpcionModal();
            } else if (document.getElementById('modalSuggestion').style.display === 'flex') {
                const btn = document.getElementById('btnDeclineSuggestion');
                if (btn) btn.click();
                closeModal('modalSuggestion');
            } else if (document.getElementById('modalConfirmUpload').style.display === 'flex') {
                closeModal('modalConfirmUpload');
            } else if (document.getElementById('modalFolderUpload').style.display === 'flex') {
                closeModal('modalFolderUpload');
            }
        }
    });
};
"""

## **COMPLETO**

In [12]:
# ============================================================
# ARMAR LA PLANTILLA FINAL
# ============================================================

HTML_TEMPLATE = (
    '<!DOCTYPE html>\n'
    '<html lang="es">\n'
    '<head>\n'
    '    <meta charset="UTF-8">\n'
    '    <meta name="viewport" content="width=device-width, initial-scale=1.0">\n'
    '    <title>Gestor de Variables</title>\n'
    '    <link href="https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&display=swap" rel="stylesheet">\n'
    '    <style>\n'
    + CSS +
    '\n    </style>\n'
    '</head>\n'
    '<body>\n'
    + HTML_BODY +
    '\n    <script>\n'
    + JS +
    '\n    </script>\n'
    '</body>\n'
    '</html>'
)

# **Bloque 9**

In [13]:
# ============================================================
# BLOQUE 9: APLICACIÓN FLASK – CONFIGURACIÓN Y FUNCIONES DE ML
# ============================================================
import pickle
import os
import datetime
import shutil
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

app = Flask(__name__)

MODEL_PROCESO_PATH = 'modelo_proceso.pkl'
MODEL_EJE_PATH = 'modelo_eje.pkl'
MODEL_TEMA_PATH = 'modelo_tema.pkl'
UMBRAL_CONFIANZA = 0.92

_models_loaded = False
_model_proceso = None
_model_eje = None
_model_tema = None

# ============================================================
# FUNCIONES DE LIMPIEZA Y SCORE PARA FALLBACK
# ============================================================
def normalizar_texto(texto):
    if not texto:
        return ""
    import unicodedata
    import re
    texto = str(texto)
    texto = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('utf-8')
    texto = re.sub(r'[^a-zA-Z0-9\s]', '', texto)
    return texto.lower().strip()

def puntuar_coincidencias(texto, lista_palabras):
    texto_norm = normalizar_texto(texto)
    if not texto_norm:
        return 0
    score = 0
    for palabra in lista_palabras:
        if normalizar_texto(palabra) in texto_norm:
            score += 1
    return score

def predecir_por_palabras_clave(texto):
    if not texto:
        return None, None, None

    mejor_proceso = None
    mejor_puntaje_proceso = 0
    for categoria, palabras in KEYWORDS['proceso'].items():
        score = puntuar_coincidencias(texto, palabras)
        if score > mejor_puntaje_proceso:
            mejor_puntaje_proceso = score
            mejor_proceso = categoria
    if mejor_puntaje_proceso == 0:
        mejor_proceso = list(KEYWORDS['proceso'].keys())[0]

    mejor_eje = None
    mejor_puntaje_eje = 0
    for categoria, palabras in KEYWORDS['eje'].items():
        score = puntuar_coincidencias(texto, palabras)
        if score > mejor_puntaje_eje:
            mejor_puntaje_eje = score
            mejor_eje = categoria
    if mejor_puntaje_eje == 0:
        mejor_eje = list(KEYWORDS['eje'].keys())[0]

    mejor_tema = None
    mejor_puntaje_tema = 0
    for categoria, palabras in KEYWORDS['tema'].items():
        score = puntuar_coincidencias(texto, palabras)
        if score > mejor_puntaje_tema:
            mejor_puntaje_tema = score
            mejor_tema = categoria
    if mejor_puntaje_tema == 0:
        mejor_tema = list(KEYWORDS['tema'].keys())[0]

    return mejor_proceso, mejor_eje, mejor_tema

# ============================================================
# FUNCIONES DE ML
# ============================================================
def clean_text(text):
    if not text:
        return ""
    import unicodedata
    import re
    text = str(text)
    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8')
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    return text.lower().strip()

def load_models():
    global _models_loaded, _model_proceso, _model_eje, _model_tema
    try:
        if os.path.exists(MODEL_PROCESO_PATH) and os.path.exists(MODEL_EJE_PATH) and os.path.exists(MODEL_TEMA_PATH):
            with open(MODEL_PROCESO_PATH, 'rb') as f:
                _model_proceso = pickle.load(f)
            with open(MODEL_EJE_PATH, 'rb') as f:
                _model_eje = pickle.load(f)
            with open(MODEL_TEMA_PATH, 'rb') as f:
                _model_tema = pickle.load(f)
            _models_loaded = True
            print("✅ Modelos ML cargados correctamente.")
        else:
            _models_loaded = False
            print("⚠️ Modelos ML no encontrados. Usando fallback por palabras clave.")
    except Exception as e:
        _models_loaded = False
        print(f"❌ Error al cargar modelos: {e}. Usando fallback por palabras clave.")

def train_models():
    try:
        for path in [MODEL_PROCESO_PATH, MODEL_EJE_PATH, MODEL_TEMA_PATH]:
            if os.path.exists(path):
                shutil.copy2(path, path + '.bak')
                print(f"📦 Backup guardado: {path}.bak")

        res = supabase.from_('variables').select('nombre, proceso, eje, tema').execute()
        if not res.data:
            return {"status": "error", "message": "No hay datos etiquetados para entrenar."}

        df = pd.DataFrame(res.data)
        df = df.dropna(subset=['nombre', 'proceso', 'eje', 'tema'])
        df = df[df['nombre'].str.strip() != '']

        if len(df) < 10:
            return {"status": "error", "message": f"Solo hay {len(df)} filas etiquetadas. Se necesitan al menos 10."}

        import re
        def is_consistent(row):
            eje_num = re.search(r'Eje\s*(\d+)', str(row['eje']))
            tema_num = re.search(r'^(\d+)\.', str(row['tema']))
            if eje_num and tema_num:
                return eje_num.group(1) == tema_num.group(1)
            return False

        df_consistent = df[df.apply(is_consistent, axis=1)]
        if len(df_consistent) == 0:
            return {"status": "error", "message": "No hay filas con coherencia entre Eje y Tema. Corrige manualmente antes de reentrenar."}

        if len(df_consistent) < len(df):
            print(f"⚠️ Se descartaron {len(df) - len(df_consistent)} filas inconsistentes.")

        df = df_consistent
        df['nombre_limpio'] = df['nombre'].apply(clean_text)
        X = df['nombre_limpio']

        modelos = {
            'proceso': (df['proceso'], MODEL_PROCESO_PATH),
            'eje': (df['eje'], MODEL_EJE_PATH),
            'tema': (df['tema'], MODEL_TEMA_PATH)
        }
        resultados = {}

        for nombre_col, (y, path) in modelos.items():
            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
            pipeline = Pipeline([
                ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words='english')),
                ('clf', MultinomialNB())
            ])
            pipeline.fit(X_train, y_train)
            y_pred = pipeline.predict(X_test)
            report = classification_report(y_test, y_pred, output_dict=True)
            accuracy = report['accuracy']
            with open(path, 'wb') as f:
                pickle.dump(pipeline, f)
            resultados[nombre_col] = {
                'accuracy': accuracy,
                'samples': len(y_test)
            }

        load_models()
        return {
            "status": "success",
            "message": f"Modelos reentrenados con {len(df)} filas consistentes.",
            "report": resultados
        }
    except Exception as e:
        import traceback
        return {"status": "error", "message": str(e), "traceback": traceback.format_exc()}

def predict_categories(texto):
    if not texto:
        return None, None, None

    if _models_loaded:
        texto_limpio = clean_text(texto)
        if texto_limpio:
            prob_proc = _model_proceso.predict_proba([texto_limpio])[0]
            max_prob_proc = max(prob_proc)
            proc_pred = _model_proceso.classes_[prob_proc.argmax()] if max_prob_proc >= UMBRAL_CONFIANZA else None

            prob_eje = _model_eje.predict_proba([texto_limpio])[0]
            max_prob_eje = max(prob_eje)
            eje_pred = _model_eje.classes_[prob_eje.argmax()] if max_prob_eje >= UMBRAL_CONFIANZA else None

            prob_tema = _model_tema.predict_proba([texto_limpio])[0]
            max_prob_tema = max(prob_tema)
            tema_pred = _model_tema.classes_[prob_tema.argmax()] if max_prob_tema >= UMBRAL_CONFIANZA else None

            if eje_pred and tema_pred:
                import re
                eje_num_match = re.search(r'Eje\s*(\d+)', eje_pred)
                tema_num_match = re.search(r'^(\d+)\.', tema_pred)
                if eje_num_match and tema_num_match:
                    if eje_num_match.group(1) != tema_num_match.group(1):
                        tema_pred = None

            if proc_pred and eje_pred and tema_pred:
                return proc_pred, eje_pred, tema_pred

    return predecir_por_palabras_clave(texto)

load_models()

# ============================================================
# RELLENAR CATEGORÍAS (con normalización)
# ============================================================
def rellenar_categorias(row_data, debug_log=None):
    if debug_log is None:
        debug_log = []

    for col in ['version', 'año', 'valor']:
        if col in row_data and row_data[col] is not None and row_data[col] != '':
            row_data[col] = normalizar_numero_texto(row_data[col], col)
            debug_log.append(f"🔢 Normalizado {col}: {row_data[col]}")

    nombre = row_data.get('nombre')
    if not nombre:
        debug_log.append("❌ Nombre vacío, no se pueden asignar categorías.")
        return row_data

    for key in ['proceso', 'eje', 'tema']:
        if key in row_data and row_data[key] is not None:
            row_data[key] = str(row_data[key]).strip()

    if (not row_data.get('proceso') or not row_data.get('eje') or not row_data.get('tema')):
        proc, eje, tema = predict_categories(nombre)
        if proc and not row_data.get('proceso'):
            row_data['proceso'] = proc
            debug_log.append(f"✅ Proceso asignado: '{proc}' para '{nombre}'")
        if eje and not row_data.get('eje'):
            row_data['eje'] = eje
            debug_log.append(f"✅ Eje asignado: '{eje}' para '{nombre}'")
        if tema and not row_data.get('tema'):
            row_data['tema'] = tema
            debug_log.append(f"✅ Tema asignado: '{tema}' para '{nombre}'")

    if not row_data.get('proceso'):
        row_data['proceso'] = list(KEYWORDS['proceso'].keys())[0]
        debug_log.append(f"⚠️ Forzado proceso por defecto: '{row_data['proceso']}' para '{nombre}'")
    if not row_data.get('eje'):
        row_data['eje'] = list(KEYWORDS['eje'].keys())[0]
        debug_log.append(f"⚠️ Forzado eje por defecto: '{row_data['eje']}' para '{nombre}'")
    if not row_data.get('tema'):
        row_data['tema'] = list(KEYWORDS['tema'].keys())[0]
        debug_log.append(f"⚠️ Forzado tema por defecto: '{row_data['tema']}' para '{nombre}'")

    for col in ['proceso', 'eje', 'tema']:
        if row_data.get(col):
            original = row_data[col]
            normalizado = normalizar_categoria(original, col, debug=True)
            if normalizado != original:
                row_data[col] = normalizado
                debug_log.append(f"🔁 Normalizado {col}: '{original}' -> '{normalizado}'")

    debug_log.append(f"📌 Datos finales: proceso='{row_data.get('proceso')}', eje='{row_data.get('eje')}', tema='{row_data.get('tema')}'")
    return row_data

⚠️ Modelos ML no encontrados. Usando fallback por palabras clave.


# **Bloque 10**

In [14]:
# ============================================================
# BLOQUE 10: RUTAS DE LA API (CORREGIDO - CONFIRMAR CARPETA ZIP)
# ============================================================
@app.route('/')
def indice():
    try:
        js_col_defs = []
        for name in COLUMN_DISPLAY_NAMES:
            key = remover_acentos(name.lower().replace(" ", "_"))
            if name == "ID": key = "id"
            elif name == "Versión": key = "version"
            elif name == "Tipo de dato": key = "tipo_de_dato"
            elif name == "Liga Web": key = "liga_web"
            elif name == "Año": key = "año"
            elif name == "Valor": key = "valor"

            info = {"displayName": name, "keyName": key, "type": "input", "readonly": (key == 'id')}
            if key in DYNAMIC_OPTIONS_COLUMNS:
                info["type"] = "select"
                try:
                    opts_res = supabase.from_(key).select('name').execute()
                    info["options"] = sorted({r['name'] for r in opts_res.data if r.get('name')}) if opts_res.data else []
                except:
                    info["options"] = []
            js_col_defs.append(info)

        headers_html = "".join(f"<th><div class='th-content'>{name}</div></th>" for name in COLUMN_DISPLAY_NAMES)
        headers_html = "<th style='width:40px;'></th>" + headers_html

        script_block = f"<script>window.COLUMN_DEFINITIONS = {json.dumps(js_col_defs).replace('</script>', '<\\\\\\\\/script>')};</script>"
        res_html = HTML_TEMPLATE.replace('<tr id="headerRow"></tr>', f'<tr id="headerRow">{headers_html}</tr>')
        res_html = res_html.replace("</head>", f"{script_block}</head>")
        return render_template_string(res_html)
    except Exception as e:
        traceback.print_exc()
        return f"Error al renderizar el índice: {str(e)}", 500

# ============================================================
# API: VARIABLES
# ============================================================
@app.route('/api/variables', methods=['GET', 'POST'])
def api_vars():
    if request.method == 'GET':
        try:
            data = _get_all_variables_data()
            return jsonify(data)
        except Exception as e:
            traceback.print_exc()
            return jsonify({"error": str(e)}), 500
    elif request.method == 'POST':
        try:
            existing_ids = _get_all_existing_ids()
            new_id = generar_siguiente_id(existing_ids, set())
            data = request.json or {}
            new_row = {
                "id": new_id,
                "version": data.get('version', '-'),
                "proceso": data.get('proceso', ''),
                "eje": data.get('eje', ''),
                "tema": data.get('tema', ''),
                "nombre": data.get('nombre', ''),
                "institucion": data.get('institucion', ''),
                "cobertura": data.get('cobertura', ''),
                "periodicidad": data.get('periodicidad', ''),
                "liga_web": data.get('liga_web', ''),
                "tipo_de_dato": data.get('tipo_de_dato', ''),
                "fuente": data.get('fuente', ''),
                "año": data.get('año', ''),
                "valor": data.get('valor', '')
            }
            if new_row['nombre']:
                proc, eje, tema = predict_categories(new_row['nombre'])
                if proc and not new_row['proceso']:
                    new_row['proceso'] = proc
                if eje and not new_row['eje']:
                    new_row['eje'] = eje
                if tema and not new_row['tema']:
                    new_row['tema'] = tema
            new_row['version'] = normalizar_numero_texto(new_row['version'], 'version')
            if pd.isna(new_row['version']) or str(new_row['version']).strip() == '':
                new_row['version'] = '-'
            new_row['año'] = normalizar_numero_texto(new_row['año'], 'año')
            new_row['valor'] = normalizar_numero_texto(new_row['valor'], 'valor')
            res = supabase.from_('variables').insert(new_row).execute()
            if res.data:
                return jsonify(res.data[0]), 201
            else:
                return jsonify({"error": "Error al crear variable"}), 500
        except Exception as e:
            traceback.print_exc()
            return jsonify({"error": str(e), "traceback": traceback.format_exc()}), 500

@app.route('/api/variables/<id_val>', methods=['PUT', 'DELETE'])
def api_item(id_val):
    if request.method == 'PUT':
        try:
            data = request.json
            print(f"📥 Recibido PUT para {id_val}: {data}")
            existing = supabase.from_('variables').select('*').eq('id', id_val).execute()
            if not existing.data:
                return jsonify({"error": "Variable no encontrada"}), 404
            current = existing.data[0]
            cleaned_data = {}
            for k, v in data.items():
                if k in EXPECTED_COLUMNS:
                    cleaned_data[k] = str(v) if not pd.isna(v) else None
            if 'nombre' in cleaned_data and cleaned_data['nombre']:
                nombre_nuevo = cleaned_data['nombre']
                if not current.get('proceso') or current.get('proceso') == '' or \
                   not current.get('eje') or current.get('eje') == '' or \
                   not current.get('tema') or current.get('tema') == '':
                    proc, eje, tema = predict_categories(nombre_nuevo)
                    if proc and (not current.get('proceso') or current.get('proceso') == ''):
                        cleaned_data['proceso'] = proc
                    if eje and (not current.get('eje') or current.get('eje') == ''):
                        cleaned_data['eje'] = eje
                    if tema and (not current.get('tema') or current.get('tema') == ''):
                        cleaned_data['tema'] = tema
            if 'version' in cleaned_data:
                cleaned_data['version'] = normalizar_numero_texto(cleaned_data['version'], 'version')
                if pd.isna(cleaned_data['version']) or str(cleaned_data['version']).strip() == '':
                    cleaned_data['version'] = '-'
            if 'año' in cleaned_data:
                cleaned_data['año'] = normalizar_numero_texto(cleaned_data['año'], 'año')
            if 'valor' in cleaned_data:
                cleaned_data['valor'] = normalizar_numero_texto(cleaned_data['valor'], 'valor')
            res = supabase.from_('variables').update(cleaned_data).eq('id', id_val).execute()
            if res.data:
                return jsonify({"status": "ok"}), 200
            else:
                return jsonify({"error": "Error al actualizar variable"}), 500
        except Exception as e:
            traceback.print_exc()
            return jsonify({"error": str(e)}), 500

    elif request.method == 'DELETE':
        try:
            # Elimina el registro de Supabase usando delete() en lugar de update()
            res = supabase.from_('variables').delete().eq('id', id_val).execute()
            if res.error:
                return jsonify({"error": res.error.message}), 500
            elif res.data and len(res.data) > 0:
                return jsonify({"status": "ok"}), 200
            else:
                return jsonify({"error": "Registro no encontrado"}), 404
        except Exception as e:
            traceback.print_exc()
            return jsonify({"error": str(e)}), 500

# ============================================================
# API: DUPLICAR
# ============================================================
@app.route('/api/variables/<id_val>/duplicate', methods=['POST'])
def duplicate_var(id_val):
    try:
        res = supabase.from_('variables').select('*').eq('id', id_val).execute()
        if not res.data:
            return jsonify({"error": "Variable no encontrada"}), 404
        original_row = res.data[0]
        duplicated_row = original_row.copy()
        existing_ids = _get_all_existing_ids()
        new_id = generar_siguiente_id(existing_ids, set())
        duplicated_row['id'] = new_id
        if duplicated_row.get('version') is None or duplicated_row.get('version') == '':
            duplicated_row['version'] = '-'
        duplicated_row['version'] = normalizar_numero_texto(duplicated_row['version'], 'version')
        duplicated_row['año'] = normalizar_numero_texto(duplicated_row['año'], 'año')
        duplicated_row['valor'] = normalizar_numero_texto(duplicated_row['valor'], 'valor')
        insert_res = supabase.from_('variables').insert(duplicated_row).execute()
        if insert_res.data:
            return jsonify(insert_res.data[0]), 201
        else:
            return jsonify({"error": "Error al duplicar variable"}), 500
    except Exception as e:
        traceback.print_exc()
        return jsonify({"error": str(e)}), 500

# ============================================================
# API: CATÁLOGO
# ============================================================
@app.route('/api/catalog/<table_name>', methods=['POST'])
def add_catalog_option(table_name):
    try:
        data = request.json
        value = data.get('value', '').strip()
        if not value:
            return jsonify({"error": "Valor no puede estar vacío"}), 400
        all_options = _get_catalog_options(table_name)
        match_val, score = _find_best_value_match(value, all_options, threshold=0.95)
        if score == 1.0:
            return jsonify({"status": "already_exists", "value": match_val}), 200
        elif score > 0.7:
            return jsonify({"status": "suggestion", "original_input": value, "suggested_value": match_val}), 200
        else:
            res = supabase.from_(table_name).insert({"name": value}).execute()
            if res.data:
                return jsonify({"status": "inserted", "value": res.data[0]['name']}), 201
            else:
                return jsonify({"error": "Error al agregar opción de catálogo"}), 500
    except Exception as e:
        traceback.print_exc()
        return jsonify({"error": str(e)}), 500

# ============================================================
# API: DESCARGAR CSV
# ============================================================
@app.route('/api/download')
def download_csv():
    try:
        data = _get_all_variables_data()
        df = pd.DataFrame(data)
        if df.empty:
            csv_output = ",".join(COLUMN_DISPLAY_NAMES) + "\n"
        else:
            df_display = df[EXPECTED_COLUMNS].copy()
            display_name_map = {remover_acentos(name.lower().replace(" ", "_")): name for name in COLUMN_DISPLAY_NAMES}
            df_display.rename(columns=display_name_map, inplace=True)
            for col_name in COLUMN_DISPLAY_NAMES:
                if col_name not in df_display.columns:
                    df_display[col_name] = ''
            df_display = df_display[COLUMN_DISPLAY_NAMES]
            csv_output = df_display.to_csv(index=False, encoding='utf-8-sig')
        response = make_response(csv_output)
        now = datetime.datetime.now()
        fecha_hora = now.strftime('%Y%m%d%H%M%S')
        filename = f"variables_{fecha_hora}.csv"
        response.headers["Content-Disposition"] = f"attachment; filename={filename}"
        response.headers["Content-type"] = "text/csv; charset=utf-8"
        return response
    except Exception as e:
        traceback.print_exc()
        return jsonify({"error": str(e)}), 500

# ============================================================
# API: SUBIR CSV (CORREGIDO)
# ============================================================
@app.route('/api/upload', methods=['POST'])
def upload_csv():
    try:
        if 'file' not in request.files:
            return jsonify({"error": "No se encontró el archivo"}), 400
        file = request.files['file']
        if file.filename == '' or not file.filename.endswith('.csv'):
            return jsonify({"error": "Archivo inválido. Debe ser CSV."}), 400

        file_content = file.read()
        file_stream = io.BytesIO(file_content)
        df = None
        messages = []

        try:
            df = pd.read_csv(file_stream, encoding='utf-8-sig')
            messages.append("CSV procesado correctamente (UTF-8).")
        except UnicodeDecodeError:
            file_stream.seek(0)
            try:
                df = pd.read_csv(file_stream, encoding='latin-1')
                messages.append("CSV procesado correctamente (Latin-1).")
            except Exception as e:
                raise ValueError(f"No se pudo decodificar el CSV: {e}")
        except Exception as e:
            raise ValueError(f"Error al leer el CSV: {e}")

        print("📋 Columnas leídas del CSV:", list(df.columns))

        year_aliases = ['año', 'ano', 'anio', 'Año', 'AÑO']
        year_col_found = None
        for col in df.columns:
            col_clean = remover_acentos(str(col).lower().strip().replace(' ', '_'))
            if col_clean in year_aliases or col_clean == 'año':
                year_col_found = col
                break

        if year_col_found:
            if year_col_found != 'año':
                df.rename(columns={year_col_found: 'año'}, inplace=True)
                messages.append(f"Columna '{year_col_found}' renombrada a 'año' por alias.")
            if not df['año'].isna().all() and not (df['año'].astype(str).str.strip() == '').all():
                messages.append("✅ Año extraído de la columna 'año'.")
            else:
                messages.append("⚠️ La columna 'año' está vacía. Se dejará vacía.")
                df['año'] = pd.NA
        else:
            messages.append("⚠️ No se encontró columna de año. Se dejará vacía.")
            df['año'] = pd.NA

        rename_map = {}
        for col in df.columns:
            norm_col = _normalize_col_for_matching(col)
            if norm_col in EXPECTED_COLUMNS:
                rename_map[col] = norm_col
        if rename_map:
            df.rename(columns=rename_map, inplace=True)
            messages.append("Nombres de columnas normalizados.")

        existing_data = _get_all_variables_data()
        processed_rows, process_msgs = _process_csv_data(df, existing_data, debug=True)
        messages.extend(process_msgs)

        if request.args.get('preview') == 'true':
            cleaned = _clean_data_for_json(processed_rows)
            return jsonify({"status": "preview", "messages": messages, "preview_data": cleaned}), 200

        actions_str = request.form.get('actions_for_rows', '[]')
        actions = json.loads(actions_str)

        data_to_insert = []
        data_to_update = []
        existing_ids = _get_all_existing_ids()
        used_ids = set(existing_ids)
        debug_log = []

        for action_info in actions:
            action = action_info['action']
            row_data = action_info['data']
            supabase_id = action_info.get('_supabase_matching_id')

            row_data = rellenar_categorias(row_data, debug_log)

            if action == 'insert':
                new_id = generar_siguiente_id(used_ids, set())
                row_data['id'] = new_id
                used_ids.add(new_id)
                cleaned_row = {}
                for k, v in row_data.items():
                    cleaned_row[k] = str(v) if not pd.isna(v) else None
                    if k == 'version' and (cleaned_row[k] is None or cleaned_row[k] == ''):
                        cleaned_row[k] = '-'
                data_to_insert.append(cleaned_row)
            elif action == 'overwrite':
                row_data['id'] = supabase_id
                cleaned_row = {}
                for k, v in row_data.items():
                    if k in EXPECTED_COLUMNS:
                        cleaned_row[k] = str(v) if not pd.isna(v) else None
                        if k == 'version' and (cleaned_row[k] is None or cleaned_row[k] == ''):
                            cleaned_row[k] = '-'
                data_to_update.append(cleaned_row)

        inserted = 0
        inserted_data = []
        if data_to_insert:
            res = supabase.from_('variables').insert(data_to_insert).execute()
            if res.data:
                inserted = len(res.data)
                inserted_data = res.data
                messages.append(f"Se insertaron {inserted} filas nuevas.")
        updated = 0
        updated_data = []
        if data_to_update:
            for row in data_to_update:
                row_id = row.pop('id')
                res = supabase.from_('variables').update(row).eq('id', row_id).execute()
                if res.data:
                    updated += 1
                    updated_data.append(res.data[0])
            if updated > 0:
                messages.append(f"Se actualizaron {updated} filas existentes.")
            else:
                messages.append("Advertencia: No se pudo actualizar ninguna fila duplicada.")
        if not inserted and not updated:
            messages.append("No se realizaron inserciones ni actualizaciones.")

        try:
            supabase.from_('cargas').insert({
                'fecha': datetime.datetime.now().isoformat(),
                'archivo': file.filename,
                'filas_agregadas': inserted,
                'estado': 'completado' if inserted > 0 else 'error'
            }).execute()
        except Exception as e:
            print(f"Error al guardar historial: {e}")

        summary = []
        for msg in messages:
            if "CSV procesado" in msg:
                summary.append("CSV procesado.")
            elif "Columnas ignoradas" in msg:
                summary.append("Columnas no esperadas ignoradas.")
            elif "Se encontraron" in msg and "duplicados" in msg:
                match = re.search(r'(\d+)', msg)
                if match:
                    summary.append(f"{match.group(1)} filas duplicadas encontradas.")
            elif "Se insertaron" in msg:
                match = re.search(r'(\d+)', msg)
                if match:
                    summary.append(f"{match.group(1)} filas insertadas.")
            elif "Se actualizaron" in msg:
                match = re.search(r'(\d+)', msg)
                if match:
                    summary.append(f"{match.group(1)} filas actualizadas.")
        final = "; ".join(summary) or "Proceso de carga de CSV completado."
        return jsonify({
            "status": "ok",
            "message": final,
            "debug": debug_log,
            "inserted": inserted_data,
            "updated": updated_data
        }), 200

    except Exception as e:
        traceback.print_exc()
        return jsonify({"error": str(e), "traceback": traceback.format_exc()}), 500

# ============================================================
# API: PROCESAR CARPETA ZIP (DEFINITIVO - BÚSQUEDA ROBUSTA)
# ============================================================
@app.route('/api/process-folder', methods=['POST'])
def process_folder():
    log_messages = []
    try:
        if 'file' not in request.files:
            return jsonify({"error": "No se encontró el archivo ZIP"}), 400
        file = request.files['file']
        if file.filename == '' or not file.filename.endswith('.zip'):
            return jsonify({"error": "El archivo debe ser un ZIP"}), 400

        zip_filename = file.filename

        with tempfile.TemporaryDirectory() as tmpdir:
            zip_path = os.path.join(tmpdir, 'uploaded.zip')
            file.save(zip_path)
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(tmpdir)

            # ---- FASE 1: REORGANIZAR ARCHIVOS SUELTOS ----
            root_items = os.listdir(tmpdir)
            conjunto_files = []
            diccionario_files = []

            for item in root_items:
                item_path = os.path.join(tmpdir, item)
                if os.path.isfile(item_path):
                    lower_name = item.lower()
                    if 'conjunto_de_datos' in lower_name or 'conjunto_datos' in lower_name:
                        conjunto_files.append(item_path)
                    elif 'diccionario_de_datos' in lower_name or 'diccionario_datos' in lower_name:
                        diccionario_files.append(item_path)

            # Crear carpetas base
            conjunto_path = os.path.join(tmpdir, 'conjunto_de_datos')
            diccionario_path = os.path.join(tmpdir, 'diccionario_de_datos')
            os.makedirs(conjunto_path, exist_ok=True)
            os.makedirs(diccionario_path, exist_ok=True)

            # Mover archivos sueltos a conjunto
            for src in conjunto_files:
                basename = os.path.basename(src.replace('\\', '/'))
                dest = os.path.join(conjunto_path, basename)
                shutil.move(src, dest)
                log_messages.append(f"📦 Movido archivo suelto a conjunto: {basename}")

            # Mover archivos sueltos a diccionario
            for src in diccionario_files:
                basename = os.path.basename(src.replace('\\', '/'))
                dest = os.path.join(diccionario_path, basename)
                shutil.move(src, dest)
                log_messages.append(f"📦 Movido archivo suelto a diccionario: {basename}")

            # ---- FASE 2: BÚSQUEDA DE CARPETAS (si no se crearon automáticamente) ----
            # Si las carpetas recién creadas no contienen CSV, buscar otras carpetas
            if not any(f.endswith('.csv') for f in os.listdir(conjunto_path)):
                # Buscar cualquier carpeta con CSV que contenga 'conjunto' o 'datos'
                for root, dirs, files in os.walk(tmpdir):
                    if any(f.endswith('.csv') for f in files):
                        if 'conjunto' in root.lower() or 'datos' in root.lower():
                            conjunto_path = root
                            log_messages.append(f"🔍 Conjunto encontrado en: {conjunto_path}")
                            break
                else:
                    # Si no, buscar cualquier carpeta con CSV que no sea diccionario
                    for root, dirs, files in os.walk(tmpdir):
                        if any(f.endswith('.csv') for f in files) and 'diccionario' not in root.lower():
                            conjunto_path = root
                            log_messages.append(f"🔍 Conjunto (alternativo) en: {conjunto_path}")
                            break

            if not any(f.endswith('.csv') for f in os.listdir(diccionario_path)):
                for root, dirs, files in os.walk(tmpdir):
                    if any(f.endswith('.csv') for f in files) and 'diccionario' in root.lower():
                        diccionario_path = root
                        log_messages.append(f"🔍 Diccionario encontrado en: {diccionario_path}")
                        break

            # Buscar metadatos (cualquier archivo .txt en carpeta con 'metadatos')
            metadatos_path = None
            for root, dirs, files in os.walk(tmpdir):
                txt_files = [f for f in files if f.endswith('.txt')]
                if txt_files and 'metadatos' in root.lower():
                    metadatos_path = os.path.join(root, txt_files[0])
                    log_messages.append(f"📄 Metadatos encontrado en: {metadatos_path}")
                    break

            # ---- VALIDACIONES FINALES CON SEGURIDAD ----
            # Verificar que conjunto_path y diccionario_path no sean None y sean directorios con CSV
            if (not conjunto_path or not os.path.isdir(conjunto_path) or
                not any(f.endswith('.csv') for f in os.listdir(conjunto_path))):
                log_messages.append("❌ No se encontró una carpeta de conjunto de datos válida.")
                return jsonify({
                    "error": "No se encontró la carpeta 'conjunto_de_datos' con archivos CSV",
                    "messages": log_messages
                }), 400

            if (not diccionario_path or not os.path.isdir(diccionario_path) or
                not any(f.endswith('.csv') for f in os.listdir(diccionario_path))):
                log_messages.append("❌ No se encontró una carpeta de diccionario de datos válida.")
                return jsonify({
                    "error": "No se encontró la carpeta 'diccionario_de_datos' con archivos CSV",
                    "messages": log_messages
                }), 400

            log_messages.append(f"✅ Carpetas encontradas: conjunto={conjunto_path}, diccionario={diccionario_path}, metadatos={metadatos_path}")

            # ---- DETECCIÓN DE FORMATO NUEVO ----
            new_format = False
            indice_filename = None
            if conjunto_path and os.path.isdir(conjunto_path):
                for f in os.listdir(conjunto_path):
                    if f.endswith('.csv'):
                        f_norm = unicodedata.normalize('NFKD', f.lower()).encode('ascii', 'ignore').decode('utf-8')
                        if 'indice' in f_norm:
                            new_format = True
                            indice_filename = f
                            log_messages.append(f"📌 Formato nuevo detectado: archivo índice '{indice_filename}'")
                            break

            if new_format:
                aggregation_keywords = ['total', 'totales', 'cantidad', 'sum']
            else:
                aggregation_keywords = ['cantidad de', 'total']

            # ---- PROCESAR ----
            final_df, global_metadata = process_data_folders_with_return(
                data_dir=conjunto_path,
                dict_dir=diccionario_path,
                metadatos_path=metadatos_path,
                new_format=new_format,
                indice_filename=indice_filename,
                aggregation_keywords=aggregation_keywords,
                log_messages=log_messages
            )

            # ---- PREVISUALIZACIÓN ----
            if not final_df.empty:
                preview_raw = final_df.to_dict(orient='records')
            else:
                preview_raw = []

            if preview_raw:
                df_temp = pd.DataFrame(preview_raw)
                for col in EXPECTED_COLUMNS:
                    if col not in df_temp.columns:
                        df_temp[col] = ''
                df_temp = df_temp[EXPECTED_COLUMNS]

                existing_data = _get_all_variables_data()
                processed_rows, dup_messages = _process_csv_data(df_temp, existing_data, debug=True)
                log_messages.extend(dup_messages)

                preview_data = []
                for row_info in processed_rows:
                    row_data = row_info['data']
                    if 'valor' not in row_data:
                        row_data['valor'] = None
                    preview_data.append({
                        'data': row_data,
                        '_is_duplicate': row_info['_is_duplicate'],
                        '_supabase_matching_id': row_info['_supabase_matching_id'],
                        'original_csv_index': row_info['original_csv_index'],
                        '_action': 'insert' if not row_info['_is_duplicate'] else 'ignore'
                    })
            else:
                preview_data = []

            log_messages.append(f"📊 Total de filas extraídas: {len(preview_data)}")
            if len(preview_data) == 0:
                log_messages.append("💡 No se encontraron filas. Verifica que las palabras clave de agregación coincidan con las descripciones de tus columnas.")

            return jsonify({
                "status": "preview",
                "messages": log_messages,
                "preview_data": preview_data,
                "global_metadata": global_metadata,
                "zip_filename": zip_filename
            }), 200

    except Exception as e:
        traceback.print_exc()
        return jsonify({
            "error": str(e),
            "traceback": traceback.format_exc(),
            "messages": log_messages
        }), 500

# ============================================================
# API: CONFIRMAR CARPETA ZIP (CORREGIDO - INSERTAR TODAS LAS FILAS)
# ============================================================
@app.route('/api/confirm-folder', methods=['POST'])
def confirm_folder():
    try:
        actions_str = request.form.get('actions_for_rows', '[]')
        actions = json.loads(actions_str)
        zip_filename = request.form.get('zip_filename', 'carga_carpeta')

        if not actions:
            return jsonify({"error": "No hay datos para confirmar"}), 400

        data_to_insert = []
        data_to_update = []
        existing_ids = _get_all_existing_ids()
        used_ids = set(existing_ids)
        debug_log = []

        for action_info in actions:
            action = action_info.get('action', 'insert')
            row_data = action_info.get('data', {})
            supabase_id = action_info.get('_supabase_matching_id')

            row_data = rellenar_categorias(row_data, debug_log)

            if action == 'insert' or action == 'ignore':
                new_id = generar_siguiente_id(used_ids, set())
                row_data['id'] = new_id
                used_ids.add(new_id)
                cleaned_row = {}
                for k, v in row_data.items():
                    if k in EXPECTED_COLUMNS or k == 'valor':
                        cleaned_row[k] = str(v) if not pd.isna(v) else None
                        if k == 'version' and (cleaned_row[k] is None or cleaned_row[k] == ''):
                            cleaned_row[k] = '-'
                data_to_insert.append(cleaned_row)
            elif action == 'overwrite':
                if supabase_id:
                    row_data['id'] = supabase_id
                    cleaned_row = {}
                    for k, v in row_data.items():
                        if k in EXPECTED_COLUMNS or k == 'valor':
                            cleaned_row[k] = str(v) if not pd.isna(v) else None
                            if k == 'version' and (cleaned_row[k] is None or cleaned_row[k] == ''):
                                cleaned_row[k] = '-'
                    data_to_update.append(cleaned_row)

        inserted = 0
        inserted_data = []
        if data_to_insert:
            res = supabase.from_('variables').insert(data_to_insert).execute()
            if res.data:
                inserted = len(res.data)
                inserted_data = res.data

        updated = 0
        updated_data = []
        if data_to_update:
            for row in data_to_update:
                row_id = row.pop('id')
                res = supabase.from_('variables').update(row).eq('id', row_id).execute()
                if res.data:
                    updated += 1
                    updated_data.append(res.data[0])

        try:
            supabase.from_('cargas').insert({
                'fecha': datetime.datetime.now().isoformat(),
                'archivo': zip_filename,
                'filas_agregadas': inserted,
                'estado': 'completado' if inserted > 0 or updated > 0 else 'error'
            }).execute()
        except Exception as e:
            print(f"Error al guardar historial: {e}")

        msg = f"Se insertaron {inserted} filas nuevas."
        if updated > 0:
            msg += f" Se actualizaron {updated} filas existentes."
        return jsonify({
            "status": "ok",
            "message": msg,
            "debug": debug_log,
            "inserted": inserted_data,
            "updated": updated_data
        }), 200

    except Exception as e:
        traceback.print_exc()
        return jsonify({"error": str(e), "traceback": traceback.format_exc()}), 500

# ============================================================
# API: REENTRENAR MODELOS
# ============================================================
@app.route('/api/retrain-models', methods=['POST'])
def retrain_models():
    try:
        result = train_models()
        if result.get('status') == 'success':
            return jsonify({"status": "ok", "message": result['message'], "report": result.get('report')}), 200
        else:
            return jsonify({"error": result.get('message', 'Error al entrenar')}), 500
    except Exception as e:
        traceback.print_exc()
        return jsonify({"error": str(e), "traceback": traceback.format_exc()}), 500

# ============================================================
# API: ROLLBACK DE MODELOS
# ============================================================
@app.route('/api/rollback-models', methods=['POST'])
def rollback_models():
    try:
        restored = []
        for path in [MODEL_PROCESO_PATH, MODEL_EJE_PATH, MODEL_TEMA_PATH]:
            backup_path = path + '.bak'
            if os.path.exists(backup_path):
                shutil.copy2(backup_path, path)
                restored.append(path)
        if restored:
            load_models()
            return jsonify({"status": "ok", "message": f"Restaurados: {', '.join(restored)}"}), 200
        else:
            return jsonify({"error": "No hay backups disponibles"}), 404
    except Exception as e:
        traceback.print_exc()
        return jsonify({"error": str(e)}), 500

# ============================================================
# API: ELIMINAR MODELOS
# ============================================================
@app.route('/api/delete-models', methods=['DELETE'])
def delete_models():
    try:
        deleted = []
        for path in [MODEL_PROCESO_PATH, MODEL_EJE_PATH, MODEL_TEMA_PATH]:
            if os.path.exists(path):
                os.remove(path)
                deleted.append(path)
        global _models_loaded
        _models_loaded = False
        return jsonify({"status": "ok", "message": f"Eliminados: {', '.join(deleted)}"}), 200
    except Exception as e:
        traceback.print_exc()
        return jsonify({"error": str(e)}), 500

# ============================================================
# API: HISTORIAL
# ============================================================
@app.route('/api/upload-history', methods=['GET'])
def upload_history():
    try:
        res = supabase.from_('cargas').select('*').order('fecha', desc=True).limit(50).execute()
        return jsonify(res.data if res.data else [])
    except Exception as e:
        print(f"Error al obtener historial: {e}")
        return jsonify([])

# **Bloque 11**

In [15]:
# ============================================================
# BLOQUE 11: RUTAS DE LA API (CORREGIDO - CONFIRMAR CARPETA ZIP)
# ============================================================
import uuid
import threading
import time
import socket
from pyngrok import ngrok
from pyngrok.exception import PyngrokNgrokHTTPError

FLASK_PORT = 5000
ngrok.kill()
time.sleep(5)

def find_available_port(start_port):
    port = start_port
    while True:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            try:
                s.bind(('0.0.0.0', port))
                return port
            except OSError:
                print(f"Puerto {port} en uso, intentando siguiente...")
                port += 1

available_port = find_available_port(FLASK_PORT)
FLASK_PORT = available_port

def ejecutar_app_flask():
    app.run(host='0.0.0.0', port=FLASK_PORT, debug=False, use_reloader=False)

hilo_flask = threading.Thread(target=ejecutar_app_flask)
hilo_flask.start()
time.sleep(3)

NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

url_publica = None
MAX_RETRIES = 5
RETRY_DELAY = 10

for attempt in range(MAX_RETRIES):
    try:
        url_publica = ngrok.connect(FLASK_PORT).public_url
        print(f" * Túnel ngrok disponible en: {url_publica} (Flask puerto {FLASK_PORT})")
        break
    except PyngrokNgrokHTTPError as e:
        if "ERR_NGROK_334" in str(e) and attempt < MAX_RETRIES - 1:
            print(f"Advertencia: ngrok ya en línea. Reintentando en {RETRY_DELAY}s... (Intento {attempt+1}/{MAX_RETRIES})")
            time.sleep(RETRY_DELAY)
        else:
            print(f"Error fatal ngrok: {e}")
            raise
    except Exception as e:
        print(f"Error inesperado: {e}")
        raise
else:
    print(f"Error: No se pudo establecer túnel ngrok.")

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


 * Túnel ngrok disponible en: https://unpaid-barterer-erasable.ngrok-free.dev (Flask puerto 5000)
